# web item

In [ ]:
from __future__ import annotations
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *

import random
import math
import pandas as pd
import httpx
import re
import heapq
import json
import logging

import networkx as nx
from itertools import combinations

from datetime import datetime
import numpy as np
from dataclasses import dataclass, field
import matplotlib.pyplot as plt
import string
from typing import NamedTuple


In [ ]:
from HexMagic.geology import Geology, DrainageBasins, Watershed
from HexMagic.overlay import  TerrainDisplay, TerrainOverlay, ClimateOverlay, TerraDemo, DrainageBasins, OverlaySpec
from HexMagic.game.flag import CountryFlag , PieceType, GameContext, DiagramGlyphs
from HexMagic.styles import StyleCSS,  SVGBuilder, SVGDef
from HexMagic.overlay import OverlaySpec, FlowOverlay, CreamOverlay, RiverOverlay, OceanWaveOverlay
from HexMagic.overlay import ClimateOverlay, WatershedOverlay, TemperatureOverlay, PrecipitationOverlay,SoilOverlay
from HexMagic.climate import  TerraDemo, TerrainFactory
from HexMagic.primitives import HexTouchMap, HexPosition, HexGrid, HexDragMap, HexTouchMap, MapCord , PrimitiveDemo, Hex, HexWrapper
from HexMagic.primitives import MapRect, MapPath,MapCord,  MapSize, HexRegion
from HexMagic.water.soil import SoilSystem
from HexMagic.terrainpatterns import TerrainPatterns

In [ ]:


@dataclass
class FoodYield:
    """Compute per-hex food yield tiers from temperature, water, and soil."""
    terrain: object  # Terrain
    basins: object   # DrainageBasins
    n_tiers: int = 8
    
    # Climate zone index → temperature factor (tune these!)
    temp_curve:  dict[int, float] = field(default_factory=lambda: {
        0: 0.0,   # Ocean/ice
        1: 0.15,  # Tundra
        2: 0.4,   # Boreal
        3: 0.8,   # Temperate
        4: 1.0,   # Subtropical (peak)
        5: 0.6,   # Tropical/hot
    })
    
    # Soil type index → fertility multiplier
    soil_mult: list[float] = field(default_factory=lambda: [
        0.2,  # Granite
        0.3,  # Basalt
        0.7,  # Limestone
        0.6,  # Sandstone
        1.0,  # Alluvial
    ])
    
    def compute(self) -> np.ndarray:
        """Returns per-hex yield as float 0–1, and stores tier (0..n_tiers-1)."""
        t = self.terrain
        n = len(t.elevations)
        
        # --- Temperature factor ---
        climate = t.fields.get('climate', t.compute_climate())
        temp_f = np.array([self.temp_curve.get(int(c), 0.0) for c in climate])
        
        # --- Water factor (precip + flow, diminishing returns) ---
        precip = t.fields.get('precipitation', np.zeros(n))
        
        # Ensure flow field exists
        if 'flow' not in t.fields:
            all_flows = {}
            for ws in self.basins.sheds:
                for idx, fl in ws.tributary._calculate_flow().items():
                    all_flows[idx] = all_flows.get(idx, 0) + fl
            t.fields['flow'] = np.zeros(n)
            for idx, fl in all_flows.items():
                t.fields['flow'][idx] = fl
        
        flow = t.fields['flow']
        
        # Combine precip + flow, log-scale for diminishing returns
        water_raw = precip / 1000.0 + np.log1p(flow) * 0.3
        water_max = np.percentile(water_raw[water_raw > 0], 95) if np.any(water_raw > 0) else 1.0
        water_f = np.clip(water_raw / water_max, 0, 1.0)
        
        # --- Soil factor ---
        soil_type = t.fields.get('soil_type', np.zeros(n, dtype=int))
        soil_f = np.array([self.soil_mult[min(int(s), len(self.soil_mult)-1)] for s in soil_type])
        
        # --- Combine ---
        raw = temp_f * water_f * soil_f
        
        # Zero out ocean
        raw[t.elevations <= 0] = 0.0
        
        # Normalize to 0–1
        rmax = np.percentile(raw[raw > 0], 95) if np.any(raw > 0) else 1.0
        normalized = raw / rmax
        
        # Store
        self.raw = raw
        self.normalized = normalized
        self.tiers = np.clip((normalized * self.n_tiers).astype(int), 0, self.n_tiers - 1)
        t.fields['food_yield'] = self.tiers
        
        return normalized
    
    def summary(self):
        """Print tier distribution."""
        land = self.tiers[self.terrain.elevations > 0]
        print(f"Land hexes: {len(land)}")
        for tier in range(self.n_tiers):
            count = np.sum(land == tier)
            bar = '█' * (count // 10)
            print(f"  Tier {tier}: {count:5d} {bar}")


In [ ]:
def FoodOverlay(color: str = "#558B2F", n_tiers: int = 8, skip_zero: bool = True, **kw):
    """Dotted density overlay showing food yield per hex."""
    
    def render(ctx):
        fy = FoodYield(ctx.terrain, ctx.basins, n_tiers=n_tiers)
        fy.compute()
        
        grid = ctx.grid
        patGen = TerrainPatterns(ctx.terrain)
        patterns = patGen.ballDensity(
            levels=n_tiers,
            fills=[color],
            prefix="food_yield"
        )
        
        overlay = ""
        used = set()
        
        for i in range(len(ctx.terrain.elevations)):
            if ctx.terrain.elevations[i] <= 0:
                continue
            tier = int(fy.tiers[i])
            if skip_zero and tier == 0:
                continue
            
            used.add(tier)
            pat_name = patterns[tier].attributes['id']
            hex_obj = grid.hexes[i]
            pts = " ".join(f"{p.x:.0f},{p.y:.0f}" for p in hex_obj.vertices())
            overlay += f'\t<polygon points="{pts}" style="fill:url(#{pat_name})"/>\n'
        
        for tier in sorted(used):
            ctx.builder.add_definition(patterns[tier])
        
        return overlay
    
    return OverlaySpec("food_yield", render, requires={'basins'}, priority=44)


In [ ]:
@patch
def __ft__(self: FoodYield):
    """Distribution charts for food yield tuning."""
    t = self.terrain
    land = t.elevations > 0

    tiers     = self.tiers[land]
    elevs     = t.elevations[land]
    climate   = t.fields.get('climate', t.compute_climate())[land]

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

    # ── 1. Tier distribution bar chart ──
    ax = axes[0]
    counts = [np.sum(tiers == i) for i in range(self.n_tiers)]
    colors = plt.cm.YlGn(np.linspace(0.2, 0.9, self.n_tiers))
    bars = ax.bar(range(self.n_tiers), counts, color=colors, edgecolor='#555', linewidth=0.5)
    ax.set_title('Tier Distribution', fontweight='bold', fontsize=10)
    ax.set_xlabel('Tier'); ax.set_ylabel('Land hexes')
    ax.set_xticks(range(self.n_tiers))
    for bar, count in zip(bars, counts):
        if count > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                    str(count), ha='center', va='bottom', fontsize=7, color='#444')
    ax.grid(axis='y', alpha=0.3, lw=0.5)

    # ── 2. Tier by elevation band ──
    ax = axes[1]
    elev_max = np.percentile(elevs, 98)
    n_bands = 6
    band_edges = np.linspace(0, elev_max, n_bands + 1)
    band_data  = []
    band_labels = []
    for lo, hi in zip(band_edges, band_edges[1:]):
        mask = (elevs >= lo) & (elevs < hi)
        band_data.append(tiers[mask].tolist() if mask.any() else [0])
        band_labels.append(f"{lo:.0f}–{hi:.0f}")

    bp = ax.boxplot(band_data, tick_labels=band_labels, patch_artist=True,
                    medianprops=dict(color='#222', linewidth=1.5),
                    whiskerprops=dict(color='#888'), capprops=dict(color='#888'))
    for patch in bp['boxes']:
        patch.set_facecolor('#8D6E63'); patch.set_alpha(0.55)
    ax.set_title('Tier by Elevation Band', fontweight='bold', fontsize=10)
    ax.set_xlabel('Elevation'); ax.set_ylabel('Food tier')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.grid(axis='y', alpha=0.3, lw=0.5)

    # ── 3. Tier by climate zone ──
    ax = axes[2]
    climate_names = {0:'Ocean/Ice', 1:'Tundra', 2:'Boreal',
                     3:'Temperate', 4:'Subtropical', 5:'Tropical'}
    zones_present = sorted(set(int(c) for c in climate))
    zone_data   = [tiers[climate == z].tolist() for z in zones_present]
    zone_labels = [climate_names.get(z, str(z)) for z in zones_present]
    zone_colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(zones_present)))

    bp2 = ax.boxplot(zone_data, tick_labels=zone_labels, patch_artist=True,
                     medianprops=dict(color='#222', linewidth=1.5),
                     whiskerprops=dict(color='#888'), capprops=dict(color='#888'))
    for patch, color in zip(bp2['boxes'], zone_colors):
        patch.set_facecolor(color); patch.set_alpha(0.65)
    ax.set_title('Tier by Climate Zone', fontweight='bold', fontsize=10)
    ax.set_xlabel('Climate'); ax.set_ylabel('Food tier')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.grid(axis='y', alpha=0.3, lw=0.5)

    land_count = int(np.sum(land))
    fig.suptitle(
        f"FoodYield — {land_count} land hexes, {self.n_tiers} tiers",
        fontsize=11, fontweight='bold', y=1.02)
    fig.tight_layout()

    buf = io.StringIO()
    fig.savefig(buf, format='svg', bbox_inches='tight')
    plt.close(fig)

    return Div(
        NotStr(buf.getvalue()),
        P(f"temp_curve · soil_mult · {self.n_tiers} tiers",
          cls="text-xs opacity-50 text-center"),
        cls="space-y-1")


In [ ]:
class GameParts:

    def __init__(self, radius: int = 20, useKoreanMap: bool = False,size=300):
        if useKoreanMap:
            self.terr: TerraDemo = TerraDemo().japan_korea_map()
        else:
            world = TerrainFactory.create_world(
                bounds=MapRect(MapCord(0, 0), MapSize(size, size)),
                preset='temperate',
                name='Maiden Lane',
                radius=15,
                lon_span=10.0,
                num_plates=8,
                subdivisions=3,
                ocean_fraction=0.3,
                oceanic_sides=['N'],
                terrain_age='young',
                formation_type='ridge',
                elevation_scale=1.5,
                erosion_age=0.1,
                num_lakes=0,
                seed=23,
                debug=True
            )

            self.terr = world.terrain
        self.grid: HexGrid = self.terr.hexGrid
        self.referenceIndex: int = self.grid.middle
        self.grid.adjustRadius(radius)
        rivers = self.terr.carve_to_ocean(num_lakes=0)
        self.builder: SVGBuilder = self.grid.builder
        self.basins: DrainageBasins = world.basins
        for i in range(len(self.grid.hexes)):
            self.grid.hexes[i].label = str(i)

        fy: FoodYield = FoodYield(self.terr, self.basins)
        
        self.yields: np.ndarray = fy.compute()
        self.fy: FoodYield = fy


In [ ]:

@patch
def hp2i(self: GameParts, hexpos: HexPosition) -> int:
    return self.grid.hexposition_to_index(hexpos=hexpos, origin_index=self.referenceIndex)

@patch
def i2hp(self: GameParts, index: int) -> HexPosition:
   return self.grid.index_to_hexposition(index=index, origin_index=self.referenceIndex)
        

In [ ]:
@patch
def overlayContext(parts: GameParts, *, region=None, padding=1, radius=None,
                   corridors=None, extra_squads=None, extra_pieces=None):
    """Build a GameContext from a GameParts instance."""
    terrain = parts.terr
    if region is not None:
        terrain = terrain.zoom(region, padding=padding)

    grid = terrain.hexGrid
    if radius:
        grid.adjustRadius(radius)

    # Collect all squads and pieces, merging extras if provided
    squads = list(getattr(parts, 'squads', []))
    if extra_squads:
        squads.extend(extra_squads)

    pieces = []
    for sq in squads:
        pieces.extend(sq.alive if hasattr(sq, 'alive') else sq.pieces)
    if extra_pieces:
        pieces.extend(extra_pieces)

    return GameContext(
    terrain=terrain, grid=grid, builder=grid.builder,
    c2f=getattr(terrain, 'c2f', None),
    extras=dict(
        basins=parts.basins,
        coarse_basins=parts.basins,
        corridors=corridors or [],
        pieces=pieces,
        squads=squads,
    )
)

In [ ]:
showDemo = True

In [ ]:
showDemo = False

In [ ]:
myStuff = GameParts(radius=30)

In [ ]:
ctx = myStuff.overlayContext()
if showDemo:
    TerrainDisplay(
        CreamOverlay(stylized=True),
        RiverOverlay(max_width=4),
        FoodOverlay(),
        OceanWaveOverlay(
        num_waves=5,
        spacing=8,
        amplitude_start=3.0,
        amplitude_decay=0.6,
        wavelength=40,
        stroke_color="#4a7fb5",
        opacity_start=0.5,
        opacity_decay=0.7,
        stroke_width=1.2),
        ctx=ctx,
        debug = not showDemo
    )

In [ ]:
ctx = myStuff.overlayContext()
myStuff.builder.layers = []
if showDemo:
    TerrainDisplay(
        TerrainOverlay(),
        RiverOverlay(max_width=4),
        

        ctx=ctx,
        debug = not showDemo
    )

## MapPlace

In [ ]:
@dataclass
class MapPlace:
    location: HexPosition = HexPosition.origin()
    facing: HexPosition = HexPosition.W
    level: int = 0
    harvest: int = 0
    parent: GameParts | None = None
    tenure: int = 0  # turns spent at this location
    pantry: int = 0  # per-piece food storage


In [ ]:
@patch
def location(self:GameParts,location, facing = HexPosition.W):
    index = self.hp2i(location)
    retPlace=MapPlace(location,facing)
    retPlace.level = self.terr.elevationLevel(index)
    retPlace.harvest  = int(self.fy.tiers[index])
    retPlace.parent = self
    return retPlace

In [ ]:
@patch
def location_by_index(self:GameParts,index, facing = HexPosition.W):
    location = self.i2hp(index)
    return self.location(location,facing)

In [ ]:
myStuff.grid.middle, myStuff.grid.index_to_hexposition(myStuff.grid.middle), myStuff.i2hp(myStuff.grid.middle)

In [ ]:
place = myStuff.location_by_index(336)
place.level, place.harvest

In [ ]:
@patch
def sight(self:MapPlace) -> HexRegion:
    """All hex indices visible from this piece's location."""
  
    effective_sight = max(2,self.level)
      
    retRegion = HexRegion(
        hexes=set([self.parent.hp2i(hp) for hp in self.location.field_of_view( self.facing, effective_sight)]),
        hexGrid = self.parent.grid)

    return retRegion


In [ ]:
place.sight()

In [ ]:
place = myStuff.location_by_index(336)
place.level, place.sight()

In [ ]:
place = myStuff.location_by_index(273)
place.level, place.harvest

## Pieces

In [ ]:
LIFESPAN: dict[PieceType, int] = {
    PieceType.PAWN:   6,
    PieceType.BISHOP: 4,
    PieceType.KNIGHT: 1,
    PieceType.QUEEN:  8,
    PieceType.KING:   999,
}

# after the strategy
LIFESPAN[PieceType.PAWN]   = 6
LIFESPAN[PieceType.BISHOP] = 3
LIFESPAN[PieceType.KNIGHT] = 2
LIFESPAN[PieceType.QUEEN]  = 6
LIFESPAN[PieceType.KING]   = 999


In [ ]:
@dataclass
class Piece:
    id:str = ""
    flag:CountryFlag | None = None
    type:PieceType = PieceType.PAWN
    place:MapPlace | None = None
    name:str = ""
    year:int = 1900
    lantern = StyleCSS("blank",fill="#007",opacity=0.3)

In [ ]:
@patch
def __eq__(self: Piece, other) -> bool:
    if not isinstance(other, Piece): return False
    return self.id == other.id

@patch
def __hash__(self: Piece) -> int:
    return hash(self.id)


In [ ]:
@patch
def visible_allies(self: Piece, squad: 'Squad') -> list['Piece']:
    """Allies in this piece's sight cone, sorted by birth year then name."""
    if self.place is None:
        return []
    my_cone = self.sight().hexes
    parts = self.place.parent
    allies = []
    for p in squad.pieces:
        if p is self or p.place is None:
            continue
        idx = parts.hp2i(p.place.location)
        if idx in my_cone:
            allies.append(p)
    # Oldest first (lowest year), then alphabetically
    allies.sort(key=lambda p: (p.year, p.name))
    return allies


In [ ]:


@patch
def fetch_avatar(self: Piece, style: str = "avataaars") -> str:
    """Fetch and cache DiceBear SVG avatar using piece name as seed."""
    if not hasattr(self, '_avatar_svg') or self._avatar_svg is None:
        seed = self.name or f"piece_{id(self)}"
        resp = httpx.get(f"https://api.dicebear.com/9.x/{style}/svg?seed={seed}")
        resp.raise_for_status()
        self._avatar_svg = resp.text
    return self._avatar_svg

@patch
def avatar_id(self: Piece) -> str:
    return f"avatar_{self.name or id(self)}"

@patch
def register_avatar(self: Piece, builder: SVGBuilder, style: str = "avataaars"):
    """Register avatar as a <symbol> definition on the builder (idempotent)."""
    aid = self.avatar_id()
    # Skip if already registered
    if any(getattr(d, 'attributes', {}).get('id') == aid for d in builder.definitions):
        return
    svg_raw = self.fetch_avatar(style)
    # Extract viewBox from the SVG
    
    vb_match = re.search(r'viewBox="([^"]+)"', svg_raw)
    vb = vb_match.group(1) if vb_match else "0 0 280 280"
    # Strip outer <svg> tags to get inner content
    inner = re.sub(r'<svg[^>]*>', '', svg_raw)
    inner = re.sub(r'</svg>\s*$', '', inner).strip()
    symbol = SVGDef("symbol", aid, inner, viewBox=vb)
    builder.add_definition(symbol)

@patch
def draw_avatar(self: Piece, center: MapCord, builder: SVGBuilder,
                size: float = 40, layer: str = None,
                opacity: float = 1.0, attrs: dict = None) -> str:
    """Render avatar at center with given size. Registers symbol if needed."""
    self.register_avatar(builder)
    aid = self.avatar_id()
    x, y = center.x - size / 2, center.y - size / 2
    extra = CountryFlag._render_attrs(attrs) if attrs else ""
    svg = (f'<use href="#{aid}" x="{x:.1f}" y="{y:.1f}" '
           f'width="{size:.1f}" height="{size:.1f}"{extra}/>')
    if opacity < 1.0:
        svg = f'<g opacity="{opacity:.2f}">{svg}</g>'
    if layer:
        builder.adjust(layer, svg)
    return svg


In [ ]:
class FoodStats(NamedTuple):
    """Per-piece-type food economics."""
    diet: int        # upkeep per turn while on board
    cost: int        # one-time cost to place
    harvest: float   # multiplier applied to tile's food yield tier
    drain: float     # cost imposed on EACH enemy piece in sight cone
    bonus_sight: int # extra sight range beyond elevation-based default

FOOD_TABLE: dict[PieceType, FoodStats] = {
    PieceType.PAWN:   FoodStats(diet=2, cost=7,  harvest=1.25, drain=0.25, bonus_sight=0),
    PieceType.BISHOP: FoodStats(diet=3, cost=14, harvest=0.75, drain=1.5,  bonus_sight=2),
    PieceType.KNIGHT: FoodStats(diet=2, cost=6,  harvest=0.0,  drain=3.0,  bonus_sight=0),
    PieceType.QUEEN:  FoodStats(diet=5, cost=27, harvest=0.0,  drain=4.25, bonus_sight=0),
    PieceType.KING:   FoodStats(diet=2, cost=0,  harvest=1.0,  drain=0.5,  bonus_sight=1),
}

#This is after the new strategies to even them
FOOD_TABLE[PieceType.PAWN]   = FoodStats(diet=1, cost=3,  harvest=1.25, drain=1.0,  bonus_sight=0)
FOOD_TABLE[PieceType.BISHOP] = FoodStats(diet=3, cost=9,  harvest=1.0,  drain=0.75, bonus_sight=1)
FOOD_TABLE[PieceType.KNIGHT] = FoodStats(diet=2, cost=10, harvest=0.0,  drain=3.0,  bonus_sight=0)
FOOD_TABLE[PieceType.QUEEN]  = FoodStats(diet=4, cost=18, harvest=0.0,  drain=4.5,  bonus_sight=2)
FOOD_TABLE[PieceType.KING]   = FoodStats(diet=2, cost=0,  harvest=1.0,  drain=0.5,  bonus_sight=1)


@patch(as_prop=True)
def food(self: Piece) -> FoodStats:
    return FOOD_TABLE[self.type]


In [ ]:
@patch
def sight(self: Piece) -> HexRegion:
    """All hex indices visible from this piece's location."""
    bonus = self.food.bonus_sight
    effective_sight = max(2, self.place.level) + bonus
    
    return HexRegion(
        hexes=set([self.place.parent.hp2i(hp) 
                   for hp in self.place.location.field_of_view(self.place.facing, effective_sight)]),
        hexGrid=self.place.parent.grid)


## Squad

In [ ]:
@dataclass
class Squad:
    pieces: list[Piece] = field(default_factory=list)
    name: str = ""
    year: int = 1900
    animal: str = ""
    flag: CountryFlag | None = None
    countryName: str = ""
    storage:int = 100

    @classmethod
    def squads(cls, count: int = 4, flag: CountryFlag | None = None) -> list['Squad']:
        if flag is None:
            flag = CountryFlag.seaborn("husl", 1)[0]
        return [Squad(name=name, animal=animal, flag=flag) 
                for animal, name in flag.country_squads(count)]

In [ ]:
@patch
def make_squad(self: CountryFlag, piece_types: list[PieceType], year: int, seed: int = 0) -> Squad:
    """Create a Squad with animal name and age-appropriate named Pieces."""
    animal = self.animal_name()
    squad_nm = self.squad_name(animal, seed)
    
    rng = random.Random(hash((self.name, year, seed)))
    
    pieces = []
    for pt in piece_types:
        birth_year = year + rng.randint(20, 38)
        names = self.commonNames(year=birth_year)
        name = rng.choice(names)
        player = Piece(flag=self, type=pt, name=name, year=birth_year)
        player.id = ''.join(random.choices(string.ascii_letters, k=16))
        player.lantern = StyleCSS(f"{player.id}_lantern", fill=self.primary, opacity=0.3)
        pieces.append(player)

    
    return Squad(pieces=pieces, name=squad_nm, year=year, animal=animal, flag=self)


In [ ]:
@patch
def loadSquads(self:GameParts, year: int=1880, seed: int = 0,levels=3):
    self.flags = CountryFlag.seaborn("bright",levels=levels)
    self.squads = []
    for i,flag in enumerate(self.flags):
        roster = {
            PieceType.PAWN: 8, 
            PieceType.BISHOP: 4, 
            PieceType.QUEEN: 2, 
            PieceType.KING: 1, 
            PieceType.KNIGHT: 4
        }
        pieceKinds = [pt for pt, n in roster.items() for _ in range(n)]


        squad = flag.make_squad(pieceKinds, year=year, seed =seed) 
        self.squads.append(squad)

In [ ]:
myStuff.loadSquads()

In [ ]:
myStuff.squads[0].pieces[0]

In [ ]:
from fasthtml.common import *
from fasthtml.components import Uk_input_tag
from fasthtml.svg import *
from monsterui.all import *

In [ ]:
read_url("https://monsterui.answer.ai/cards/md")

In [ ]:

@patch
def __ft__(self: Piece):
    b = SVGBuilder()
    b.width, b.height = 60, 60
    self.draw_avatar(MapCord(30, 30), b, size=50, layer="avatar")

    return Card(
        DivCentered(
            NotStr(b.xml()),
            P(self.name, cls=(TextT.sm, TextT.medium)),
            P(f"{self.type.icon} · {self.year}", cls=TextPresets.muted_sm)),
        cls=CardT.hover)


In [ ]:
myStuff.loadSquads()
show(myStuff.squads[0].pieces[0])

In [ ]:
#show(myStuff.squads[0].pieces[0])

In [ ]:
@patch
def __ft__(self: Squad):
    # Header: flag banner + squad name + animal
    flag_builder = self.flag.flag_banner(width=210, height=120, wavy=True)

    # Animal icon
    ab = SVGBuilder()
    ab.width, ab.height = 250,250
    ab.adjust("animal", self.flag.animal_svg(
        size='large', animal=self.animal,
        center=MapCord(125, 125),
        piece_id=f"sq_{self.name}"))


    header = DivFullySpaced(
       # NotStr(flag_builder.xml()),
        H3(self.name),
        NotStr(ab.xml()),
        

        cls=TextPresets.muted_sm)

    # Group pieces by type
    from collections import defaultdict
    by_type = defaultdict(list)
    for p in self.pieces:
        by_type[p.type].append(p)

    sections = []
    for pt, pieces in by_type.items():
        # Section header with piece icon
        sec_header = DivLAligned(
            Span(pt.icon, cls=TextT.lg),
            H4(pt.value.title()),
            P(f"× {len(pieces)}", cls=TextPresets.muted_sm))
        # 4-wide grid of piece cards
        grid = Grid(*[p.__ft__() for p in pieces],
                    cols_sm=2, cols_md=4)
        sections.append(Div(sec_header, grid, cls='space-y-2'))

    return DivCentered(header,Card( *sections, cls='space-y-4'))


In [ ]:
@patch
def table(self: Squad):
    rows = []
    for p in self.pieces:
        loc = ""
        facing = ""
        if p.place and p.place.parent:
            loc = str(p.place.parent.hp2i(p.place.location))
            facing = str(p.place.facing.label)
        rows.append(Tr(
            Td(p.type.icon),
            Td(p.name, cls=TextT.medium),
            Td(str(p.year), cls=TextPresets.muted_sm),
            Td(loc, cls=TextPresets.muted_sm),
            Td(facing, cls=TextPresets.muted_sm),
        ))
    
    return Card(
        Table(
            Thead(Tr(Th(""), Th("Name"), Th("Born"), Th("Loc"), Th("Facing"))),
            Tbody(*rows),
        ),
        header=(H4(self.name), P(f"🐾 {self.animal} · {len(self.pieces)} pieces", cls=TextPresets.muted_sm)),
    )


In [ ]:
myStuff.loadSquads()
if showDemo:
    show(myStuff.squads[0])
else:
    show(myStuff.squads[0].table())

In [ ]:
unit = myStuff.squads[0].pieces[0]
unit.place = myStuff.location_by_index(273)
show(unit)

In [ ]:
unit.place.sight()

In [ ]:
unit.sight()

## Overlays

In [ ]:
def FogOverlay( fill="#ffffff", stroke="#cccccc", **kw) -> OverlaySpec:
    """White out hexes with no terrain data."""
    def render(ctx: OverlayContext) -> str:
        c2f = getattr(ctx, 'c2f', None)
        if not c2f: return ""
        mapped = {ni for indices in c2f.values() for ni in (indices if isinstance(indices, list) else [indices])}
        fog = StyleCSS("fog", fill=fill, stroke=stroke, stroke_width=1)
        ctx.builder.add_style(fog)
        for i in range(len(ctx.grid.hexes)):
            if i not in mapped:
                ctx.grid.hexes[i].style = fog
        return ""
    return OverlaySpec("fog", render, priority=6)


In [ ]:
sight = unit.sight()
ctx = myStuff.overlayContext(region=sight, padding=2, radius=30)
myStuff.builder.layers = []
TerrainDisplay(
    TerrainOverlay(),
    RiverOverlay(max_width=4),
    FogOverlay(),
    
    

    ctx=ctx,
    debug = not showDemo
)

In [ ]:
sight = unit.sight()
ctx = myStuff.overlayContext(region=sight, padding=2, radius=30)
myStuff.builder.layers = []
if showDemo:
    TerrainDisplay(
        TerrainOverlay(),
        RiverOverlay(max_width=4),
        FogOverlay(fill="purple"),
        

        ctx=ctx,
        debug = not showDemo
    )

In [ ]:
def LanternOverlay(**kw) -> OverlaySpec:
    """Render sight regions for placed pieces using their lantern style, plus facing bars."""
    def render(ctx) -> str:
        grid, builder = ctx.grid, ctx.builder
        N = len(grid.hexes)
        r = grid.radius
        parts = []

        for piece in ctx.pieces:
            if piece.place is None: continue
            builder.add_style(piece.lantern)

            # Sight region
            for idx in sorted(piece.sight().hexes):
                for fi in ctx.fine_indices(idx):
                    if 0 <= fi < N:
                        h = grid.hexes[fi]
                        parts.append(Hex(h.radius, h.center, piece.lantern, v=h.v).svg())

            # Facing bar at piece's location
            loc_idx = piece.place.parent.hp2i(piece.place.location)
            for fi in ctx.fine_indices(loc_idx):
                if 0 <= fi < N:
                    c = grid.hexes[fi].center
                    glyphs = DiagramGlyphs(piece.flag, size=r * 0.7)
                    glyphs.register_styles(builder)
                    dirs = HexPosition.directions()
                    facing_dir = dirs.index(piece.place.facing) if piece.place.facing in dirs else 0
                    parts.append(glyphs.facing_bar(c, facing_dir))

        return '\n'.join(parts)

    return OverlaySpec("lanterns", render, requires={'pieces'}, priority=62)


In [ ]:
def PieceOverlay(**kw) -> OverlaySpec:
    """Render all pieces using their flag's chess-piece SVG."""
    def render(ctx) -> str:
        if not ctx.pieces:
            return ""

        grid = ctx.grid
        N = len(grid.hexes)
        scale = grid.radius * 0.8 / 22.5
        parts = []

        for i, piece in enumerate(ctx.pieces):
            if piece.place is None or piece.flag is None:
                continue

            coarse_idx = piece.place.parent.hp2i(piece.place.location)
            for fi in ctx.fine_indices(coarse_idx):
                if 0 <= fi < N:
                    center = grid.hexes[fi].center
                    pid = f"pc_{piece.id}_{i}"

                    svg_str, pat_def = piece.flag.piece_svg(
                        piece.type,
                        MapCord(center.x, center.y),
                        scale=scale,
                        size='board',
                        piece_id=pid,
                    )

                    if pat_def is not None:
                        ctx.builder.add_definition(pat_def)

                    parts.append(svg_str)

        return '\n'.join(parts)

    return OverlaySpec("pieces", render, requires={'pieces'}, priority=80)


In [ ]:
unit = myStuff.squads[0].pieces[0]
unit.place = myStuff.location_by_index(273)
sight = unit.place.sight()
ctx = myStuff.overlayContext(region=sight, padding=2, radius=30)
myStuff.builder.layers = []

if showDemo:
    TerrainDisplay(
        TerrainOverlay(),
        RiverOverlay(max_width=4),
        FogOverlay(),
        LanternOverlay(),
        PieceOverlay(),
        pieces=[unit],

        ctx=ctx,
        debug = not showDemo
    )

In [ ]:
unit = myStuff.squads[0].pieces[0]
unit.place = myStuff.location_by_index(273)
unit.place.facing = HexPosition.SE
sight = unit.sight()
ctx = myStuff.overlayContext(region=sight, padding=2, radius=30)
myStuff.builder.layers = []
if showDemo:
    TerrainDisplay(
        TerrainOverlay(),
        RiverOverlay(max_width=4),
        FogOverlay(),
        LanternOverlay(),
        PieceOverlay(),
        pieces=[unit],
        ctx=ctx,
        debug = not showDemo
    )

In [ ]:
if showDemo:
    show(myStuff.squads[0])
else:
    show(myStuff.squads[0].table())

In [ ]:
def SquadNamesOverlay(font_size: int = 14, font_family: str = "Cinzel", **kw) -> OverlaySpec:
    """Render squad names at the centroid of their placed pieces."""
    
    def render(ctx) -> str:
        grid = ctx.grid
        N = len(grid.hexes)
        parts = []
        
        for squad in ctx.squads:
            if not squad.name:
                continue
            
            # Collect fine-grid indices for all placed pieces
            placed_fine = []
            for piece in squad.pieces:
                if piece.place is None:
                    continue
                coarse_idx = piece.place.parent.hp2i(piece.place.location)
                for fi in ctx.fine_indices(coarse_idx):
                    if 0 <= fi < N:
                        placed_fine.append(fi)
            
            if not placed_fine:
                continue
            
            # Centroid of placed piece centers
            cx = sum(grid.hexes[fi].center.x for fi in placed_fine) / len(placed_fine)
            cy = sum(grid.hexes[fi].center.y for fi in placed_fine) / len(placed_fine)
            
            # Label style from flag
            label_style = squad.flag.labelStyle(f"sqname_{squad.name.replace(' ', '_')}")
            ctx.builder.add_style(label_style)
            ctx.builder.add_font(font_family)
            
            # Offset below centroid so it doesn't sit on top of pieces
            label_y = cy + grid.radius * 1.8
            
            parts.append(
                f'<text x="{cx:.1f}" y="{label_y:.1f}" text-anchor="middle" '
                f'font-size="{font_size}" font-family="\'{font_family}\', sans-serif" '
                f'dominant-baseline="middle" class="{label_style.name}">'
                f'{squad.name}</text>'
            )
        
        return '\n'.join(parts)
    
    return OverlaySpec("squad_names", render, requires={'squads'}, priority=90)


## Moves

In [ ]:
class MoveError(Exception):
    """Raised when a Move fails validation."""
    pass

In [ ]:
class Rotation(NamedTuple):
    piece: Piece
    facing: HexPosition
class Placement(NamedTuple):
    piece: Piece
    location: MapPlace
    
@dataclass
class Move:
    rotations: list[Rotation] = field(default_factory=list)
    placement: Placement | None = None

    # Add occupied-hex check to Move validation
    @classmethod
    def from_squad(cls, squad: Squad, rotations: list[Rotation] = None, 
                placement: Placement | None = None) -> 'Move':
        rotations = rotations or []
        errors = []
        
        for r in rotations:
            if r.piece not in squad.pieces:
                errors.append(f"{r.piece.name}: not in squad '{squad.name}'")
            elif r.piece.place is None:
                errors.append(f"{r.piece.name}: cannot rotate — not yet placed")
        
        if placement is not None:
            if placement.piece not in squad.pieces:
                errors.append(f"{placement.piece.name}: not in squad '{squad.name}'")
            elif placement.piece.place is not None:
                errors.append(f"{placement.piece.name}: cannot place — already on the board")
        
        if placement is not None:
            rotated_pieces = {r.piece for r in rotations}
            if placement.piece in rotated_pieces:
                errors.append(f"{placement.piece.name}: appears in both rotation and placement")
        
        placed_allies = [p for p in squad.pieces if p.place is not None]
        if placement is not None and not placed_allies:
            if placement.piece.type != PieceType.KING:
                errors.append(f"{placement.piece.name}: first piece placed must be the King")
        
        # Sight check — use POST-rotation facing for rotated pieces
        if placement is not None and placement.piece in squad.pieces:
            if placed_allies:
                target_idx = placement.location.parent.hp2i(placement.location.location)
                
                # Build rotation lookup: piece -> new_facing
                rot_map = {r.piece: r.facing for r in rotations}
                
                visible = set()
                for ally in placed_allies:
                    facing = rot_map.get(ally, ally.place.facing)
                    bonus = ally.food.bonus_sight
                    effective_sight = max(2, ally.place.level) + bonus
                    sight_hexes = set(
                        placement.location.parent.hp2i(hp)
                        for hp in ally.place.location.field_of_view(facing, effective_sight)
                    )
                    visible |= sight_hexes
                
                if target_idx not in visible:
                    errors.append(f"{placement.piece.name}: target hex {target_idx} not in any ally's sight")
        
        # Check hex not already occupied
        if placement is not None and placement.location.parent is not None:
            target_idx = placement.location.parent.hp2i(placement.location.location)
            parts = placement.location.parent
            for sq in parts.squads:
                for p in sq.pieces:
                    if p.place is not None:
                        if parts.hp2i(p.place.location) == target_idx:
                            errors.append(f"{placement.piece.name}: hex {target_idx} already occupied by {p.name}")
        
        if errors:
            raise MoveError("; ".join(errors))
        
        return Move(rotations=rotations, placement=placement)


    #Move.from_squad = from_squad




In [ ]:
@patch
def apply(self: Move, parts: GameParts):
    """Apply this move to the board."""
    for r in self.rotations:
        if r.piece.place is None:
            raise MoveError(f"{r.piece.name}: can't rotate — not placed")
        r.piece.place.facing = r.facing
    
    if self.placement is not None:
        self.placement.piece.place = self.placement.location

In [ ]:
@dataclass
class MoveLog:
    parts: GameParts
    _df: pd.DataFrame = field(default_factory=lambda: pd.DataFrame(
        columns=['turn', 'squad', 'piece', 'action', 'detail']))
    
    def record(self, squad: Squad, move: Move):
        """Validate, apply, and log."""
        validated = Move.from_squad(squad, move.rotations, move.placement)
        validated.apply(self.parts)
        turn = len(self._df)
        
        rows = []
        for r in validated.rotations:
            rows.append(dict(turn=turn, squad=squad.name, piece=r.piece.name,
                             action='rotate', detail=r.facing.label))
        if validated.placement:
            p = validated.placement
            idx = self.parts.hp2i(p.location.location)
            rows.append(dict(turn=turn, squad=squad.name, piece=p.piece.name,
                             action='place', detail=str(idx)))
        
        self._df = pd.concat([self._df, pd.DataFrame(rows)], ignore_index=True)
    
    @property
    def df(self) -> pd.DataFrame: return self._df
    
    def for_squad(self, name: str) -> pd.DataFrame:
        return self._df[self._df.squad == name]
    
    def turn_count(self) -> int:
        return int(self._df.turn.max() + 1) if len(self._df) else 0


@patch
def food_income(self: Squad, parts: GameParts) -> float:
    """Total food harvested by all placed pieces."""
    return sum(p.food.harvest * p.place.harvest
               for p in self.pieces if p.place is not None)

@patch
def food_upkeep(self: Squad) -> float:
    """Total diet cost for all placed pieces."""
    return sum(p.food.diet for p in self.pieces if p.place is not None)

@patch
def food_drain_suffered(self: Squad, all_squads: list['Squad']) -> float:
    """Total drain imposed on this squad by enemy sight cones."""
    my_placed = {p.place.parent.hp2i(p.place.location)
                 for p in self.pieces if p.place is not None}
    total = 0.0
    for enemy_squad in all_squads:
        if enemy_squad is self:
            continue
        for enemy in enemy_squad.pieces:
            if enemy.place is None:
                continue
            cone = enemy.sight().hexes
            caught = len(my_placed & cone)
            total += enemy.food.drain * caught
    return total

@patch
def food_net(self: Squad, parts: GameParts, all_squads: list['Squad']) -> float:
    """Net food change this turn (before placement costs)."""
    return self.food_income(parts) - self.food_upkeep() - self.food_drain_suffered(all_squads)


## Squad Network

In [ ]:
@patch
def to_nx(self: Squad) -> nx.DiGraph:
    """Build directed sight graph as NetworkX DiGraph with food attributes."""
    G = nx.DiGraph()
    placed = [p for p in self.pieces if p.place is not None]
    parts = placed[0].place.parent if placed else None
    if not parts:
        return G
    
    for p in placed:
        idx = parts.hp2i(p.place.location)
        G.add_node(p.id, 
                   piece=p,  # back-reference
                   piece_type=p.type.value,
                   harvest=int(p.food.harvest * p.place.harvest),
                   diet=p.food.diet,
                   drain=p.food.drain,
                   cost=p.food.cost,
                   pantry=p.place.pantry,
                   hex_idx=idx,
                   is_sink=p.food.harvest == 0.0 or p.type == PieceType.KING,
                   is_king=p.type == PieceType.KING)
    
    for p in placed:
        cone = p.sight().hexes
        for other in placed:
            if other is p:
                continue
            idx = parts.hp2i(other.place.location)
            if idx in cone:
                G.add_edge(p.id, other.id)
    
    return G


In [ ]:
@patch
def supply_bottlenecks(self: Squad) -> list[Piece]:
    """Pieces whose removal disconnects the sight network."""
    G = self.to_nx().to_undirected()
    return [G.nodes[n]['piece'] for n in nx.articulation_points(G)]

@patch
def isolated_pieces(self: Squad) -> list[Piece]:
    """Pieces not connected to the king's component."""
    G = self.to_nx().to_undirected()
    king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
    if not king_id: return []
    king_component = nx.node_connected_component(G, king_id)
    return [G.nodes[n]['piece'] for n in G.nodes if n not in king_component]


In [ ]:
@patch
def food_sinks(self: Squad) -> list[Piece]:
    """Pieces that consume but don't produce — need to be fed by the network."""
    G = self.to_nx()
    return [G.nodes[n]['piece'] for n in G.nodes 
            if G.nodes[n]['is_sink']]

@patch
def food_sources(self: Squad) -> list[Piece]:
    """Pieces that can produce surplus (harvest > diet)."""
    G = self.to_nx()
    return [G.nodes[n]['piece'] for n in G.nodes 
            if G.nodes[n]['harvest'] > G.nodes[n]['diet']]

@patch
def flow_network(self: Squad) -> dict:
    """Analyse food flow using the sight graph.
    
    Returns dict with:
        G: the DiGraph
        sinks: set of sink node ids
        reachable: dict[node_id, set[sink_id]] — which sinks each piece can reach
        disconnected: list[Piece] — pieces that can't reach ANY sink
        sink_layers: dict[sink_id, dict[node_id, int]] — BFS distance from each sink
    """
    G = self.to_nx()
    if not G.nodes:
        return {'G': G, 'sinks': set(), 'reachable': {}, 
                'disconnected': [], 'sink_layers': {}}
    
    sinks = {n for n, d in G.nodes(data=True) if d['is_sink']}
    
    # For each sink, find all nodes that can reach it (reverse BFS)
    R = G.reverse()
    sink_connected = {}
    for sink in sinks:
        sink_connected[sink] = nx.ancestors(R, sink) | {sink}
    
    # Per-node: which sinks can it reach?
    reachable = {n: set() for n in G.nodes}
    for sink, connected in sink_connected.items():
        for n in connected:
            reachable[n].add(sink)
    
    disconnected = [G.nodes[n]['piece'] for n in G.nodes 
                    if not reachable[n] and n not in sinks]
    
    # BFS layers from each sink (along reverse edges = upstream distance)
    sink_layers = {}
    for sink in sinks:
        sink_layers[sink] = nx.shortest_path_length(R, sink)
    
    return {
        'G': G,
        'sinks': sinks,
        'sink_connected': sink_connected,
        'sink_layers': sink_layers,
        'reachable': reachable,
        'disconnected': disconnected,
    }




In [ ]:
@patch
def resolve_flow(self: Squad):
    """Apply food flow using nx max-flow through sight network."""
    G = self.to_nx()
    if len(G) < 2:
        return
    
    king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
    
    # Build flow graph with super-source and super-sink
    F = nx.DiGraph()
    F.add_node('S')  # super-source: pieces with surplus
    F.add_node('T')  # super-sink: pieces in deficit
    
    for nid, data in G.nodes(data=True):
        pantry = data['piece'].place.pantry
        if pantry > 0:
            F.add_edge('S', nid, capacity=pantry)
        elif pantry < 0:
            F.add_edge(nid, 'T', capacity=-pantry)
    
    # Sight edges = relay capacity (effectively unlimited)
    for u, v in G.edges():
        F.add_edge(u, v, capacity=999)
        F.add_edge(v, u, capacity=999)  # sharing is bidirectional along sight
    
    if F.out_degree('S') == 0 or F.in_degree('T') == 0:
        return
    
    flow_val, flow_dict = nx.maximum_flow(F, 'S', 'T')
    
    # Apply: surplus pieces give, deficit pieces receive
    for nid in G.nodes:
        p = G.nodes[nid]['piece']
        donated = flow_dict.get('S', {}).get(nid, 0)
        received = flow_dict.get(nid, {}).get('T', 0)
        p.place.pantry -= donated   # gave from surplus
        p.place.pantry += received  # covered deficit
    
    # Remaining surplus → king (if connected)
    # Compute undirected view ONCE, no deepcopy
    if king_id:
        U = G.to_undirected(as_view=True)
        king_piece = G.nodes[king_id]['piece']
        for nid, data in G.nodes(data=True):
            if nid == king_id:
                continue
            p = data['piece']
            if p.place.pantry > 0 and nx.has_path(U, nid, king_id):
                king_piece.place.pantry += p.place.pantry
                p.place.pantry = 0


@patch
def starve(self: Squad, parts: GameParts, all_squads: list['Squad']) -> list[Piece]:
    RANK_ORDER = {PieceType.PAWN: 0, PieceType.BISHOP: 1, 
                  PieceType.KNIGHT: 1, PieceType.QUEEN: 2, PieceType.KING: 99}
    
    eliminated = []
    while self.food_net(parts, all_squads) < 0:
        G = self.to_nx()
        king_id = next((nid for nid, d in G.nodes(data=True) if d['is_king']), None)
        
        placed = [p for p in self.pieces 
                  if p.place is not None and p.type != PieceType.KING]
        if not placed: break
        
        # Graph distance to king (disconnected = infinity)
        if king_id:
            lengths = nx.shortest_path_length(G.to_undirected(), king_id)
        else:
            lengths = {}
        
        def starve_key(p):
            deficit = p.food.diet - (p.food.harvest * p.place.harvest)
            rank = RANK_ORDER.get(p.type, 0)
            graph_dist = lengths.get(p.id, 999)  # disconnected = first to go
            return (-deficit, rank, -graph_dist)
        
        victim = sorted(placed, key=starve_key)[0]
        victim.place = None
        eliminated.append(victim)
    
    return eliminated


@patch
def king_alive(self: Squad) -> bool:
    """Is the king still on the board?"""
    return any(p.type == PieceType.KING and p.place is not None for p in self.pieces)


In [ ]:
@patch
def king_alive(self: Squad) -> bool:
    """Is the king still on the board?"""
    return any(p.type == PieceType.KING and p.place is not None for p in self.pieces)

In [ ]:
@patch
def orient_facings(self: Squad, parts: GameParts):
    """Fast role-based facing: sources→sinks, sinks→sources, king→centroid."""
    dirs = HexPosition.directions()
    placed = [p for p in self.pieces if p.place is not None]
    if len(placed) < 2: return
    
    # Classify
    sources = [p for p in placed if p.food.harvest > 0 and p.type != PieceType.KING]
    sinks   = [p for p in placed if p.food.harvest == 0]
    king    = next((p for p in placed if p.type == PieceType.KING), None)
    
    def best_facing_toward(piece, targets):
        """Pick the direction that puts the most targets in our sight cone."""
        if not targets: return piece.place.facing
        
        best_dir = piece.place.facing
        best_count = -1
        bonus = piece.food.bonus_sight
        eff = max(2, piece.place.level) + bonus
        
        for d in dirs:
            cone = set(parts.hp2i(hp) 
                       for hp in piece.place.location.field_of_view(d, eff))
            count = sum(1 for t in targets 
                        if parts.hp2i(t.place.location) in cone)
            if count > best_count:
                best_count = count
                best_dir = d
        
        return best_dir
    
    # Sources face toward sinks (+ king)
    sink_targets = sinks + ([king] if king else [])
    for p in sources:
        p.place.facing = best_facing_toward(p, sink_targets)
    
    # Sinks face toward sources
    for p in sinks:
        p.place.facing = best_facing_toward(p, sources)
    
    # King faces toward largest cluster of allies
    if king:
        king.place.facing = best_facing_toward(king, [p for p in placed if p is not king])


In [ ]:
@patch
def optimize_facings(self: Squad, parts: GameParts, max_iters: int = 3) -> int:
    """Greedy coordinate descent over facings. Returns number of changes made."""
    return self.orient_facings(parts)
    dirs = HexPosition.directions()
    placed = [p for p in self.pieces if p.place is not None]
    if len(placed) < 2: return 0
    
    total_changes = 0
    for _ in range(max_iters):
        changed = False
        
        # Heuristic 3: optimize bottlenecks and king-adjacent pieces first
        G = self.to_nx()
        U = G.to_undirected()
        king_id = next((n for n,d in G.nodes(data=True) if d['is_king']), None)
        
        bottlenecks = set(nx.articulation_points(U)) if len(G) > 1 else set()
        king_neighbors = set(U.neighbors(king_id)) if king_id else set()
        
        def priority(p):
            nid = p.id
            is_bn = nid in bottlenecks
            is_kn = nid in king_neighbors
            return (not is_bn, not is_kn, p.food.harvest == 0)  # False sorts first
        
        ordered = sorted(placed, key=priority)
        
        for piece in ordered:
            original = piece.place.facing
            best_facing = original
            best_score = _score_network(self, parts)
            
            for d in dirs:
                if d == original: continue
                piece.place.facing = d
                s = _score_network(self, parts)
                if s > best_score:
                    best_score = s
                    best_facing = d
            
            piece.place.facing = best_facing
            if best_facing != original:
                changed = True
                total_changes += 1
        
        if not changed: break
    
    return total_changes


def _score_network(squad: Squad, parts: GameParts) -> float:
    """Score a squad's current facing configuration."""
    G = squad.to_nx()
    if len(G) < 2: return 0.0
    
    king_id = next((n for n,d in G.nodes(data=True) if d['is_king']), None)
    if not king_id: return 0.0
    U = G.to_undirected()
    
    # 1. Connectivity — most important
    king_comp = nx.node_connected_component(U, king_id)
    n_connected = len(king_comp)
    n_total = len(G)
    connectivity = (n_connected / n_total) * 100
    
    # 2. Total harvest reachable to king via the network
    food_reach = sum(G.nodes[n]['harvest'] for n in king_comp)
    
    # 3. Enemy drain coverage
    drain_score = 0.0
    for n in G.nodes:
        p = G.nodes[n]['piece']
        if p.food.drain > 0:
            cone = p.sight().hexes
            for enemy_sq in parts.squads:
                if enemy_sq is squad: continue
                for ep in enemy_sq.pieces:
                    if ep.place is not None:
                        if parts.hp2i(ep.place.location) in cone:
                            drain_score += p.food.drain * 0.1
    
    # 4. Penalize bottlenecks (fragile network)
    bottleneck_penalty = len(list(nx.articulation_points(U))) * 5
    
    # 5. Bonus for redundant sight links (robustness)
    edge_bonus = U.number_of_edges() * 0.5
    
    return connectivity + food_reach + drain_score - bottleneck_penalty + edge_bonus


In [ ]:
@patch
def tick_lifespan(self: Squad) -> list[Piece]:
    """Increment tenure for placed pieces. Evict any that exceed lifespan."""
    evicted = []
    for p in self.pieces:
        if p.place is None or p.type == PieceType.KING:
            continue
        p.place.tenure += 1
        if p.place.tenure >= LIFESPAN.get(p.type, 4):
            p.place = None
            evicted.append(p)
    return evicted


@patch
def tick_food(self: Squad, parts: GameParts, all_squads: list['Squad']):
    """Per-piece food resolution with graph-based sharing."""
    placed = [p for p in self.pieces if p.place is not None]
    
    # 1. Harvest + pay diet
    for p in placed:
        p.place.pantry += int(p.food.harvest * p.place.harvest) - p.food.diet
    
    # 2. Sharing via max-flow through sight network
    G = self.to_nx()
    if len(G) > 1:
        F = nx.DiGraph()
        F.add_node('S')  # super-source
        F.add_node('T')  # super-sink
        for nid, data in G.nodes(data=True):
            pantry = data['piece'].place.pantry
            if pantry > 0:
                F.add_edge('S', nid, capacity=pantry)
            elif pantry < 0:
                F.add_edge(nid, 'T', capacity=-pantry)
        # Sight edges = unlimited sharing capacity
        for u, v in G.edges():
            F.add_edge(u, v, capacity=999)
            F.add_edge(v, u, capacity=999)  # sharing is bidirectional
        
        if F.out_degree('S') > 0 and F.in_degree('T') > 0:
            flow_val, flow_dict = nx.maximum_flow(F, 'S', 'T')
            for nid in G.nodes:
                sent = sum(v for k, v in flow_dict.get(nid, {}).items() if k != 'S' and k != 'T')
                received = sum(flow_dict.get(other, {}).get(nid, 0) 
                              for other in G.nodes if other != nid)
                G.nodes[nid]['piece'].place.pantry += (received - sent)
    
    # 3. Enemy drain
    for p in placed:
        cone = p.sight().hexes
        for enemy_squad in all_squads:
            if enemy_squad is self: continue
            for enemy in enemy_squad.pieces:
                if enemy.place is None: continue
                if parts.hp2i(enemy.place.location) in cone:
                    enemy.place.pantry -= int(p.food.drain)


In [ ]:
def FoodNetworkOverlay(
    edge_opacity=0.6, edge_dash="6,4",
    bottleneck_r_mult=1.4,
    **kw
) -> OverlaySpec:
    """Food sharing network using flag-themed glyphs and MapPaths."""
    
    def render(ctx) -> str:
        grid = ctx.grid
        N = len(grid.hexes)
        parts = []
        
        for squad in ctx.squads:
            G = squad.to_nx()
            if len(G) < 2:
                continue
            
            flag = squad.flag
            glyphs = DiagramGlyphs(flag, size=grid.radius * 0.5)
            glyphs.register_styles(ctx.builder)
            
            # Edge style from flag colors
            edge_style = StyleCSS(
                f"foodnet_edge_{flag.name}",
                fill="none", stroke=flag.darkPrimary,
                stroke_width=max(1.5, grid.radius * 0.06),
                stroke_linecap="round", opacity=edge_opacity)
            ctx.builder.add_style(edge_style)
            
            # Surplus/deficit styles
            surplus_style = StyleCSS(
                f"foodnet_surplus_{flag.name}",
                fill=flag.primary, stroke=flag.darkPrimary,
                stroke_width=1.5, opacity=0.85)
            deficit_style = StyleCSS(
                f"foodnet_deficit_{flag.name}",
                fill=flag.comp, stroke=flag.darkPrimary,
                stroke_width=1.5, opacity=0.85)
            ctx.builder.add_style(surplus_style)
            ctx.builder.add_style(deficit_style)
            
            # Bottleneck detection
            bottlenecks = set(nx.articulation_points(G.to_undirected()))
            
            # --- Edges as MapPath arrows ---
            seen_edges = set()
            for u, v in G.to_undirected().edges():
                key = tuple(sorted([u, v]))
                if key in seen_edges:
                    continue
                seen_edges.add(key)
                
                u_idx = G.nodes[u]['hex_idx']
                v_idx = G.nodes[v]['hex_idx']
                
                for fi_u in ctx.fine_indices(u_idx):
                    for fi_v in ctx.fine_indices(v_idx):
                        if 0 <= fi_u < N and 0 <= fi_v < N:
                            c1 = grid.hexes[fi_u].center
                            c2 = grid.hexes[fi_v].center
                            
                            # Shorten endpoints toward center (don't overlap nodes)
                            shrink = grid.radius * 0.4
                            dx = c2.x - c1.x
                            dy = c2.y - c1.y
                            dist = math.sqrt(dx*dx + dy*dy)
                            if dist > 0:
                                ux, uy = dx/dist, dy/dist
                                p1 = MapCord(c1.x + ux*shrink, c1.y + uy*shrink)
                                p2 = MapCord(c2.x - ux*shrink, c2.y - uy*shrink)
                            else:
                                p1, p2 = c1, c2
                            
                            path = MapPath([p1, p2], edge_style)
                            parts.append(
                                path.drawPolygon(f'stroke-dasharray="{edge_dash}"'))
            
            # --- Nodes ---
            for nid, data in G.nodes(data=True):
                piece = data['piece']
                harvest = data['harvest']
                diet = data['diet']
                net = harvest - diet
                idx = data['hex_idx']
                is_bottleneck = nid in bottlenecks
                
                for fi in ctx.fine_indices(idx):
                    if 0 <= fi < N:
                        c = grid.hexes[fi].center
                        r = grid.radius * 0.25
                        
                        # Bottleneck: warning ring
                        if is_bottleneck:
                            parts.append(glyphs.ring(
                                c, r * bottleneck_r_mult,
                                glyphs.danger_style()))
                        
                        # Node glyph based on role
                        if net > 0:
                            # Harvester: sheaf icon
                            parts.append(glyphs.sheaf(c, size=r * 2.5))
                        elif net < 0:
                            # Consumer: shield icon
                            parts.append(glyphs.shield(c, size=r * 2))
                        else:
                            # Break-even: simple ring
                            parts.append(glyphs.ring(c, r))
                        
                        # Food pip (red-green gradient circle)
                        pip_frac = (net + 5) / 10.0  # normalize roughly
                        pip_center = MapCord(c.x + r * 1.8, c.y - r * 1.8)
                        parts.append(glyphs.food_pip(
                            pip_center, pip_frac,
                            max_r=r * 0.8, min_r=r * 0.3))
                        
                        # Net label
                        label = f"{net:+d}" if net != 0 else "0"
                        parts.append(
                            f'<text x="{c.x:.1f}" y="{c.y + r * 2.5:.1f}" '
                            f'text-anchor="middle" font-size="{max(9, grid.radius * 0.3):.0f}" '
                            f'fill="{flag.darkPrimary}" font-weight="bold" '
                            f'opacity="0.8">{label}</text>')
        
        return '\n'.join(parts)
    
    return OverlaySpec("food_network", render, requires={'squads'}, priority=85)


In [ ]:
if showDemo:
    ctx = myStuff.overlayContext(region=sight, padding=2, radius=30)
    TerrainDisplay(
        TerrainOverlay(),
        RiverOverlay(max_width=4),
        FogOverlay(),
        LanternOverlay(),
        PieceOverlay(),
        FoodNetworkOverlay(),
        pieces=myStuff.squads[0].pieces,
        ctx=ctx,
        debug = not showDemo
    )


## TurnProcessor

In [ ]:
@dataclass
class TurnProcessor:
    parts: GameParts
    squads: list[Squad]
    log: MoveLog = None
    turn: int = 0
    game_over: bool = False
    winner: Squad | None = None
    
    def __post_init__(self):
        if self.log is None:
            self.log = MoveLog(self.parts)
    
    def _king(self, squad: Squad) -> Piece | None:
        return next((p for p in squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
    
    def resolve_move(self, squad: Squad, move: Move) -> dict:
        if self.game_over:
            raise MoveError("Game is over")
        
        king = self._king(squad)
        
        summary = {
            'squad': squad.name, 'turn': self.turn,
            'placement_cost': 0, 'placement_pantry': 0,
            'food_harvested': 0.0, 'food_diet': 0.0,
            'food_drain_dealt': 0.0,
            'starved': [], 'expired': [],
            'connected': 0, 'disconnected': 0,
            'king_pantry_before': king.place.pantry if king else 0,
            'king_pantry_after': 0,
            'pantries': {},
            'game_over': False, 'eliminated': None,
        }
        
        # 1. Validate & apply
        self.log.record(squad, move)
        
        # 2. Placement cost from king, seed pantry
        if move.placement is not None:
            piece = move.placement.piece
            cost = piece.food.cost
            king = self._king(squad)
            if king and piece.type != PieceType.KING:
                king.place.pantry -= cost
            piece.place.pantry = int(cost * 0.75)
            summary['placement_cost'] = cost
            summary['placement_pantry'] = piece.place.pantry
        
        # 3. Tick lifespans
        expired = squad.tick_lifespan()
        summary['expired'] = [p.name for p in expired]
        
        king = self._king(squad)
        placed = [p for p in squad.pieces if p.place is not None]
        
        # 4a. Harvest
        for p in placed:
            earned = int(p.food.harvest * p.place.harvest)
            p.place.pantry += earned
            summary['food_harvested'] += earned
        
        # 4b. Diet
        for p in placed:
            p.place.pantry -= p.food.diet
            summary['food_diet'] += p.food.diet
        
        # 4c. Enemy drain
        for p in placed:
            cone = p.sight().hexes
            for enemy_sq in self.squads:
                if enemy_sq is squad:
                    continue
                for enemy in enemy_sq.pieces:
                    if enemy.place is None:
                        continue
                    if self.parts.hp2i(enemy.place.location) in cone:
                        drain = int(p.food.drain)
                        enemy.place.pantry -= drain
                        summary['food_drain_dealt'] += drain
        
        # 5. Network flow — the nx magic
        G = squad.to_nx()
        squad.resolve_flow()
        
        # Track connectivity
        king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
        if king_id:
            U = G.to_undirected()
            connected = sum(1 for n in G.nodes if nx.has_path(U, n, king_id))
            summary['connected'] = connected
            summary['disconnected'] = len(G) - connected
        
        # 6. Starve pieces still in deficit
        RANK_ORDER = {
            PieceType.PAWN: 0, PieceType.KNIGHT: 1,
            PieceType.BISHOP: 2, PieceType.QUEEN: 3, PieceType.KING: 99,
        }
        king_hp = king.place.location if king else None
        starving = [p for p in placed 
                    if p.place is not None and p.place.pantry < 0 
                    and p.type != PieceType.KING]
        starving.sort(key=lambda p: (
            p.place.pantry,
            RANK_ORDER.get(p.type, 0),
            -(p.place.location.distance(king_hp) if king_hp else 0)
        ))
        for p in starving:
            p.place = None
            summary['starved'].append(p.name)
        
        # 7. Snapshot
        summary['pantries'] = {
            p.name: p.place.pantry for p in squad.pieces if p.place is not None
        }
        summary['king_pantry_after'] = king.place.pantry if king else 0
        
        
        # 8. Win condition
        if not squad.king_alive():
            summary['game_over'] = True
            summary['eliminated'] = squad.name
            self._check_game_over()
        
        # 8b. King death — bankrupt with no one left to sacrifice
        king = self._king(squad)
        if king and king.place.pantry < 0:
            # Can any connected piece cover the deficit?
            remaining = [p for p in squad.pieces 
                        if p.place is not None and p is not king]
            if not remaining:
                king.place = None
                summary['game_over'] = True
                summary['eliminated'] = squad.name
                self._check_game_over()

        return summary
    
    def play_round(self, moves: dict[Squad, Move]) -> list[dict]:
        summaries = []
        for squad in self.squads:
            if squad not in moves or not squad.king_alive():
                continue
            s = self.resolve_move(squad, moves[squad])
            summaries.append(s)
            if self.game_over:
                break
        self.turn += 1
        return summaries
    
    def _check_game_over(self):
        alive = [s for s in self.squads if s.king_alive()]
        if len(alive) <= 1:
            self.game_over = True
            self.winner = alive[0] if alive else None
    
    def status(self) -> pd.DataFrame:
        rows = []
        for s in self.squads:
            king = self._king(s)
            placed = [p for p in s.pieces if p.place is not None]
            G = s.to_nx()
            king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
            bottlenecks = len(list(nx.articulation_points(G.to_undirected()))) if len(G) > 1 else 0
            rows.append({
                'squad': s.name,
                'king_pantry': king.place.pantry if king else 0,
                'placed': len(placed),
                'bottlenecks': bottlenecks,
                'king': '👑' if s.king_alive() else '💀',
            })
        return pd.DataFrame(rows)


In [ ]:
@patch
def resolve_move(self:TurnProcessor, squad: Squad, move: Move) -> dict:
    if self.game_over:
        raise MoveError("Game is over")
    
    king = self._king(squad)
    
    summary = {
        'squad': squad.name, 'turn': self.turn,
        'placement_cost': 0, 'placement_pantry': 0,
        'food_harvested': 0.0, 'food_diet': 0.0,
        'food_drain_dealt': 0.0,
        'starved': [], 'expired': [],
        'connected': 0, 'disconnected': 0,
        'king_pantry_before': king.place.pantry if king else 0,
        'king_pantry_after': 0,
        'pantries': {},
        'game_over': False, 'eliminated': None,
    }
    
    # 1. Validate & apply
    self.log.record(squad, move)
    
    # 2. Placement cost from king, seed pantry
    if move.placement is not None:
        piece = move.placement.piece
        cost = piece.food.cost
        king = self._king(squad)
        if king and piece.type != PieceType.KING:
            king.place.pantry -= cost
        piece.place.pantry = int(cost * 0.75)
        summary['placement_cost'] = cost
        summary['placement_pantry'] = piece.place.pantry
    
    # 3. Tick lifespans
    expired = squad.tick_lifespan()
    summary['expired'] = [p.name for p in expired]
    
    king = self._king(squad)
    placed = [p for p in squad.pieces if p.place is not None]
    
    # 4a. Harvest
    for p in placed:
        earned = int(p.food.harvest * p.place.harvest)
        p.place.pantry += earned
        summary['food_harvested'] += earned
    
    # 4b. Diet
    for p in placed:
        p.place.pantry -= p.food.diet
        summary['food_diet'] += p.food.diet
    
    # 4c. Enemy drain
    for p in placed:
        cone = p.sight().hexes
        for enemy_sq in self.squads:
            if enemy_sq is squad:
                continue
            for enemy in enemy_sq.pieces:
                if enemy.place is None:
                    continue
                if self.parts.hp2i(enemy.place.location) in cone:
                    drain = int(p.food.drain)
                    enemy.place.pantry -= drain
                    summary['food_drain_dealt'] += drain
    
    # 5. Network flow — the nx magic
    G = squad.to_nx()
    squad.resolve_flow()
    
    # Track connectivity
    king_id = next((n for n, d in G.nodes(data=True) if d['is_king']), None)
    if king_id:
        U = G.to_undirected()
        connected = sum(1 for n in G.nodes if nx.has_path(U, n, king_id))
        summary['connected'] = connected
        summary['disconnected'] = len(G) - connected
    
    # 6. Starve pieces still in deficit
    RANK_ORDER = {
        PieceType.PAWN: 0, PieceType.KNIGHT: 1,
        PieceType.BISHOP: 2, PieceType.QUEEN: 3, PieceType.KING: 99,
    }
    king_hp = king.place.location if king else None
    starving = [p for p in placed 
                if p.place is not None and p.place.pantry < 0 
                and p.type != PieceType.KING]
    starving.sort(key=lambda p: (
        p.place.pantry,
        RANK_ORDER.get(p.type, 0),
        -(p.place.location.distance(king_hp) if king_hp else 0)
    ))
    for p in starving:
        p.place = None
        summary['starved'].append(p.name)
    
    # 7. Snapshot
    summary['pantries'] = {
        p.name: p.place.pantry for p in squad.pieces if p.place is not None
    }
    summary['king_pantry_after'] = king.place.pantry if king else 0
    
    
    # 8. Win condition
    if not squad.king_alive():
        summary['game_over'] = True
        summary['eliminated'] = squad.name
        self._check_game_over()
    
    # 8b. King death — bankrupt
    king = self._king(squad)
    if king and king.place.pantry <= 0:
        king.place = None
        summary['game_over'] = True
        summary['eliminated'] = squad.name
        self._check_game_over()

    return summary




Which functions should i remove from squads since resolve_move

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Looking at what `resolve_move` now handles inline, these Squad methods are redundant:

- **`food_income`** — harvest is done in step 4a
- **`food_upkeep`** — diet is done in step 4b
- **`food_drain_suffered`** — drain is done in step 4c (from the dealing side)
- **`food_net`** — just combined the above three
- **`starve`** — starvation logic is in step 6
- **`tick_food`** — all of 4a/4b/4c + flow are in resolve_move

These are **still used** by `resolve_move` and should stay:

- `tick_lifespan` · `resolve_flow` · `to_nx` · `king_alive` · `orient_facings` / `optimize_facings`

Did refactor correctly?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Looks correct! You've moved the deprecated functions (`food_income`, `food_upkeep`, `food_drain_suffered`, `food_net`, `starve`, `tick_food`) into **note** cells so they're no longer executed, and kept `king_alive` as a live **code** cell since `resolve_move` still calls it.

One thing to double-check: `resolve_flow` is called in step 5 of `resolve_move` — make sure that's still in a code cell (it is, in `_a006c806`). 👍

In [ ]:
@patch(as_prop=True)
def land_components(self: GameParts) -> dict[int, int]:
    """Map each land hex to its connected component ID. Computed once."""
    if hasattr(self, '_land_comp'):
        return self._land_comp
    
    comp = {}
    cid = 0
    for i in range(len(self.terr.elevations)):
        if self.terr.elevations[i] <= 0 or i in comp:
            continue
        frontier = {i}
        while frontier:
            cur = frontier.pop()
            if cur in comp:
                continue
            comp[cur] = cid
            for nb in self.grid.neighborsOf(cur):
                if nb not in comp and self.terr.elevations[nb] > 0:
                    frontier.add(nb)
        cid += 1
    self._land_comp = comp
    return comp

@patch
def land_connected(self: GameParts, indices: list[int]) -> bool:
    comp = self.land_components
    cids = {comp.get(i) for i in indices}
    return len(cids) == 1 and None not in cids


In [ ]:
@patch
def find_starts(self: GameParts, n: int, ring_radius: int = 3, 
                min_distance: int = 6, balance_tolerance: float = 0.3,
                top_k: int = 40) -> list[int]:
    grid, terr, tiers = self.grid, self.terr, self.fy.tiers
    n_hexes = len(terr.elevations)
    
    # 1. Richness scores
    scores = np.zeros(n_hexes)
    for i in range(n_hexes):
        if terr.elevations[i] <= 0: continue
        neighbors = grid.indices_in_range(i, ring_radius)
        scores[i] = sum(tiers[j] for j in neighbors if terr.elevations[j] > 0)
    
    # 2. Top candidates sorted by score
    ranked = sorted([(i, scores[i]) for i in range(n_hexes) if scores[i] > 0],
                    key=lambda x: -x[1])[:top_k]
    
    # 3. Precompute hex distances between candidates
    def hex_dist(a, b):
        return grid.index_to_hexposition(a).distance(grid.index_to_hexposition(b))
    
    # 4. Try combinations, best balanced first
    best_set, best_spread = None, float('inf')
    
    for combo in combinations(range(len(ranked)), n):
        idxs = [ranked[c][0] for c in combo]
        scs = [ranked[c][1] for c in combo]
        
        # Distance check
        if any(hex_dist(a, b) < min_distance 
               for a, b in combinations(idxs, 2)):
            continue
        
        # Connectivity check
        if not self.land_connected(idxs):
            continue
        
        # Balance check
        median = np.median(scs)
        if median == 0: continue
        spread = (max(scs) - min(scs)) / median
        
        if spread < best_spread:
            best_spread = spread
            best_set = list(zip(idxs, scs))
            if spread <= balance_tolerance:
                break
    
    if best_set is None:
        raise ValueError(f"Could not find {n} balanced land-connected positions")
    
    result = [idx for idx, _ in best_set]
    print(f"Start positions (spread={best_spread:.2f}):")
    for idx, score in best_set:
        print(f"  Hex {idx}: richness={score:.0f}, level={terr.elevationLevel(idx)}, tier={tiers[idx]}")
    
    return result


In [ ]:
playerCount = 5
myStuff = GameParts(radius=30, size=450)
myStuff.grid.adjustRadius(20)
myStuff.loadSquads( year=1985, seed = 23,levels=playerCount)
starts = myStuff.find_starts(n=playerCount, ring_radius=3, min_distance=6)
# Place each squad's king
for squad, start_idx in zip(myStuff.squads, starts):
    king = next(p for p in squad.pieces if p.type == PieceType.KING)
    loc = myStuff.location_by_index(start_idx)
    move = Move.from_squad(squad, placement=Placement(king, loc))
    move.apply(myStuff)

# Check it worked
if showDemo:
    for sq in myStuff.squads:
        show(sq.table())

## Best Moves

In [ ]:
@patch
def best_pawn_move(self: Squad, parts: GameParts) -> Move | None:
    """Find the best pawn placement: highest yield hex in or near ally sight.
    
    Returns a Move with optional rotation + pawn placement, or None if no pawns available.
    """
    # Find an unplaced pawn
    pawn = next((p for p in self.pieces 
                 if p.type == PieceType.PAWN and p.place is None), None)
    if pawn is None:
        return None
    
    placed = [p for p in self.pieces if p.place is not None]
    if not placed:
        return None
    
    grid = parts.grid
    terr = parts.terr
    tiers = parts.fy.tiers
    occupied = {parts.hp2i(p.place.location) 
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    
    # First: check all hexes visible from current sight (no rotation needed)
    current_visible = set()
    for ally in placed:
        current_visible |= ally.sight().hexes
    
    # Filter to valid land hexes not already occupied
    candidates = {idx for idx in current_visible 
                  if 0 <= idx < len(terr.elevations) 
                  and terr.elevations[idx] > 0 
                  and idx not in occupied}
    
    best_idx = None
    best_yield = -1
    best_rotation = None  # (piece, new_facing) if rotation needed
    
    # Check current sight first
    for idx in candidates:
        if tiers[idx] > best_yield:
            best_yield = tiers[idx]
            best_idx = idx
            best_rotation = None
    
    # Now try each piece × each facing to find even better options
    dirs = HexPosition.directions()
    for ally in placed:
        original_facing = ally.place.facing
        for new_facing in dirs:
            if new_facing == original_facing:
                continue
            # Temporarily compute sight with new facing
            bonus = ally.food.bonus_sight
            effective_sight = max(2, ally.place.level) + bonus
            new_sight = set(
                parts.hp2i(hp) 
                for hp in ally.place.location.field_of_view(new_facing, effective_sight)
            )
            for idx in new_sight:
                if (0 <= idx < len(terr.elevations) 
                    and terr.elevations[idx] > 0 
                    and idx not in occupied
                    and tiers[idx] > best_yield):
                    best_yield = tiers[idx]
                    best_idx = idx
                    best_rotation = Rotation(ally, new_facing)
    
    if best_idx is None:
        return None
    
    loc = parts.location_by_index(best_idx)
    rotations = [best_rotation] if best_rotation else []
    
    return Move.from_squad(self, rotations=rotations, 
                           placement=Placement(pawn, loc))


In [ ]:
@patch
def best_queen_move(self: Squad, parts: GameParts, 
                    stance: str = "defend") -> Move | None:
    """Place a queen strategically based on stance.
    
    stance='defend': prioritize king protection + ally coverage
    stance='attack': prioritize draining enemy pieces
    """
    queen = next((p for p in self.pieces 
                  if p.type == PieceType.QUEEN and p.place is None), None)
    if queen is None:
        return None
    
    placed = [p for p in self.pieces if p.place is not None]
    if not placed:
        return None
    
    king = next((p for p in placed if p.type == PieceType.KING), None)
    if king and king.place.pantry < queen.food.cost + 10:
        return None
    
    terr = parts.terr
    dirs = HexPosition.directions()
    occupied = {parts.hp2i(p.place.location) 
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    
    # Ally + enemy positions
    ally_idxs = {parts.hp2i(p.place.location) for p in placed}
    enemy_idxs = set()
    for sq in parts.squads:
        if sq is self: continue
        for p in sq.pieces:
            if p.place is not None:
                enemy_idxs.add(parts.hp2i(p.place.location))
    
    king_hp = king.place.location if king else None
    
    # All valid placement hexes
    current_visible = set()
    for ally in placed:
        current_visible |= ally.sight().hexes
    
    candidates = {idx for idx in current_visible
                  if 0 <= idx < len(terr.elevations)
                  and terr.elevations[idx] > 0
                  and idx not in occupied}
    
    best_idx = None
    best_facing = None
    best_score = -999
    
    for idx in candidates:
        hp = parts.i2hp(idx)
        loc = parts.location_by_index(idx)
        bonus = queen.food.bonus_sight
        eff_sight = max(2, loc.level) + bonus
        
        # Distance to king (lower = better for defense)
        king_dist = hp.distance(king_hp) if king_hp else 0
        
        for d in dirs:
            cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))
            caught_enemies = len(enemy_idxs & cone)
            allies_covered = len(ally_idxs & cone)
            
            if stance == "defend":
                # Fortress: protect the king, cover allies, still punish nearby enemies
                score = (
                    allies_covered * 3.0
                    - king_dist * 4.0
                    + caught_enemies * queen.food.drain * 1.0
                    + (5.0 if king_hp and parts.hp2i(king_hp) in cone else 0)  # bonus: can see king
                )
            else:  # "attack" — you miss 100% of the shots you don't take
                score = (
                    caught_enemies * queen.food.drain * 3.0
                    + allies_covered * 1.0
                    - king_dist * 1.0  # still prefer not to be stranded
                )
            
            if score > best_score:
                best_score = score
                best_idx = idx
                best_facing = d
    
    if best_idx is None:
        return None
    
    loc = parts.location_by_index(best_idx)
    loc.facing = best_facing
    
    # Rotation if target isn't currently visible
    rotations = []
    if best_idx not in current_visible:
        for ally in placed:
            b = ally.food.bonus_sight
            eff = max(2, ally.place.level) + b
            for d in dirs:
                cone = set(parts.hp2i(h) for h in ally.place.location.field_of_view(d, eff))
                if best_idx in cone:
                    rotations = [Rotation(ally, d)]
                    break
            if rotations:
                break
    
    return Move.from_squad(self, rotations=rotations,
                           placement=Placement(queen, loc))


In [ ]:
@patch
def best_bishop_move(self: Squad, parts: GameParts) -> Move | None:
    """Place a bishop using auto-detected role: relay, harvester, or frontier."""
    bishop = next((p for p in self.pieces 
                   if p.type == PieceType.BISHOP and p.place is None), None)
    if bishop is None:
        return None
    
    placed = [p for p in self.pieces if p.place is not None]
    if not placed:
        return None
    
    king = next((p for p in placed if p.type == PieceType.KING), None)
    if king and king.place.pantry < bishop.food.cost + 10:
        return None
    
    terr = parts.terr
    dirs = HexPosition.directions()
    occupied = {parts.hp2i(p.place.location) 
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not self
                  for p in sq.pieces if p.place is not None}
    
    ally_idxs = {parts.hp2i(p.place.location) for p in placed}
    
    # Current collective sight
    current_visible = set()
    for ally in placed:
        current_visible |= ally.sight().hexes
    
    # Land hex set for frontier scoring
    land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
    
    # --- Auto-detect role weights ---
    G = self.to_nx()
    U = G.to_undirected() if len(G) > 1 else nx.Graph()
    has_bottleneck = len(list(nx.articulation_points(U))) > 0 if len(G) > 1 else False
    isolated = self.isolated_pieces()
    king_pantry = king.place.pantry if king else 0
    food_tight = king_pantry < 30
    
    if has_bottleneck or len(isolated) > 0:
        # Network fragile — bridge it
        w_connect, w_drain, w_harvest, w_frontier = 15, 1, 1, 3
        role = "relay"
    elif food_tight:
        # Running low — earn food
        w_connect, w_drain, w_harvest, w_frontier = 5, 1, 8, 1
        role = "harvester"
    else:
        # Healthy — expand vision for future placements
        w_connect, w_drain, w_harvest, w_frontier = 5, 3, 2, 6
        role = "frontier"
    
    candidates = {idx for idx in current_visible
                  if idx in land_set and idx not in occupied}
    
    best_idx = None
    best_facing = None
    best_score = -999
    
    for idx in candidates:
        hp = parts.i2hp(idx)
        loc = parts.location_by_index(idx)
        bonus = bishop.food.bonus_sight
        eff_sight = max(2, loc.level) + bonus
        tier = int(parts.fy.tiers[idx])
        
        for d in dirs:
            cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))
            
            # Connectivity: allies this bishop can see
            allies_seen = len(ally_idxs & cone)
            
            # Drain: enemies caught
            enemies_caught = len(enemy_idxs & cone)
            
            # Harvest: net food over 4-turn lifespan
            harvest_per_turn = bishop.food.harvest * tier - bishop.food.diet
            lifetime_net = harvest_per_turn * LIFESPAN[PieceType.BISHOP] - bishop.food.cost
            
            # Frontier: new land hexes revealed that nobody currently sees
            new_land = len((cone & land_set) - current_visible)
            
            score = (allies_seen * w_connect
                     + enemies_caught * bishop.food.drain * w_drain
                     + lifetime_net * w_harvest
                     + new_land * w_frontier
                     + loc.level * 0.5)  # slight high-ground tiebreak
            
            if score > best_score:
                best_score = score
                best_idx = idx
                best_facing = d
    
    if best_idx is None:
        return None
    
    loc = parts.location_by_index(best_idx)
    loc.facing = best_facing
    
    # Rotation if target isn't in current sight
    rotations = []
    if best_idx not in current_visible:
        for ally in placed:
            b = ally.food.bonus_sight
            eff = max(2, ally.place.level) + b
            for d in dirs:
                cone = set(parts.hp2i(h) for h in ally.place.location.field_of_view(d, eff))
                if best_idx in cone:
                    rotations = [Rotation(ally, d)]
                    break
            if rotations:
                break
    
    return Move.from_squad(self, rotations=rotations,
                           placement=Placement(bishop, loc))


In [ ]:
@patch
def best_knight_move(self: Squad, parts: GameParts) -> Move | None:
    """Place a knight as a one-turn drain missile targeting enemy clusters."""
    knight = next((p for p in self.pieces 
                   if p.type == PieceType.KNIGHT and p.place is None), None)
    if knight is None:
        return None
    
    placed = [p for p in self.pieces if p.place is not None]
    if not placed:
        return None
    
    king = next((p for p in placed if p.type == PieceType.KING), None)
    if king and king.place.pantry < knight.food.cost + 5:
        return None  # don't fire blanks when we're broke
    
    terr = parts.terr
    dirs = HexPosition.directions()
    occupied = {parts.hp2i(p.place.location) 
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    
    # Build enemy position map with piece value for targeting priority
    PIECE_VALUE = {
        PieceType.KING: 20, PieceType.QUEEN: 10,
        PieceType.BISHOP: 4, PieceType.PAWN: 2, PieceType.KNIGHT: 1,
    }
    enemy_map = {}  # idx -> total value of enemies at that hex
    for sq in parts.squads:
        if sq is self:
            continue
        for p in sq.pieces:
            if p.place is not None:
                idx = parts.hp2i(p.place.location)
                enemy_map[idx] = enemy_map.get(idx, 0) + PIECE_VALUE.get(p.type, 1)
    
    if not enemy_map:
        return None  # no targets, save the food
    
    # Current collective sight
    current_visible = set()
    for ally in placed:
        current_visible |= ally.sight().hexes
    
    land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
    
    candidates = {idx for idx in current_visible
                  if idx in land_set and idx not in occupied}
    
    best_idx = None
    best_facing = None
    best_score = -999
    best_rotation = None
    
    # Also try with one rotation to unlock new hexes
    sight_configs = [(None, current_visible)]  # (rotation, visible_set)
    for ally in placed:
        original = ally.place.facing
        b = ally.food.bonus_sight
        eff = max(2, ally.place.level) + b
        for d in dirs:
            if d == original:
                continue
            new_cone = set(
                parts.hp2i(h) for h in ally.place.location.field_of_view(d, eff))
            # Recompute collective sight with this rotation
            expanded = (current_visible - ally.sight().hexes) | new_cone
            sight_configs.append((Rotation(ally, d), expanded))
    
    for rotation, visible in sight_configs:
        cands = {idx for idx in visible
                 if idx in land_set and idx not in occupied}
        
        for idx in cands:
            hp = parts.i2hp(idx)
            loc = parts.location_by_index(idx)
            eff_sight = max(2, loc.level)  # knight has 0 bonus_sight
            
            for d in dirs:
                cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff_sight))
                
                # Total drain value: enemies caught weighted by importance
                drain_value = sum(enemy_map.get(eidx, 0) 
                                  for eidx in (cone & set(enemy_map.keys())))
                enemies_hit = len(cone & set(enemy_map.keys()))
                
                # ROI: is draining worth the 6 food cost?
                total_drain = enemies_hit * knight.food.drain
                roi = total_drain - knight.food.cost - knight.food.diet
                
                if roi <= 0:
                    continue  # not worth deploying
                
                score = (drain_value * knight.food.drain * 2.0
                         + roi * 1.5
                         + loc.level * 0.3)  # high ground = bigger cone
                
                if score > best_score:
                    best_score = score
                    best_idx = idx
                    best_facing = d
                    best_rotation = rotation
    
    if best_idx is None:
        return None
    
    loc = parts.location_by_index(best_idx)
    loc.facing = best_facing
    
    rotations = [best_rotation] if best_rotation else []
    
    return Move.from_squad(self, rotations=rotations,
                           placement=Placement(knight, loc))


In [ ]:
# Reset and replay
myStuff = GameParts(radius=30, size=450)
myStuff.grid.adjustRadius(20)
myStuff.loadSquads(year=1950, seed=23, levels=playerCount)
starts = myStuff.find_starts(n=playerCount, ring_radius=3, min_distance=6)

for squad, start_idx in zip(myStuff.squads, starts):
    king = next(p for p in squad.pieces if p.type == PieceType.KING)
    loc = myStuff.location_by_index(start_idx)
    move = Move.from_squad(squad, placement=Placement(king, loc))
    move.apply(myStuff)
    king.place.pantry = 100

tp = TurnProcessor(myStuff, myStuff.squads)

for round_num in range(3):
    print(f"\n=== Pawn round {round_num + 1} ===")
    for squad in myStuff.squads:
        move = squad.best_pawn_move(myStuff)
        if move:
            summary = tp.resolve_move(squad, move)
            idx = myStuff.hp2i(move.placement.location.location) if move.placement else "?"
            print(f"  {squad.name}: → hex {idx}, "
                  f"king_pantry={summary['king_pantry_after']}, "
                  f"connected={summary['connected']}")

print(f"\n{tp.status()}")


In [ ]:
# Reset and replay
myStuff = GameParts(radius=30, size=450)
myStuff.grid.adjustRadius(20)
myStuff.loadSquads(year=1950, seed=23, levels=playerCount)
starts = myStuff.find_starts(n=playerCount, ring_radius=3, min_distance=6)

for squad, start_idx in zip(myStuff.squads, starts):
    king = next(p for p in squad.pieces if p.type == PieceType.KING)
    loc = myStuff.location_by_index(start_idx)
    move = Move.from_squad(squad, placement=Placement(king, loc))
    move.apply(myStuff)
    king.place.pantry = 100

tp = TurnProcessor(myStuff, myStuff.squads)

for round_num in range(3):
    print(f"\n=== Pawn round {round_num + 1} ===")
    for squad in myStuff.squads:
        move = squad.best_pawn_move(myStuff)
        if move:
            summary = tp.resolve_move(squad, move)
            idx = myStuff.hp2i(move.placement.location.location) if move.placement else "?"
            print(f"  {squad.name}: → hex {idx}, "
                  f"king_pantry={summary['king_pantry_after']}, "
                  f"connected={summary['connected']}")
    
    # Optimize facings after each round of placements
    print(f"  --- Optimizing facings ---")
    for squad in myStuff.squads:
        changes = squad.optimize_facings(myStuff)
        if changes:
            print(f"  {squad.name}: {changes} facing(s) adjusted")

print(f"\n{tp.status()}")


In [ ]:
# Module-level scorer functions (shared by overlay + panel)
def _score_pawn(idx, parts, squad, current_visible, occupied, land_set):
    if idx not in land_set or idx in occupied: return -999
    return int(parts.fy.tiers[idx])

def _score_bishop(idx, parts, squad, current_visible, occupied, land_set):
    if idx not in land_set or idx in occupied: return -999
    hp = parts.i2hp(idx)
    loc = parts.location_by_index(idx)
    bonus = FOOD_TABLE[PieceType.BISHOP].bonus_sight
    eff = max(2, loc.level) + bonus
    tier = int(parts.fy.tiers[idx])
    dirs = HexPosition.directions()
    ally_idxs = {parts.hp2i(p.place.location) for p in squad.pieces if p.place is not None}
    best = -999
    for d in dirs:
        cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff))
        allies_seen = len(ally_idxs & cone)
        new_land = len((cone & land_set) - current_visible)
        harvest_net = FOOD_TABLE[PieceType.BISHOP].harvest * tier - FOOD_TABLE[PieceType.BISHOP].diet
        score = allies_seen * 5 + new_land * 4 + harvest_net * 3 + loc.level * 0.5
        best = max(best, score)
    return best

def _score_knight(idx, parts, squad, current_visible, occupied, land_set):
    if idx not in land_set or idx in occupied: return -999
    hp = parts.i2hp(idx)
    loc = parts.location_by_index(idx)
    eff = max(2, loc.level)
    dirs = HexPosition.directions()
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not squad
                  for p in sq.pieces if p.place is not None}
    if not enemy_idxs: return -999
    best = -999
    for d in dirs:
        cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff))
        enemies_hit = len(cone & enemy_idxs)
        roi = enemies_hit * FOOD_TABLE[PieceType.KNIGHT].drain - FOOD_TABLE[PieceType.KNIGHT].cost
        if roi <= 0: continue
        best = max(best, roi + loc.level * 0.3)
    return best

def _score_queen(idx, parts, squad, current_visible, occupied, land_set):
    if idx not in land_set or idx in occupied: return -999
    hp = parts.i2hp(idx)
    loc = parts.location_by_index(idx)
    bonus = FOOD_TABLE[PieceType.QUEEN].bonus_sight
    eff = max(2, loc.level) + bonus
    dirs = HexPosition.directions()
    king = next((p for p in squad.pieces if p.type == PieceType.KING and p.place is not None), None)
    king_hp = king.place.location if king else None
    ally_idxs = {parts.hp2i(p.place.location) for p in squad.pieces if p.place is not None}
    enemy_idxs = {parts.hp2i(p.place.location)
                  for sq in parts.squads if sq is not squad
                  for p in sq.pieces if p.place is not None}
    king_dist = hp.distance(king_hp) if king_hp else 0
    best = -999
    for d in dirs:
        cone = set(parts.hp2i(h) for h in hp.field_of_view(d, eff))
        allies_covered = len(ally_idxs & cone)
        enemies_caught = len(enemy_idxs & cone)
        sees_king = 1 if king_hp and parts.hp2i(king_hp) in cone else 0
        score = (allies_covered * 3.0 - king_dist * 4.0
                 + enemies_caught * FOOD_TABLE[PieceType.QUEEN].drain
                 + sees_king * 5.0)
        best = max(best, score)
    return best

PLACEMENT_SCORERS = {
    PieceType.PAWN:   _score_pawn,
    PieceType.BISHOP: _score_bishop,
    PieceType.KNIGHT: _score_knight,
    PieceType.QUEEN:  _score_queen,
}


In [ ]:
def OptimalPlacementOverlay(top_n: int = 3, **kw) -> OverlaySpec:
    """Show top-N candidate hexes per piece type using flag-colored chess pieces."""
    
    def render(ctx) -> str:
        grid = ctx.grid
        N = len(grid.hexes)
        parts_list = []
        
        for squad in ctx.squads:
            placed = [p for p in squad.pieces if p.place is not None]
            if not placed: continue
            parts_obj = placed[0].place.parent
            terr = parts_obj.terr
            flag = squad.flag
            
            current_visible = set()
            for ally in placed:
                current_visible |= ally.sight().hexes
            land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
            occupied = {parts_obj.hp2i(p.place.location)
                        for sq in parts_obj.squads for p in sq.pieces if p.place is not None}
            
            for pt, scorer in PLACEMENT_SCORERS.items():
                unplaced = [p for p in squad.pieces if p.type == pt and p.place is None]
                if not unplaced: continue
                
                scored = []
                for idx in current_visible:
                    s = scorer(idx, parts_obj, squad, current_visible, occupied, land_set)
                    if s > -999:
                        scored.append((idx, s))
                if not scored: continue
                
                scored.sort(key=lambda x: -x[1])
                top = scored[:top_n]
                
                scale = grid.radius * 0.7 / 22.5
                
                for rank, (idx, score) in enumerate(top):
                    opacity = 0.8 - rank * 0.2
                    
                    for fi in ctx.fine_indices(idx):
                        if 0 <= fi < N:
                            c = grid.hexes[fi].center
                            
                            # Ghost ring background
                            r = grid.radius * 0.6
                            parts_list.append(
                                f'<circle cx="{c.x:.1f}" cy="{c.y:.1f}" r="{r:.1f}" '
                                f'fill="{flag.lightPrimary}" fill-opacity="{opacity * 0.5:.2f}" '
                                f'stroke="{flag.darkPrimary}" stroke-opacity="{opacity:.2f}" '
                                f'stroke-width="1.5" stroke-dasharray="3,2"/>')
                            
                            # Chess piece silhouette
                            svg_str, pat_def = flag.piece_svg(
                                pt, MapCord(c.x, c.y),
                                scale=scale, size='board',
                                piece_id=f"opt_{squad.name}_{pt.value}_{rank}_{fi}")
                            if pat_def is not None:
                                ctx.builder.add_definition(pat_def)
                            parts_list.append(
                                f'<g opacity="{opacity:.2f}">{svg_str}</g>')
                            
                            # Rank number (top-right)
                            parts_list.append(
                                f'<text x="{c.x + r * 0.7:.1f}" y="{c.y - r * 0.7:.1f}" '
                                f'text-anchor="middle" dominant-baseline="central" '
                                f'font-size="{max(9, grid.radius * 0.35):.0f}" '
                                f'fill="{flag.darkPrimary}" font-weight="bold" '
                                f'opacity="{opacity:.2f}">{rank + 1}</text>')
                            
                            # Hex index (bottom)
                            parts_list.append(
                                f'<text x="{c.x:.1f}" y="{c.y + r * 1.1:.1f}" '
                                f'text-anchor="middle" dominant-baseline="central" '
                                f'font-size="{max(7, grid.radius * 0.28):.0f}" '
                                f'fill="{flag.darkPrimary}" '
                                f'opacity="{opacity * 0.7:.2f}">#{idx}</text>')
        
        return '\n'.join(parts_list)
    
    return OverlaySpec("optimal_placement", render, requires={'squads'}, priority=75)


# HexMagic Game — Architecture Reference

## Overview

A hex-based strategy game where squads of chess-typed pieces compete for territory on a procedurally generated terrain map. The core mechanic is **food economics** — pieces harvest food from tiles, share it along directed sight networks, and drain enemy resources. The king's pantry is the squad treasury; lose the king and you lose the game.

---

## Core Data Structures

### GameParts
The central game state container.
- `terr: TerraDemo` — procedural terrain (elevations, climate, precipitation)
- `grid: HexGrid` — hex layout with coordinate conversions
- `basins: DrainageBasins` — river/watershed data
- `fy: FoodYield` — per-hex food tier (0–7), computed from temperature × water × soil
- `squads: list[Squad]` — all players
- `flags: list[CountryFlag]` — visual identity per squad
- `referenceIndex: int` — grid center, used for `HexPosition ↔ index` conversion

Key methods:
- `hp2i(HexPosition) → int` — hex position to grid index
- `i2hp(int) → HexPosition` — grid index to hex position
- `location_by_index(int) → MapPlace` — creates a MapPlace with elevation level and food tier
- `find_starts(n, ring_radius, min_distance, balance_tolerance)` — finds n balanced king start positions
- `land_components` (property) — connected component IDs for land hexes
- `land_connected(indices)` — checks all indices share one land component
- `overlayContext(region, padding, radius)` — builds a GameContext for SVG rendering

### MapPlace
Per-hex placement info for a piece.
- `location: HexPosition` — cube coordinates
- `facing: HexPosition` — one of 6 hex directions (W, E, NW, NE, SW, SE)
- `level: int` — elevation tier (used for sight range)
- `harvest: int` — food yield tier of this tile
- `parent: GameParts` — back-reference
- `tenure: int` — turns spent at this location
- `pantry: int` — per-piece food storage (the individual ledger)

### Piece
An individual game unit.
- `id: str` — unique 16-char random string
- `flag: CountryFlag` — visual identity
- `type: PieceType` — PAWN, BISHOP, KNIGHT, QUEEN, or KING
- `place: MapPlace | None` — None if not on board
- `name: str` — generated name (era-appropriate)
- `year: int` — birth year
- `lantern: StyleCSS` — sight cone visual style

Key methods:
- `food` (property) → `FoodStats` — diet, cost, harvest multiplier, drain, bonus_sight
- `sight() → HexRegion` — hex indices in this piece's field of view (facing + level + bonus)
- `visible_allies(squad) → list[Piece]` — allies in sight cone, sorted oldest-first then alphabetically

### FoodStats (NamedTuple)

## Piece Strategy Reference

### ♟ Pawn — "Settler"
**Profile:** diet=2, cost=7, harvest=1.25, drain=0.25, sight=base, lifespan=6
**Role:** Primary food engine. Pawns pay for themselves and feed the network.
**Strategy:** Greedy yield-maximizer. Place on the highest food-tier hex visible.
- Break-even: tier ≥ 2 (1.25×2=2.5 > diet 2). Tier 5+ generates real surplus.
- Lifespan ROI: 6 turns × (harvest - diet) - 7 cost. Tier 4 → 6×(5-2)-7 = +11 net.
- No connectivity concern — pawns are plentiful and cheap enough to lose.
- Drain 0.25 is negligible; it's a side effect, not a goal.
**Why greedy works:** Pawns are the squad's income. Every turn without a pawn
  harvesting is food left on the table. Fancy positioning loses to "biggest number."

### ♝ Bishop — "Swiss Army Knife"  
**Profile:** diet=3, cost=14, harvest=0.75, drain=1.5, sight=base+2, lifespan=4
**Role:** Adapts to what the network needs: relay, harvester, or frontier scout.
**Strategy:** Auto-detect role from network health, then score candidates.
- **Relay** (fragile network): weight allies_seen ×15. Bridge articulation points.
- **Harvester** (food tight): weight lifetime_net ×8. Need tier ≥ 4 to break even
  (0.75×4=3.0 = diet). Tier 5+ over 4 turns: 4×(3.75-3)-14 = -11... still negative.
  Bishops are marginal harvesters — this role is damage control, not profit.
- **Frontier** (healthy): weight new_land ×6. The +2 sight is the best in the roster
  for revealing new hexes, unlocking better pawn placements next turn.
**Why auto-detect:** 4-turn lifespan means you can't afford a bad role pick. A relay
  bishop in a healthy network wastes 14 food; a frontier bishop in a fragile network
  lets the chain snap. Reading the graph tells you what's actually needed.

### ♞ Knight — "Guided Missile"
**Profile:** diet=2, cost=6, harvest=0.0, drain=3.0, sight=base, lifespan=1
**Role:** One-shot drain strike against enemy clusters. Fire and forget.
**Strategy:** ROI-gated targeting. Only deploy if drain value > cost + diet.
- Minimum viable target: 3 enemies hit → 9 drain dealt for 8 food spent.
- Target weighting: kings ×20, queens ×10, bishops ×4, pawns ×2. Aim for HVTs.
- Rotation search: worth rotating an ally to unlock a better firing angle, since
  the knight dies anyway — the ally's facing can be fixed next turn.
- ROI gate prevents waste: if no cluster is reachable, returns None and saves food.
**Why ROI-gated:** Knights are pure consumables. Without the gate, the AI would
  fire knights at lone pawns (3 drain for 8 cost = net loss). The gate ensures
  every knight deployed is a profitable trade.

### ♛ Queen — "Fortress"
**Profile:** diet=5, cost=27, harvest=0.0, drain=4.25, sight=base, lifespan=8
**Role:** Area denial and king protection. Defensive anchor with offensive upside.
**Strategy:** Stance-based scoring (defend vs attack).
- **Defend** (default): allies_covered ×3, king_distance ×-4, king_visible +5 bonus.
  The queen's 4.25 drain makes approaching the king extremely costly for enemies.
  Placed 1-2 hexes from king, she creates a "no-go zone."
- **Attack**: enemies_caught × drain ×3. Used when ahead on food and wanting to
  close out a weakened opponent.
- Affordability gate: king needs pantry > cost + 10 (= 37). Queens are expensive;
  deploying one while food-starved can cascade into starvation deaths.
**Why fortress-first:** At cost=27 and 8-turn lifespan, the queen is the biggest
  investment. Losing the king ends the game; the queen's drain aura is the best
  insurance policy. Offensive value is a bonus, not the primary mission.

### ♚ King — "Treasury"  
**Profile:** diet=2, cost=0, harvest=1.0, drain=0.5, sight=base+1, lifespan=∞
**Role:** Network hub and food bank. All surplus flows to the king's pantry.
**No best_move needed:** Placed at game start via find_starts(). Facing is set by
  optimize_facings() to cover the most allies.
- The king's pantry IS the squad's bank account. Every placement cost comes from here.
- harvest=1.0 means the king is a decent earner on high-tier hexes (often tier 0
  at start positions though — find_starts prioritizes *regional* richness).
- +1 sight helps maintain connectivity as the network grows.


- **diet** — upkeep per turn while on board
- **cost** — one-time placement cost (deducted from king's pantry)
- **harvest** — multiplier × tile food tier = food earned per turn
- **drain** — cost imposed on EACH enemy piece caught in this piece's sight cone
- **bonus_sight** — extra sight range beyond elevation-based default

### LIFESPAN
Turns before a piece expires: PAWN=4, BISHOP=3, KNIGHT=3, QUEEN=2, KING=999.

### Squad
A player's army.
- `pieces: list[Piece]` — all pieces (placed and unplaced)
- `name, animal, flag` — identity
- No `storage` field — **king's pantry IS the squad treasury**

Key methods:
- `to_nx() → nx.DiGraph` — the central sight/flow graph (see below)
- `resolve_flow()` — apply max-flow food sharing via nx
- `best_pawn_move(parts) → Move` — greedy: find highest-yield hex in/near sight
- `king_alive() → bool`
- `tick_lifespan() → list[Piece]` — increment tenure, evict expired
- `supply_bottlenecks() → list[Piece]` — articulation points in sight graph
- `isolated_pieces() → list[Piece]` — pieces not connected to king's component
- `food_sinks() / food_sources()` — pieces that consume vs produce

---

## The Sight/Supply Graph (NetworkX)

`Squad.to_nx()` builds a **directed graph** where:
- **Nodes** = placed pieces, with attributes: `piece`, `piece_type`, `harvest` (pre-computed), `diet`, `drain`, `cost`, `pantry`, `hex_idx`, `is_sink`, `is_king`
- **Edges** = A → B means piece A can **see** piece B (B is in A's sight cone)
- Edges represent potential food transfer paths

### Key graph queries:
- `nx.articulation_points(G.to_undirected())` — supply bottlenecks
- `nx.has_path(G.to_undirected(), a, b)` — connectivity check
- `nx.maximum_flow(F, 'S', 'T')` — food distribution (super-source/super-sink formulation)
- `nx.node_connected_component(G, king_id)` — king's connected component

### Flow algorithm (`resolve_flow`):
1. Build auxiliary flow graph `F` with super-source `S` and super-sink `T`
2. Pieces with `pantry > 0` get edge `S → piece` (capacity = surplus)
3. Pieces with `pantry < 0` get edge `piece → T` (capacity = deficit)
4. Sight edges get capacity 999 (unlimited relay), bidirectional
5. Run `nx.maximum_flow(F, 'S', 'T')`
6. Apply flow: donors lose, receivers gain
7. Remaining surplus from connected pieces flows to king

---

## Moves

### Move
- `rotations: list[Rotation]` — (piece, new_facing) pairs for already-placed pieces
- `placement: Placement | None` — (piece, MapPlace) for a new piece entering the board

### Validation (`Move.from_squad`):
1. Rotated pieces must belong to squad and be placed
2. Placed piece must belong to squad and NOT be placed yet
3. A piece can't appear in both rotation and placement
4. First piece placed must be KING
5. Placement target must be in **post-rotation** sight of at least one ally
6. Target hex must not be occupied by any piece

### MoveLog
Records all moves as a DataFrame: turn, squad, piece, action (rotate/place), detail.

---

## Turn Processing

### TurnProcessor.resolve_move(squad, move) → dict

Phase order:
1. **Validate & apply** move (rotations + placement)
2. **Placement cost** deducted from king's pantry; new piece gets 75% of cost as starting pantry
3. **Tick lifespans** — evict expired pieces
4. **Harvest** — each piece earns `harvest_mult × tile_tier`
5. **Diet** — each piece pays its diet from own pantry
6. **Enemy drain** — each of our pieces drains enemy pieces in its cone
7. **Network flow** — `Squad.to_nx()` + `resolve_flow()` redistributes food via max-flow
8. **Starvation** — pieces still in deficit (except king) are removed, prioritized by: worst deficit → lowest rank → furthest from king
9. **Win check** — if king is dead, squad is eliminated; last king standing wins

### Turn order between squads:
- You move → economy resolves
- Enemy A moves → economy resolves
- Enemy B moves → economy resolves
- Back to you (you've taken N-1 economy hits between your moves)

---

## Food Economy Mental Model

```
   🌾 Pawns (harvest oasis tiles)
       ↓ sight edge (facing toward bishop)
   ♝ Bishops (relay + partial harvest)
       ↓ sight edge (facing toward king)
   ♚ King (treasury = pantry, feeds sinks)
       ↓ withdrawal
   ♞ Knights / ♛ Queens (sinks: drain enemies, no harvest)
```

- **Sources**: Pawns and Bishops on fertile tiles (harvest > diet)
- **Relays**: Any piece that sees both upstream and downstream
- **Sinks**: King (treasury), Queens, Knights (harvest = 0)
- **Cut a link**: downstream pieces lose access to shared food, must survive on own harvest
- **Drain**: each piece imposes `drain` cost on every enemy in its cone — stacks across multiple pieces

---

## Terrain & Food Yield

### FoodYield
Computes per-hex food tiers (0–7) from three factors:
- **Temperature** (climate zone → curve): subtropical peak, tundra low
- **Water** (precipitation + river flow, log-scale diminishing returns)
- **Soil** (type → fertility multiplier: alluvial best, granite worst)

Formula: `raw = temp_f × water_f × soil_f`, normalized to 0–1, quantized to tiers.

### find_starts
Finds balanced king placements:
1. Score each hex = sum of food tiers in ring of radius `ring_radius`
2. Take top `top_k` candidates
3. Try combinations, checking: minimum hex distance, land-connected, balance tolerance (spread relative to median)
4. Returns most balanced valid set

---

## Overlay System (SVG Rendering)

`TerrainDisplay` composites multiple `OverlaySpec` layers by priority:
- `TerrainOverlay` / `CreamOverlay` — base terrain colors
- `RiverOverlay` — water features
- `FoodOverlay` — dot-density pattern showing food yield tiers
- `FogOverlay` — whites out hexes outside a region (fog of war)
- `LanternOverlay` — sight cone visualization per piece
- `PieceOverlay` — chess piece SVG glyphs on the map
- `FoodNetworkOverlay` — sight graph edges, bottleneck markers, surplus/deficit indicators
- `OceanWaveOverlay` — decorative ocean waves

Each overlay gets a `GameContext` with: terrain, grid, builder, basins, pieces, squads.

---

## Pyomo Integration Notes (Future)

The `to_nx()` graph maps directly to Pyomo:
- `G.nodes` → `model.NODES` (Set)
- `G.edges` → `model.ARCS` (Set)
- Node attributes → `model.harvest`, `model.diet`, `model.drain` (Param dicts)
- Flow conservation: `inflow + harvest >= diet + outflow` per node
- Balance constraints: steady-state food across squads within tolerance
- The flow formulation is a standard min-cost network flow problem


In [ ]:
# After pawn rounds...
for squad in myStuff.squads:
    move = squad.best_queen_move(myStuff)
    if move:
        summary = tp.resolve_move(squad, move)
        print(f"  {squad.name}: 👑→♛ draining {summary.get('food_drain_dealt', 0)} food")


In [ ]:
for squad in myStuff.squads:
    placed = [p for p in squad.pieces if p.place is not None]
    if not placed: continue
    print(f"\n{squad.name}:")
    for p in placed:
        idx = myStuff.hp2i(p.place.location)
        tier = int(myStuff.fy.tiers[idx])
        in_enemy_cones = 0
        for esq in myStuff.squads:
            if esq is squad: continue
            for ep in esq.pieces:
                if ep.place is not None and idx in ep.sight().hexes:
                    in_enemy_cones += 1
        drain_taken = sum(ep.food.drain for esq in myStuff.squads if esq is not squad
                         for ep in esq.pieces if ep.place is not None 
                         and idx in ep.sight().hexes)
        net = int(p.food.harvest * tier) - p.food.diet - int(drain_taken)
        print(f"  {p.type.icon} {p.name}: pantry={p.place.pantry}, "
              f"tier={tier}, drain_taken={drain_taken:.0f}, net/turn={net:+d}"
              f"{' ⚠️' if p.place.pantry + net < 0 else ''}")


In [ ]:
@dataclass
class PlacementStrategy:
    """Defines piece placement priority order."""
    name: str
    order: list[str]  # method names in priority order
    
    BOTTOM_UP = None  # set after class
    TOP_DOWN = None

    def next_move(self, squad: Squad, parts: GameParts) -> Move | None:
        for method_name in self.order:
            fn = getattr(squad, method_name)
            move = fn(parts)
            if move is not None:
                return move
        return None

PlacementStrategy.BOTTOM_UP = PlacementStrategy(
    "Bottom-Up", ["best_pawn_move", "best_bishop_move", "best_queen_move"])
PlacementStrategy.TOP_DOWN = PlacementStrategy(
    "Top-Down", ["best_queen_move", "best_bishop_move", "best_pawn_move"])


In [ ]:
@dataclass
class PlacementStrategy:
    """Squad personality that shapes piece priority and stance thresholds."""
    name: str
    # Stance thresholds (how easily this personality shifts)
    defend_threat: int = 2       # enemy pieces near king to trigger defend
    attack_pantry: int = 60      # min king pantry to go offensive
    attack_placed: int = 4       # min pieces on board before attacking
    # Knight eagerness: lower = fires more readily
    knight_roi_mult: float = 1.0  # multiplied into knight ROI gate
    
    def assess(self, squad: Squad, parts: GameParts) -> str:
        king = next((p for p in squad.pieces 
                     if p.type == PieceType.KING and p.place is not None), None)
        if not king: return "defend"
        
        pantry = king.place.pantry
        placed = [p for p in squad.pieces if p.place is not None]
        G = squad.to_nx()
        U = G.to_undirected(as_view=True) if len(G) > 1 else nx.Graph()
        bottlenecks = len(list(nx.articulation_points(U))) if len(G) > 1 else 0
        
        king_hp = king.place.location
        threat_count = sum(
            1 for sq in parts.squads if sq is not squad
            for p in sq.pieces if p.place is not None
            and parts.i2hp(parts.hp2i(p.place.location)).distance(king_hp) <= 4
        )
        
        if threat_count >= self.defend_threat or bottlenecks >= 2:
            return "defend"
        elif pantry > self.attack_pantry and len(placed) >= self.attack_placed:
            return "attack"
        else:
            return "expand"
    
    def next_move(self, squad: Squad, parts: GameParts) -> Move | None:
        stance = self.assess(squad, parts)
        order = self.PLAYBOOKS[self.name].get(stance, self.PLAYBOOKS["balanced"][stance])
        
        for method_name, kwargs in order:
            fn = getattr(squad, method_name)
            move = fn(parts, **kwargs) if kwargs else fn(parts)
            if move is not None:
                return move
        return None
    
    # --- Playbooks: (method_name, kwargs) per stance per personality ---
    PLAYBOOKS = {
        "expansive": {
            "expand":  [("best_pawn_move", {}), ("best_pawn_move", {}), 
                        ("best_bishop_move", {}), ("best_queen_move", {"stance": "defend"})],
            "defend":  [("best_bishop_move", {}), ("best_pawn_move", {}),
                        ("best_queen_move", {"stance": "defend"})],
            "attack":  [("best_pawn_move", {}), ("best_bishop_move", {}),
                        ("best_knight_move", {}), ("best_queen_move", {"stance": "attack"})],
        },
        "aggressive": {
            "expand":  [("best_queen_move", {"stance": "defend"}), ("best_knight_move", {}),
                        ("best_bishop_move", {}), ("best_pawn_move", {})],
            "defend":  [("best_queen_move", {"stance": "defend"}), ("best_knight_move", {}),
                        ("best_bishop_move", {}), ("best_pawn_move", {})],
            "attack":  [("best_knight_move", {}), ("best_queen_move", {"stance": "attack"}),
                        ("best_knight_move", {}), ("best_bishop_move", {})],
        },
        "balanced": {
            "expand":  [("best_pawn_move", {}), ("best_bishop_move", {}),
                        ("best_queen_move", {"stance": "defend"})],
            "defend":  [("best_queen_move", {"stance": "defend"}), ("best_bishop_move", {}),
                        ("best_pawn_move", {})],
            "attack":  [("best_knight_move", {}), ("best_queen_move", {"stance": "attack"}),
                        ("best_bishop_move", {}), ("best_pawn_move", {})],
        },
        "turtle": {
            "expand":  [("best_pawn_move", {}), ("best_pawn_move", {}),
                        ("best_queen_move", {"stance": "defend"})],
            "defend":  [("best_queen_move", {"stance": "defend"}), 
                        ("best_queen_move", {"stance": "defend"}),
                        ("best_bishop_move", {}), ("best_pawn_move", {})],
            "attack":  [("best_bishop_move", {}), ("best_pawn_move", {}),
                        ("best_queen_move", {"stance": "defend"})],  # turtles never really attack
        },
    }

# --- Factory presets ---
PlacementStrategy.EXPANSIVE = PlacementStrategy(
    "expansive", defend_threat=3, attack_pantry=80, attack_placed=6)
PlacementStrategy.AGGRESSIVE = PlacementStrategy(
    "aggressive", defend_threat=1, attack_pantry=40, attack_placed=3, knight_roi_mult=0.7)
PlacementStrategy.BALANCED = PlacementStrategy(
    "balanced", defend_threat=2, attack_pantry=60, attack_placed=4)
PlacementStrategy.TURTLE = PlacementStrategy(
    "turtle", defend_threat=1, attack_pantry=999, attack_placed=99)


In [ ]:
def run_game(parts, strategies: dict[str, PlacementStrategy], rounds=15):
    """strategies maps squad.name → PlacementStrategy"""
    tp = TurnProcessor(parts, parts.squads)
    
    for r in range(rounds):
        if tp.game_over: break
        for squad in parts.squads:
            if not squad.king_alive(): continue
            strat = strategies[squad.name]
            move = strat.next_move(squad, parts)
            if move:
                summary = tp.resolve_move(squad, move)
        
        # Optimize facings every 3 rounds
        if (r + 1) % 3 == 0:
            for squad in parts.squads:
                squad.optimize_facings(parts)
        
        if r % 5 == 0:
            print(f"\n--- Round {r} ---")
            print(tp.status())
    
    print(f"\n=== FINAL (round {tp.turn}) ===")
    print(tp.status())
    return tp


In [ ]:
# Fresh game
myStuff = GameParts(radius=30, size=450)
myStuff.grid.adjustRadius(20)
myStuff.loadSquads(year=1950, seed=23, levels=playerCount)
starts = myStuff.find_starts(n=playerCount, ring_radius=3, min_distance=6)

for squad, start_idx in zip(myStuff.squads, starts):
    king = next(p for p in squad.pieces if p.type == PieceType.KING)
    loc = myStuff.location_by_index(start_idx)
    Move.from_squad(squad, placement=Placement(king, loc)).apply(myStuff)
    king.place.pantry = 100

# Assign different personalities
personality_rotation = [
    PlacementStrategy.EXPANSIVE,
    PlacementStrategy.AGGRESSIVE,
    PlacementStrategy.BALANCED,
    PlacementStrategy.TURTLE,
    PlacementStrategy.AGGRESSIVE,
]
strategies = {}
for i, sq in enumerate(myStuff.squads):
    strategies[sq.name] = personality_rotation[i % len(personality_rotation)]
    print(f"{sq.name}: {strategies[sq.name].name}")

tp = run_game(myStuff, strategies, rounds=15)

In [ ]:
import optuna

Lets think through how we would tune. We have the parameters of 
lifespan, diet, cost, harvest, drain, bonus sight.
There should be a minimal number of pawns we need to support a queen via a bishop. queens and knights should have good combat strength against bishops. Once a terriority has been "claimed" by a queen it should be hard to move in. a group of knights shouldn't be able to take out a queen unless they are flanking her.

Any other ideas we should put it?

In [ ]:
def score_lone_king_dies(parts):
    """A king alone on barren land with pantry 20 should die ~turn 10."""
    king_diet = FOOD_TABLE[PieceType.KING].diet
    king_harvest_mult = FOOD_TABLE[PieceType.KING].harvest
    
    tiers = parts.fy.tiers
    terr = parts.terr
    king_idx = next(i for i in range(len(tiers))
                    if terr.elevations[i] > 0 and tiers[i] == 0)
    tile_tier = int(tiers[king_idx])
    
    net_per_turn = int(king_harvest_mult * tile_tier) - king_diet
    if net_per_turn >= 0:
        return 0.0
    
    death_turn = 20 / abs(net_per_turn)  # smaller starting pantry
    
    target = 10
    deviation = abs(death_turn - target)
    return max(0.0, 1.0 - deviation / 10.0)


This was our first model

```python
def apply_params(trial_or_dict):
    """Apply tuned params to global FOOD_TABLE and LIFESPAN. 
    Accepts an Optuna trial or a plain dict."""
    if isinstance(trial_or_dict, dict):
        p = trial_or_dict
    else:
        t = trial_or_dict
        p = {
            # Pawn
            'pawn_diet':    t.suggest_int('pawn_diet', 1, 2),
            'pawn_cost':    t.suggest_int('pawn_cost', 3, 8),
            'pawn_harvest': t.suggest_float('pawn_harvest', 0.5, 1.5, step=0.25),
            'pawn_drain':   t.suggest_float('pawn_drain', 0.0, 1.0, step=0.25),
            'pawn_sight':   0,
            'pawn_life':    t.suggest_int('pawn_life', 3, 6),
            # Bishop
            'bishop_diet':    t.suggest_int('bishop_diet', 1, 3),
            'bishop_cost':    t.suggest_int('bishop_cost', 8, 16),
            'bishop_harvest': t.suggest_float('bishop_harvest', 0.25, 1.0, step=0.25),
            'bishop_drain':   t.suggest_float('bishop_drain', 0.5, 2.0, step=0.25),
            'bishop_sight':   t.suggest_int('bishop_sight', 1, 3),
            'bishop_life':    t.suggest_int('bishop_life', 3, 5),
            # Knight (raider: cheap, short-lived)
            'knight_diet':    t.suggest_int('knight_diet', 1, 3),
            'knight_cost':    t.suggest_int('knight_cost', 5, 12),
            'knight_harvest': 0.0,
            'knight_drain':   t.suggest_float('knight_drain', 1.0, 3.0, step=0.25),
            'knight_sight':   0,
            'knight_life':    t.suggest_int('knight_life', 1, 3),
            # Queen (fortress: expensive, long-lived)
            'queen_diet':    t.suggest_int('queen_diet', 3, 6),
            'queen_cost':    t.suggest_int('queen_cost', 15, 30),
            'queen_harvest': 0.0,
            'queen_drain':   t.suggest_float('queen_drain', 2.0, 5.0, step=0.25),
            'queen_sight':   t.suggest_int('queen_sight', 0, 2),
            'queen_life':    t.suggest_int('queen_life', 4, 8),
            # King (fixed)
            'king_diet': 2, 'king_cost': 0, 'king_harvest': 1.0,
            'king_drain': 0.5, 'king_sight': 1, 'king_life': 999,
        }
    
    global FOOD_TABLE, LIFESPAN
    for pt, key in [(PieceType.PAWN, 'pawn'), (PieceType.BISHOP, 'bishop'),
                    (PieceType.KNIGHT, 'knight'), (PieceType.QUEEN, 'queen'),
                    (PieceType.KING, 'king')]:
        FOOD_TABLE[pt] = FoodStats(
            diet=p[f'{key}_diet'], cost=p[f'{key}_cost'],
            harvest=p[f'{key}_harvest'], drain=p[f'{key}_drain'],
            bonus_sight=p[f'{key}_sight'])
        LIFESPAN[pt] = p[f'{key}_life']
    
    return p


def score_3pawns_support_queen(parts):
    """Objective 1: 3 pawns on tier 6 tiles + 1 bishop + 1 queen, 
    fed through the network. Queen should survive 'queen_life' turns."""
    # Find good tiles near a cluster
    tiers = parts.fy.tiers
    terr = parts.terr
    grid = parts.grid
    
    # Find a tier 6+ hex for king
    king_idx = next(i for i in range(len(tiers)) 
                    if terr.elevations[i] > 0 and tiers[i] >= 4)
    king_hp = parts.i2hp(king_idx)
    
    # Find nearby good hexes
    nearby = grid.indices_in_range(king_idx, 4)
    good = [i for i in nearby 
            if terr.elevations[i] > 0 and tiers[i] >= 5 and i != king_idx]
    good.sort(key=lambda i: -tiers[i])
    
    if len(good) < 4:
        return 0.0  # can't set up scenario
    
    # Make a temporary squad
    flag = parts.flags[0]
    pieces = [
        Piece(id='test_king', flag=flag, type=PieceType.KING, name='K'),
        Piece(id='test_p1', flag=flag, type=PieceType.PAWN, name='P1'),
        Piece(id='test_p2', flag=flag, type=PieceType.PAWN, name='P2'),
        Piece(id='test_p3', flag=flag, type=PieceType.PAWN, name='P3'),
        Piece(id='test_b1', flag=flag, type=PieceType.BISHOP, name='B1'),
        Piece(id='test_q1', flag=flag, type=PieceType.QUEEN, name='Q1'),
    ]
    sq = Squad(pieces=pieces, name='test_3p', flag=flag)
    
    # Place them
    pieces[0].place = parts.location_by_index(king_idx)
    pieces[0].place.pantry = 50
    for i in range(3):
        pieces[i+1].place = parts.location_by_index(good[i])
        pieces[i+1].place.pantry = 5
    pieces[4].place = parts.location_by_index(good[3])
    pieces[4].place.pantry = 5
    # Queen adjacent to bishop
    q_candidates = [j for j in grid.indices_in_range(good[3], 2)
                    if terr.elevations[j] > 0 
                    and j not in {king_idx} | set(good[:4])]
    if not q_candidates:
        return 0.0
    pieces[5].place = parts.location_by_index(q_candidates[0])
    pieces[5].place.pantry = 10
    
    # Point everyone toward each other
    old_squads = parts.squads
    parts.squads = [sq]
    sq.orient_facings(parts)
    
    # Simulate turns — count how many turns queen survives
    queen_survived = 0
    for turn in range(LIFESPAN[PieceType.QUEEN] + 2):
        placed = [p for p in pieces if p.place is not None]
        for p in placed:
            p.place.pantry += int(p.food.harvest * p.place.harvest) - p.food.diet
        if len(placed) > 1:
            sq.resolve_flow()
        expired = sq.tick_lifespan()
        
        # Check queen
        if pieces[5].place is not None and pieces[5].place.pantry >= 0:
            queen_survived += 1
        else:
            break
    
    # Clean up
    for p in pieces:
        p.place = None
    parts.squads = old_squads
    
    # Score: queen should survive her full lifespan
    target = LIFESPAN[PieceType.QUEEN]
    return min(queen_survived / target, 1.0)


def score_2knights_beat_queen(parts):
    """Objective 2: 2 knights flanking from different directions 
    should out-drain a queen (combined drain > queen drain)."""
    kd = FOOD_TABLE[PieceType.KNIGHT].drain
    qd = FOOD_TABLE[PieceType.QUEEN].drain
    
    # 2 knights flanking = 2 × knight_drain vs queen's 1 × queen_drain
    # Knights face different directions so queen can only drain 1 back
    knight_total_drain = 2 * kd       # both drain the queen
    queen_drain_back = qd * 1          # queen only sees 1 knight (facing one way)
    
    net_advantage = knight_total_drain - queen_drain_back
    # Want net_advantage > 0, ideally > 1
    if net_advantage <= 0:
        return 0.0
    return min(net_advantage / 2.0, 1.0)


def score_queen_beats_knight(parts):
    """Objective 3: 1v1, queen out-drains knight."""
    qd = FOOD_TABLE[PieceType.QUEEN].drain
    kd = FOOD_TABLE[PieceType.KNIGHT].drain
    
    # Head-to-head, both see each other
    # Queen wins if her drain > knight's drain
    advantage = qd - kd
    if advantage <= 0:
        return 0.0
    return min(advantage / 2.0, 1.0)


def score_bottomup_beats_topdown(parts):
    """Objective 4: Run a game, bottom-up strategy should outlast top-down."""
    # Fresh squads on the existing terrain
    parts.loadSquads(year=1950, seed=42, levels=2)
    starts = parts.find_starts(n=2, ring_radius=3, min_distance=6)
    
    for squad, start_idx in zip(parts.squads, starts):
        king = next(p for p in squad.pieces if p.type == PieceType.KING)
        loc = parts.location_by_index(start_idx)
        Move.from_squad(squad, placement=Placement(king, loc)).apply(parts)
        king.place.pantry = 100
    
    strategies = {
        parts.squads[0].name: PlacementStrategy.BOTTOM_UP,
        parts.squads[1].name: PlacementStrategy.TOP_DOWN,
    }
    
    tp = TurnProcessor(parts, parts.squads)
    bu_alive_turns = 0
    td_alive_turns = 0
    
    for r in range(20):
        if tp.game_over:
            break
        for squad in parts.squads:
            if not squad.king_alive():
                continue
            strat = strategies[squad.name]
            move = strat.next_move(squad, parts)
            if move:
                try:
                    tp.resolve_move(squad, move)
                except MoveError:
                    pass
        
        if parts.squads[0].king_alive():
            bu_alive_turns += 1
        if parts.squads[1].king_alive():
            td_alive_turns += 1
    
    # Clean up pieces
    for sq in parts.squads:
        for p in sq.pieces:
            p.place = None
    
    # Score: bottom-up should survive longer
    if bu_alive_turns > td_alive_turns:
        return 1.0
    elif bu_alive_turns == td_alive_turns:
        return 0.5
    return 0.0


def score_lone_king_dies(parts):
    """Objective 5: A king with starting pantry 100, no pieces, 
    should die around turn 8-12."""
    king_diet = FOOD_TABLE[PieceType.KING].diet
    king_harvest_mult = FOOD_TABLE[PieceType.KING].harvest
    
    # King on a mediocre tile (tier 2)
    tiers = parts.fy.tiers
    terr = parts.terr
    king_idx = next(i for i in range(len(tiers))
                    if terr.elevations[i] > 0 and tiers[i] <= 2)
    tile_tier = int(tiers[king_idx])
    
    net_per_turn = int(king_harvest_mult * tile_tier) - king_diet
    if net_per_turn >= 0:
        return 0.0  # king never dies alone — bad
    
    death_turn = 100 / abs(net_per_turn)
    
    # Want death around turn 8-12
    target = 10
    deviation = abs(death_turn - target)
    return max(0.0, 1.0 - deviation / 10.0)


def objective(trial):
    """Combined Optuna objective — maximize sum of all sub-scores."""
    p = apply_params(trial)
    
    # Use a consistent terrain
    parts = GameParts(radius=30, size=450)
    parts.grid.adjustRadius(20)
    parts.loadSquads(year=1950, seed=23, levels=2)
    
    s1 = score_3pawns_support_queen(parts)
    s2 = score_2knights_beat_queen(parts)
    s3 = score_queen_beats_knight(parts)
    s4 = score_bottomup_beats_topdown(parts)
    s5 = score_lone_king_dies(parts)
    
    # Weighted sum
    total = (s1 * 3.0 +   # sustain is most important
             s2 * 2.0 +   # flanking dynamic
             s3 * 2.0 +   # queen dominance
             s4 * 2.0 +   # strategy balance  
             s5 * 1.0)    # king pressure
    
    trial.set_user_attr('sustain', s1)
    trial.set_user_attr('flank', s2)
    trial.set_user_attr('queen_1v1', s3)
    trial.set_user_attr('strat', s4)
    trial.set_user_attr('king_dies', s5)
    
    return total


# Create and run the study
study = optuna.create_study(direction='maximize', study_name='hex_balance')
study.optimize(objective, n_trials=80, show_progress_bar=True)

# Show results
print(f"\nBest score: {study.best_value:.2f}")
print(f"Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print(f"\nSub-scores:")
for k in ['sustain', 'flank', 'queen_1v1', 'strat', 'king_dies']:
    print(f"  {k}: {study.best_trial.user_attrs[k]:.2f}")
```

This was us checking parameter
```python
# Apply best params and print the new tables
params = study.best_params | {
    'pawn_sight': 0, 'knight_harvest': 0.0, 'knight_sight': 0,
    'queen_harvest': 0.0,
    'king_diet': 2, 'king_cost': 0, 'king_harvest': 1.0,
    'king_drain': 0.5, 'king_sight': 1, 'king_life': 999,
}
apply_params(params)
print("FOOD_TABLE:")
for pt, fs in FOOD_TABLE.items():
    print(f"  {pt.value:8s}: diet={fs.diet} cost={fs.cost} harvest={fs.harvest} "
          f"drain={fs.drain} sight={fs.bonus_sight} life={LIFESPAN[pt]}")
```

FOOD_TABLE:
  pawn    : diet=2 cost=7 harvest=1.25 drain=0.25 sight=0 life=6
  bishop  : diet=3 cost=14 harvest=0.75 drain=1.5 sight=2 life=4
  queen   : diet=5 cost=27 harvest=0.0 drain=4.25 sight=0 life=8
  knight  : diet=2 cost=6 harvest=0.0 drain=3.0 sight=0 life=1
  king    : diet=2 cost=0 harvest=1.0 drain=0.5 sight=1 life=999

In [ ]:
def apply_params(trial_or_dict):
    """Apply tuned params to global FOOD_TABLE and LIFESPAN. 
    Accepts an Optuna trial or a plain dict."""
    if isinstance(trial_or_dict, dict):
        p = trial_or_dict
    else:
        t = trial_or_dict
        p = {
            # Pawn
            'pawn_diet':    t.suggest_int('pawn_diet', 1, 2),
            'pawn_cost':    t.suggest_int('pawn_cost', 3, 8),
            'pawn_harvest': t.suggest_float('pawn_harvest', 0.5, 1.5, step=0.25),
            'pawn_drain':   t.suggest_float('pawn_drain', 0.0, 1.0, step=0.25),
            'pawn_sight':   0,
            'pawn_life':    t.suggest_int('pawn_life', 3, 6),
            # Bishop
            'bishop_diet':    t.suggest_int('bishop_diet', 1, 3),
            'bishop_cost':    t.suggest_int('bishop_cost', 8, 16),
            'bishop_harvest': t.suggest_float('bishop_harvest', 0.25, 1.0, step=0.25),
            'bishop_drain':   t.suggest_float('bishop_drain', 0.5, 2.0, step=0.25),
            'bishop_sight':   t.suggest_int('bishop_sight', 1, 3),
            'bishop_life':    t.suggest_int('bishop_life', 3, 5),
            # Knight (raider: cheap, short-lived)
            'knight_diet':    t.suggest_int('knight_diet', 1, 3),
            'knight_cost':    t.suggest_int('knight_cost', 5, 12),
            'knight_harvest': 0.0,
            'knight_drain':   t.suggest_float('knight_drain', 1.0, 3.0, step=0.25),
            'knight_sight':   0,
            'knight_life':    t.suggest_int('knight_life', 1, 3),
            # Queen (fortress: expensive, long-lived)
            'queen_diet':    t.suggest_int('queen_diet', 3, 6),
            'queen_cost':    t.suggest_int('queen_cost', 15, 30),
            'queen_harvest': 0.0,
            'queen_drain':   t.suggest_float('queen_drain', 2.0, 5.0, step=0.25),
            'queen_sight':   t.suggest_int('queen_sight', 0, 2),
            'queen_life':    t.suggest_int('queen_life', 4, 8),
            # King (fixed)
            'king_diet': 2, 'king_cost': 0, 'king_harvest': 1.0,
            'king_drain': 0.5, 'king_sight': 1, 'king_life': 999,
        }
    
    global FOOD_TABLE, LIFESPAN
    for pt, key in [(PieceType.PAWN, 'pawn'), (PieceType.BISHOP, 'bishop'),
                    (PieceType.KNIGHT, 'knight'), (PieceType.QUEEN, 'queen'),
                    (PieceType.KING, 'king')]:
        FOOD_TABLE[pt] = FoodStats(
            diet=p[f'{key}_diet'], cost=p[f'{key}_cost'],
            harvest=p[f'{key}_harvest'], drain=p[f'{key}_drain'],
            bonus_sight=p[f'{key}_sight'])
        LIFESPAN[pt] = p[f'{key}_life']
    
    return p


def score_3pawns_support_queen(parts):
    """Objective 1: 3 pawns on tier 6 tiles + 1 bishop + 1 queen, 
    fed through the network. Queen should survive 'queen_life' turns."""
    # Find good tiles near a cluster
    tiers = parts.fy.tiers
    terr = parts.terr
    grid = parts.grid
    
    # Find a tier 6+ hex for king
    king_idx = next(i for i in range(len(tiers)) 
                    if terr.elevations[i] > 0 and tiers[i] >= 4)
    king_hp = parts.i2hp(king_idx)
    
    # Find nearby good hexes
    nearby = grid.indices_in_range(king_idx, 4)
    good = [i for i in nearby 
            if terr.elevations[i] > 0 and tiers[i] >= 5 and i != king_idx]
    good.sort(key=lambda i: -tiers[i])
    
    if len(good) < 4:
        return 0.0  # can't set up scenario
    
    # Make a temporary squad
    flag = parts.flags[0]
    pieces = [
        Piece(id='test_king', flag=flag, type=PieceType.KING, name='K'),
        Piece(id='test_p1', flag=flag, type=PieceType.PAWN, name='P1'),
        Piece(id='test_p2', flag=flag, type=PieceType.PAWN, name='P2'),
        Piece(id='test_p3', flag=flag, type=PieceType.PAWN, name='P3'),
        Piece(id='test_b1', flag=flag, type=PieceType.BISHOP, name='B1'),
        Piece(id='test_q1', flag=flag, type=PieceType.QUEEN, name='Q1'),
    ]
    sq = Squad(pieces=pieces, name='test_3p', flag=flag)
    
    # Place them
    pieces[0].place = parts.location_by_index(king_idx)
    pieces[0].place.pantry = 50
    for i in range(3):
        pieces[i+1].place = parts.location_by_index(good[i])
        pieces[i+1].place.pantry = 5
    pieces[4].place = parts.location_by_index(good[3])
    pieces[4].place.pantry = 5
    # Queen adjacent to bishop
    q_candidates = [j for j in grid.indices_in_range(good[3], 2)
                    if terr.elevations[j] > 0 
                    and j not in {king_idx} | set(good[:4])]
    if not q_candidates:
        return 0.0
    pieces[5].place = parts.location_by_index(q_candidates[0])
    pieces[5].place.pantry = 10
    
    # Point everyone toward each other
    old_squads = parts.squads
    parts.squads = [sq]
    sq.orient_facings(parts)
    
    # Simulate turns — count how many turns queen survives
    queen_survived = 0
    for turn in range(LIFESPAN[PieceType.QUEEN] + 2):
        placed = [p for p in pieces if p.place is not None]
        for p in placed:
            p.place.pantry += int(p.food.harvest * p.place.harvest) - p.food.diet
        if len(placed) > 1:
            sq.resolve_flow()
        expired = sq.tick_lifespan()
        
        # Check queen
        if pieces[5].place is not None and pieces[5].place.pantry >= 0:
            queen_survived += 1
        else:
            break
    
    # Clean up
    for p in pieces:
        p.place = None
    parts.squads = old_squads
    
    # Score: queen should survive her full lifespan
    target = LIFESPAN[PieceType.QUEEN]
    return min(queen_survived / target, 1.0)


def score_2knights_beat_queen(parts):
    """Objective 2: 2 knights flanking from different directions 
    should out-drain a queen (combined drain > queen drain)."""
    kd = FOOD_TABLE[PieceType.KNIGHT].drain
    qd = FOOD_TABLE[PieceType.QUEEN].drain
    
    # 2 knights flanking = 2 × knight_drain vs queen's 1 × queen_drain
    # Knights face different directions so queen can only drain 1 back
    knight_total_drain = 2 * kd       # both drain the queen
    queen_drain_back = qd * 1          # queen only sees 1 knight (facing one way)
    
    net_advantage = knight_total_drain - queen_drain_back
    # Want net_advantage > 0, ideally > 1
    if net_advantage <= 0:
        return 0.0
    return min(net_advantage / 2.0, 1.0)


def score_queen_beats_knight(parts):
    """Objective 3: 1v1, queen out-drains knight."""
    qd = FOOD_TABLE[PieceType.QUEEN].drain
    kd = FOOD_TABLE[PieceType.KNIGHT].drain
    
    # Head-to-head, both see each other
    # Queen wins if her drain > knight's drain
    advantage = qd - kd
    if advantage <= 0:
        return 0.0
    return min(advantage / 2.0, 1.0)


def score_bottomup_beats_topdown(parts):
    """Objective 4: Run a game, bottom-up strategy should outlast top-down."""
    # Fresh squads on the existing terrain
    parts.loadSquads(year=1950, seed=42, levels=2)
    starts = parts.find_starts(n=2, ring_radius=3, min_distance=6)
    
    for squad, start_idx in zip(parts.squads, starts):
        king = next(p for p in squad.pieces if p.type == PieceType.KING)
        loc = parts.location_by_index(start_idx)
        Move.from_squad(squad, placement=Placement(king, loc)).apply(parts)
        king.place.pantry = 100
    
    strategies = {
        parts.squads[0].name: PlacementStrategy.BOTTOM_UP,
        parts.squads[1].name: PlacementStrategy.TOP_DOWN,
    }
    
    tp = TurnProcessor(parts, parts.squads)
    bu_alive_turns = 0
    td_alive_turns = 0
    
    for r in range(20):
        if tp.game_over:
            break
        for squad in parts.squads:
            if not squad.king_alive():
                continue
            strat = strategies[squad.name]
            move = strat.next_move(squad, parts)
            if move:
                try:
                    tp.resolve_move(squad, move)
                except MoveError:
                    pass
        
        if parts.squads[0].king_alive():
            bu_alive_turns += 1
        if parts.squads[1].king_alive():
            td_alive_turns += 1
    
    # Clean up pieces
    for sq in parts.squads:
        for p in sq.pieces:
            p.place = None
    
    # Score: bottom-up should survive longer
    if bu_alive_turns > td_alive_turns:
        return 1.0
    elif bu_alive_turns == td_alive_turns:
        return 0.5
    return 0.0


def score_lone_king_dies(parts):
    """Objective 5: A king with starting pantry 100, no pieces, 
    should die around turn 8-12."""
    king_diet = FOOD_TABLE[PieceType.KING].diet
    king_harvest_mult = FOOD_TABLE[PieceType.KING].harvest
    
    # King on a mediocre tile (tier 2)
    tiers = parts.fy.tiers
    terr = parts.terr
    king_idx = next(i for i in range(len(tiers))
                    if terr.elevations[i] > 0 and tiers[i] <= 2)
    tile_tier = int(tiers[king_idx])
    
    net_per_turn = int(king_harvest_mult * tile_tier) - king_diet
    if net_per_turn >= 0:
        return 0.0  # king never dies alone — bad
    
    death_turn = 100 / abs(net_per_turn)
    
    # Want death around turn 8-12
    target = 10
    deviation = abs(death_turn - target)
    return max(0.0, 1.0 - deviation / 10.0)

In [ ]:
def score_lone_king_dies(parts):
    """A king alone on barren land with pantry 20 should die ~turn 10."""
    king_diet = FOOD_TABLE[PieceType.KING].diet
    king_harvest_mult = FOOD_TABLE[PieceType.KING].harvest
    
    tiers = parts.fy.tiers
    terr = parts.terr
    king_idx = next(i for i in range(len(tiers))
                    if terr.elevations[i] > 0 and tiers[i] == 0)
    tile_tier = int(tiers[king_idx])
    
    net_per_turn = int(king_harvest_mult * tile_tier) - king_diet
    if net_per_turn >= 0:
        return 0.0
    
    death_turn = 20 / abs(net_per_turn)  # smaller starting pantry
    
    target = 10
    deviation = abs(death_turn - target)
    return max(0.0, 1.0 - deviation / 10.0)


In [ ]:
def make_parts(seed):
    """Create a fresh GameParts with a specific seed."""
    random.seed(seed)
    np.random.seed(seed)
    parts = GameParts(radius=30, size=450)
    parts.grid.adjustRadius(20)
    return parts

def objective_multi(trial):
    p = apply_params(trial)
    
    scores = {k: [] for k in ['sustain', 'flank', 'queen_1v1', 'strat', 'king_dies']}
    
    for seed in MAP_SEEDS:
        parts = make_parts(seed)
        parts.loadSquads(year=1950, seed=seed, levels=2)
        
        scores['sustain'].append(score_3pawns_support_queen(parts))
        scores['flank'].append(score_2knights_beat_queen(parts))
        scores['queen_1v1'].append(score_queen_beats_knight(parts))
        scores['strat'].append(score_bottomup_beats_topdown(parts))
        scores['king_dies'].append(score_lone_king_dies(parts))
    
    s1 = sum(scores['sustain']) / len(MAP_SEEDS)
    s2 = sum(scores['flank']) / len(MAP_SEEDS)
    s3 = sum(scores['queen_1v1']) / len(MAP_SEEDS)
    s4 = sum(scores['strat']) / len(MAP_SEEDS)
    s5 = sum(scores['king_dies']) / len(MAP_SEEDS)
    
    total = (s1 * 3.0 + s2 * 2.0 + s3 * 2.0 + s4 * 2.0 + s5 * 1.0)
    
    trial.set_user_attr('sustain', s1)
    trial.set_user_attr('flank', s2)
    trial.set_user_attr('queen_1v1', s3)
    trial.set_user_attr('strat', s4)
    trial.set_user_attr('king_dies', s5)
    trial.set_user_attr('strat_std', float(np.std(scores['strat'])))
    trial.set_user_attr('sustain_std', float(np.std(scores['sustain'])))
    
    return total

simulator code

```python
# Pre-build map cache (run once)
MAP_CACHE = {}
for seed in MAP_SEEDS:
    random.seed(seed)
    np.random.seed(seed)
    parts = GameParts(radius=30, size=450)
    parts.grid.adjustRadius(20)
    MAP_CACHE[seed] = parts
    print(f"  Cached seed {seed}")

def fresh_game(seed, n_squads=2):
    """Return a GameParts with cached terrain + fresh squads."""
    cached = MAP_CACHE[seed]
    # Reattach fresh squads (terrain/grid/fy are untouched)
    cached.loadSquads(year=1950, seed=seed, levels=n_squads)
    
    starts = cached.find_starts(n=n_squads, ring_radius=3, min_distance=6)
    for squad, start_idx in zip(cached.squads, starts):
        king = next(p for p in squad.pieces if p.type == PieceType.KING)
        loc = cached.location_by_index(start_idx)
        Move.from_squad(squad, placement=Placement(king, loc)).apply(cached)
        king.place.pantry = 100
    
    return cached
```


This was the long trial

```python
MAP_SEEDS = [77, 101, 256]
# Expected matchup matrix: row beats column by this margin
# Positive = row favored, negative = column favored
# Values are target win rates for the ROW player
MATCHUP_TARGETS = {
    ('expansive',  'aggressive'): 0.35,  # aggressive should win ~65%
    ('expansive',  'balanced'):   0.50,
    ('expansive',  'turtle'):     0.70,  # expansive should win ~70%
    ('aggressive', 'balanced'):   0.50,
    ('aggressive', 'turtle'):     0.30,  # turtle should win ~70%
    ('balanced',   'turtle'):     0.50,
}

def score_rps_balance(seed):
    """Round-robin tournament scored against expected matchup matrix."""
    STRATS = {
        'expansive':  PlacementStrategy.EXPANSIVE,
        'aggressive': PlacementStrategy.AGGRESSIVE,
        'balanced':   PlacementStrategy.BALANCED,
        'turtle':     PlacementStrategy.TURTLE,
    }
    
    matchups = list(combinations(STRATS.keys(), 2))
    results = {}  # (s1, s2) -> win rate for s1
    
    for s1_name, s2_name in matchups:
        s1_wins = 0
        games = 0
        
        for first, second in [(s1_name, s2_name), (s2_name, s1_name)]:
            parts = fresh_game(seed, n_squads=2)
            
            strategies = {
                parts.squads[0].name: STRATS[first],
                parts.squads[1].name: STRATS[second],
            }
            
            tp = TurnProcessor(parts, parts.squads)
            for r in range(12):  # 12 rounds is enough to decide most games
                if tp.game_over:
                    break
                for squad in parts.squads:
                    if not squad.king_alive():
                        continue
                    strat = strategies[squad.name]
                    move = strat.next_move(squad, parts)
                    if move:
                        try:
                            tp.resolve_move(squad, move)
                        except MoveError:
                            pass
                if (r + 1) % 3 == 0:
                    for squad in parts.squads:
                        squad.optimize_facings(parts)
            
            # Who won?
            sq0_alive = parts.squads[0].king_alive()
            sq1_alive = parts.squads[1].king_alive()
            
            if sq0_alive and not sq1_alive:
                winner = first
            elif sq1_alive and not sq0_alive:
                winner = second
            else:
                winner = None  # draw
            
            if winner == s1_name:
                s1_wins += 1
            elif winner is None:
                s1_wins += 0.5  # draw
            games += 1
            
            # Clean up for cache reuse
            for sq in parts.squads:
                for p in sq.pieces:
                    p.place = None
        
        results[(s1_name, s2_name)] = s1_wins / games if games > 0 else 0.5
    
    # Score: how close are actual win rates to targets?
    total_error = 0.0
    for matchup, target in MATCHUP_TARGETS.items():
        actual = results.get(matchup, 0.5)
        total_error += abs(actual - target)
    
    n_matchups = len(MATCHUP_TARGETS)
    avg_error = total_error / n_matchups
    score = max(0.0, 1.0 - avg_error * 4.0)
    
    return score, results



def objective_rps(trial):
    p = apply_params(trial)
    
    scores = {k: [] for k in ['sustain', 'flank', 'queen_1v1', 'king_dies', 'rps']}
    all_matchups = {}
    
    for seed in MAP_SEEDS:
        parts = make_parts(seed)
        parts = fresh_game(seed, n_squads=2)
        
        scores['sustain'].append(score_3pawns_support_queen(parts))
        scores['flank'].append(score_2knights_beat_queen(parts))
        scores['queen_1v1'].append(score_queen_beats_knight(parts))
        scores['king_dies'].append(score_lone_king_dies(parts))
        
        rps_score, matchup_results = score_rps_balance(seed)
        scores['rps'].append(rps_score)
        for k, v in matchup_results.items():
            all_matchups.setdefault(k, []).append(v)
    
    s1 = sum(scores['sustain']) / len(MAP_SEEDS)
    s2 = sum(scores['flank']) / len(MAP_SEEDS)
    s3 = sum(scores['queen_1v1']) / len(MAP_SEEDS)
    s4 = sum(scores['king_dies']) / len(MAP_SEEDS)
    s5 = sum(scores['rps']) / len(MAP_SEEDS)
    
    total = (s1 * 2.0 +   # sustain
             s2 * 2.0 +   # flanking
             s3 * 2.0 +   # queen 1v1
             s4 * 1.0 +   # king pressure
             s5 * 4.0)    # RPS balance — highest weight!
    
    trial.set_user_attr('sustain', s1)
    trial.set_user_attr('flank', s2)
    trial.set_user_attr('queen_1v1', s3)
    trial.set_user_attr('king_dies', s4)
    trial.set_user_attr('rps', s5)
    
    for matchup, vals in all_matchups.items():
        label = f"{matchup[0]}_v_{matchup[1]}"
        trial.set_user_attr(label, sum(vals) / len(vals))
    
    return total

# Build initial params from current globals
initial_params = {}
for key, pt in [('pawn', PieceType.PAWN), ('bishop', PieceType.BISHOP),
                ('knight', PieceType.KNIGHT), ('queen', PieceType.QUEEN)]:
    fs = FOOD_TABLE[pt]
    initial_params[f'{key}_diet'] = fs.diet
    initial_params[f'{key}_cost'] = fs.cost
    initial_params[f'{key}_harvest'] = fs.harvest
    initial_params[f'{key}_drain'] = fs.drain
    if key not in ('pawn', 'knight'):  # these have fixed sight
        initial_params[f'{key}_sight'] = fs.bonus_sight
    initial_params[f'{key}_life'] = LIFESPAN[pt]

study_rps = optuna.create_study(direction='maximize', study_name='hex_rps')
study_rps.enqueue_trial(initial_params)
study_rps.optimize(objective_rps, n_trials=40, show_progress_bar=True)

# Results
print(f"\nBest score: {study_rps.best_value:.2f}")
print(f"\nSub-scores:")
for k in ['sustain', 'flank', 'queen_1v1', 'king_dies', 'rps']:
    print(f"  {k}: {study_rps.best_trial.user_attrs[k]:.2f}")
print(f"\nMatchups (row win rate, target in parens):")
for matchup, target in MATCHUP_TARGETS.items():
    label = f"{matchup[0]}_v_{matchup[1]}"
    actual = study_rps.best_trial.user_attrs.get(label, 0.5)
    status = "✓" if abs(actual - target) < 0.15 else "✗"
    print(f"  {matchup[0]:12s} vs {matchup[1]:12s}: {actual:.0%} (target {target:.0%}) {status}")
print(f"\nBest params:")
for k, v in study_rps.best_params.items():
    print(f"  {k}: {v}")
```

## RPS Optuna Study Results (study_rps, 40 trials, 3 seeds, ~8 hrs)

### Best Trial: #35, score 8.20

**Best params:**
| Piece  | Diet | Cost | Harvest | Drain | Sight | Life |
|--------|------|------|---------|-------|-------|------|
| Pawn   | 1    | 3    | 1.25    | 1.0   | 0     | 6    |
| Bishop | 3    | 9    | 1.0     | 0.75  | 1     | 3    |
| Knight | 2    | 10   | 0.0     | 3.0   | 0     | 2    |
| Queen  | 4    | 18   | 0.0     | 4.5   | 2     | 6    |
| King   | 2    | 0    | 1.0     | 0.5   | 1     | 999  |

**Sub-scores:**
- sustain: 0.83 (3 pawns mostly keep queen alive)
- flank: 0.75 (2 knights out-drain queen ✓)
- queen_1v1: 0.75 (queen out-drains knight ✓)
- king_dies: 1.00 (perfect)
- rps: 0.63 (PROBLEM — all matchups are 50/50, no RPS cycle)

**Key issues found:**
1. All 3 map seeds produced IDENTICAL start positions (Hex 265, Hex 791) — 
   `random.seed()` before `GameParts()` creates same terrain. Multi-seed not working.
2. Strategy matchups are completely flat — every matchup is 50%. The four 
   PlacementStrategy personalities aren't producing different enough gameplay.
3. Root cause likely: strategies converge to same move sequences because 
   `best_pawn_move` etc. return similar results regardless of playbook ordering.

**Next steps to investigate:**
- Fix seed diversity (pass seed INTO GameParts or seed differently)
- Add move logging to see if strategies actually diverge in practice
- Strategies may need mechanical differences, not just priority ordering
- Consider: does the game even last long enough (12 rounds) for personalities to matter?


Ha — I know the feeling. But actually there *is* a result here, just not an exciting one:

**The good news:** the original params are **robust**. Zero variance across all 5 maps on both `strat` and `sustain`. That means we weren't overfitting — the balance holds regardless of terrain layout.

**The real problem** is the scores that are stuck:
- `queen_1v1: 0.62` — queen drain (4.25) only beats knight drain (3.0) by 1.25, needs 2.0 for a perfect score
- `sustain: 0.88` — queen doesn't *quite* survive her full 8-turn lifespan on 3 pawns
- `flank: 0.88` — 2×3.0 - 4.25 = 1.75, needs 2.0

These three objectives are **in tension** — pushing queen drain higher fixes queen_1v1 but makes sustain harder (queen costs more to feed). The optimizer found the Pareto front and there's nowhere better to go within the current search bounds.

Options to break the plateau:
1. **Relax the objectives** — is 0.88 sustain good enough? Maybe queen surviving 7/8 turns is fine
2. **Add a new lever** — e.g. queen gets partial harvest (0.25×), which helps sustain without touching drain
3. **Rethink the scoring** — the flank/1v1 tests are pure math, not simulated. Maybe the *actual* combat dynamics play out differently than `2×knight_drain - queen_drain`

Which direction appeals to you?

In [ ]:
if showDemo:
    ctx = myStuff.overlayContext( padding=2, radius=20)
    myStuff.builder.layers = []
    TerrainDisplay(
        CreamOverlay(),
        RiverOverlay(max_width=4),
        FogOverlay(),
        SquadNamesOverlay(),
        LanternOverlay(),
        PieceOverlay(),
        pieces=[unit],

        ctx=ctx,
        debug = not showDemo
    )

## Web

In [ ]:
def appRoutes():
    global app, rt, hexserver
    #if 'hexserver' not in dir() or hexserver is None:
    if 'hexserver' not in globals() or hexserver is None:

        app = FastHTML(hdrs=Theme.violet.headers())
        rt = app.route
        hexserver = JupyUvi(app)
    return app, rt, hexserver

In [ ]:
app, rt,  hexserver = appRoutes()

In [ ]:
from collections import deque
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')
log = logging.getLogger("hexgame")

# In-memory log ring buffer
game_log = deque(maxlen=50)

class GameLogHandler(logging.Handler):
    def emit(self, record):
        game_log.append(self.format(record))

log.addHandler(GameLogHandler())
log.setLevel(logging.INFO)

# Option 1: Attach to uvicorn loggers directly
for name in ('uvicorn', 'uvicorn.access', 'uvicorn.error'):
    logging.getLogger(name).addHandler(GameLogHandler())


In [ ]:

def debug_log():
    return Card(
        Pre('\n'.join(game_log), cls="text-xs whitespace-pre-wrap max-h-[60vh] overflow-y-auto"),
        header=H4(f"Game Log ({len(game_log)} entries)"),
        footer=Button("Refresh", hx_get="/debug/log", hx_target="#debug-panel"),
        id="debug-panel",
    )


In [ ]:
def webMe(*c): return HTMX(*c, host='', app=app)

In [ ]:
@rt
def hello(name: str): return P(f"Hello {name}!")

webMe(Div(
    H3("Say Hello"),
    Form(
        Input(placeholder="Your name...", id='name'),
        Button("Send"),
        hx_get=hello, hx_target="#result"
    ),
    Div(id='result'),
))

In [ ]:
from HexMagic.primitives import HexButtonGroup, HexLegend
from starlette.responses import RedirectResponse

So I want to have an interative game as a fasthtml app. For the main game part there should be 3 panels
1. left side panel - this is a list of your pieces by rank. if a piece is on the board this rotates it clockwise. if a piece is not on the board it selects it for placement
2. the main map. if you have a selected piece and you click it places it on the map. if you click a selected piece that you are thinking about moving that is on the board it removes it from the board. below the map will be radio buttons (either a hexlegend or button group) that we can toggle overlays. there should be a 'commit move' button which says that this is your final move.
3. the history right side panel - both the moves on top and then a detail one on bottom that shows/explains the log from a particular round.

We do need a landing page that lets you start a new map and you can pick the size, number of countries and perhaps seed

## GlobalGameState

In [ ]:
@dataclass
class GlobalGameState:
    parts: GameParts | None = None
    tp: TurnProcessor | None = None
    player_squad: Squad | None = None
    selected_piece: Piece | None = None
    pending_rotations: list[Rotation] = field(default_factory=list)
    pending_placement: Placement | None = None
    active_overlays: set = field(default_factory=lambda: {'terrain', 'rivers'})
    turn: int = 0
    radius:float = 25
    debug_overlays: bool = False
    ai_strategies: dict[str, PlacementStrategy] = field(default_factory=dict)
    
    def start_game(self, size: int = 300, player_count: int = 3, 
                   seed: int = 23, radius: int = 20):
        """Create map, squads, place kings, init turn processor."""
        random.seed(seed)
        np.random.seed(seed)
        self.parts = GameParts(radius=radius, size=size)
        self.parts.grid.adjustRadius(radius)
        self.parts.loadSquads(year=1950, seed=seed, levels=player_count)
        
        starts = self.parts.find_starts(
            n=player_count, ring_radius=3, min_distance=6)
        
        for squad, start_idx in zip(self.parts.squads, starts):
            king = next(p for p in squad.pieces if p.type == PieceType.KING)
            loc = self.parts.location_by_index(start_idx)
            Move.from_squad(squad, placement=Placement(king, loc)).apply(self.parts)
            king.place.pantry = 100
        
        # Player is squad 0, rest get rotating AI personalities
        self.player_squad = self.parts.squads[0]
        
        ai_pool = [
            PlacementStrategy.AGGRESSIVE,
            PlacementStrategy.EXPANSIVE,
            PlacementStrategy.TURTLE,
            PlacementStrategy.BALANCED,
        ]
        self.ai_strategies = {}
        for i, squad in enumerate(self.parts.squads[1:]):
            self.ai_strategies[squad.name] = ai_pool[i % len(ai_pool)]
        
        self.tp = TurnProcessor(self.parts, self.parts.squads)
        self.selected_piece = None
        self.pending_rotations = []
        self.pending_placement = None
        self.turn = 0
    

    
    def select_piece(self, piece: Piece):
        """Select an unplaced piece for placement."""
        if piece.place is not None:
            return  # already on board — rotation handled separately
        self.selected_piece = piece
    
    def place_piece(self, hex_idx: int):
        """Place the selected piece at a hex."""
        if self.selected_piece is None:
            return False
        loc = self.parts.location_by_index(hex_idx)
        self.pending_placement = Placement(self.selected_piece, loc)
        self.selected_piece.place = loc  # preview immediately
        self.selected_piece = None
        return True
    
    def unplace_piece(self, piece: Piece):
        """Remove a pending placement (before committing)."""
        if self.pending_placement and self.pending_placement.piece is piece:
            piece.place = None
            self.pending_placement = None
    
    
    @property
    def is_active(self) -> bool:
        return self.parts is not None and self.tp is not None
    
    @property
    def game_over(self) -> bool:
        return self.tp.game_over if self.tp else False

# Global singleton
GS = GlobalGameState()


In [ ]:
@patch
def compute_optimal(self: GlobalGameState) -> dict[PieceType, list[tuple[int, float]]]:
    """Compute top-3 placement candidates per piece type. Cached per turn."""
    if not self.is_active:
        return {}
    
    squad = self.player_squad
    placed = [p for p in squad.pieces if p.place is not None]
    if not placed:
        return {}
    
    parts = self.parts
    terr = parts.terr
    
    current_visible = set()
    for ally in placed:
        current_visible |= ally.sight().hexes
    land_set = {i for i in range(len(terr.elevations)) if terr.elevations[i] > 0}
    occupied = {parts.hp2i(p.place.location)
                for sq in parts.squads for p in sq.pieces if p.place is not None}
    
    # Reuse the same scorers from OptimalPlacementOverlay
    from types import SimpleNamespace
    SCORERS = {
        PieceType.PAWN:   _score_pawn,
        PieceType.BISHOP: _score_bishop,
        PieceType.KNIGHT: _score_knight,
        PieceType.QUEEN:  _score_queen,
    }
    
    results = {}
    for pt, scorer in SCORERS.items():
        unplaced = [p for p in squad.pieces if p.type == pt and p.place is None]
        if not unplaced:
            continue
        scored = []
        for idx in current_visible:
            s = scorer(idx, parts, squad, current_visible, occupied, land_set)
            if s > -999:
                scored.append((idx, s))
        scored.sort(key=lambda x: -x[1])
        results[pt] = scored[:3]
    
    return results


In [ ]:
@patch
def rotate_piece(self: GlobalGameState, piece: Piece):
    """Rotate a placed piece clockwise by one step."""
    if piece.place is None:
        return
    dirs = HexPosition.directions()
    cur_idx = dirs.index(piece.place.facing) if piece.place.facing in dirs else 0
    new_facing = dirs[(cur_idx + 1) % len(dirs)]
    
    # If this is the pending placement piece, just update its facing
    # (not a rotation — it hasn't been committed yet)
    if self.pending_placement and self.pending_placement.piece is piece:
        piece.place.facing = new_facing
        # Update the placement's location facing too
        self.pending_placement.location.facing = new_facing
        return
    
    # Otherwise it's a real rotation of an already-placed piece
    self.pending_rotations = [r for r in self.pending_rotations if r.piece is not piece]
    self.pending_rotations.append(Rotation(piece, new_facing))
    piece.place.facing = new_facing  # preview immediately


In [ ]:
@patch
def commit_move(self:GlobalGameState) -> dict | None:
    if self.parts is None:
        return None
    
    # Undo placement preview before validation
    if self.pending_placement is not None:
        self.pending_placement.piece.place = None
    
    try:
        move = Move.from_squad(
            self.player_squad,
            rotations=self.pending_rotations,
            placement=self.pending_placement)
    except MoveError as e:
        # Re-apply preview so map still shows it
        if self.pending_placement is not None:
            self.pending_placement.piece.place = self.pending_placement.location
        return {'error': str(e)}
    
    summary = self.tp.resolve_move(self.player_squad, move)
    
    # Run AI turns
    ai_summaries = []
    for squad in self.parts.squads:
        if squad is self.player_squad or not squad.king_alive():
            continue
        strategy = self.ai_strategies.get(squad.name, PlacementStrategy.BALANCED)
        ai_move = strategy.next_move(squad, self.parts)
        if ai_move:
            try:
                ai_summaries.append(self.tp.resolve_move(squad, ai_move))
            except MoveError:
                pass
    
    self.turn += 1
    if self.turn % 3 == 0:
        for squad in self.parts.squads:
            squad.optimize_facings(self.parts)
    
    self.pending_rotations = []
    self.pending_placement = None
    self.selected_piece = None
    
    return {'player': summary, 'ai': ai_summaries, 'turn': self.turn}


In [ ]:
# ── Landing page ──
@rt
def index():
    return Title("HexMagic"), Container(
        DivCentered(
            H1("⬡ HexMagic", cls="uk-h1"),
            P("Hex-based strategy with food economics", cls=TextPresets.muted_lg),
            Card(
                Form(
                    Grid(
                        LabelInput("Map Size", id="size", type="number", 
                                   value="300", placeholder="200-600"),
                        LabelInput("Players", id="player_count", type="number",
                                   value="3", placeholder="2-5"),
                        LabelInput("Seed", id="seed", type="number",
                                   value="23", placeholder="Any integer"),
                        LabelInput("Hex Radius", id="radius", type="number",
                                   value="20", placeholder="15-30"),
                        cols_sm=2,
                    ),
                    Button("⬡ Start Game", cls=(ButtonT.primary, "w-full mt-4"),
                           hx_post="/start_game", hx_target="body"),
                ),
                header=(H3("New Game"), Subtitle("Configure your world")),
            ),
            cls="min-h-screen py-12",
        )
    )

# ── Start game ──
@rt
def start_game(size: int = 300, player_count: int = 3, 
               seed: int = 23, radius: int = 20):
    player_count = max(2, min(5, player_count))
    size = max(200, min(600, size))
    radius = max(10, min(40, radius))
    
    GS.start_game(size=size, player_count=player_count, 
                   seed=seed, radius=radius)
    
    return RedirectResponse("/game", status_code=303)


@rt("/toggle_overlay", methods=["POST"])
def toggle_overlay(overlay: str):
    if overlay in GS.active_overlays:
        GS.active_overlays.discard(overlay)
    else:
        GS.active_overlays.add(overlay)
    return panel_map()




read_url("https://fastht.ml/docs/llms-ctx.txt")

In [ ]:
@rt("/panel/pieces")
def panel_pieces():
    squad = GS.player_squad
    by_type = {}
    for p in squad.pieces:
        by_type.setdefault(p.type, []).append(p)
    
    king = next((p for p in squad.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    pantry = king.place.pantry if king else 0
    
    sections = []
    for pt in [PieceType.KING, PieceType.QUEEN, PieceType.BISHOP, 
               PieceType.KNIGHT, PieceType.PAWN]:
        pieces = by_type.get(pt, [])
        if not pieces:
            continue
        
        rows = []
        for p in pieces:
            is_selected = GS.selected_piece is p
            on_board = p.place is not None
            
            # Build the avatar SVG (reuse piece __ft__ drawing)
            b = SVGBuilder()
            b.width, b.height = 40, 40
            p.draw_avatar(MapCord(20, 20), b, size=36, layer="avatar")
            avatar = Div(NotStr(b.xml()), style="flex-shrink:0; line-height:0;")
            
            # Action + badge per state
            if on_board:
                attrs = dict(hx_post="/piece/rotate",
                             hx_vals=json.dumps({"piece_id": p.id}))
                badge = Label(f"⟳ {p.place.facing.label}", cls=LabelT.secondary)
            elif is_selected:
                attrs = dict(hx_post="/piece/deselect",
                             hx_vals=json.dumps({"piece_id": p.id}))
                badge = Label("📍 placing", cls=LabelT.primary)
            else:
                attrs = dict(hx_post="/piece/select",
                             hx_vals=json.dumps({"piece_id": p.id}))
                badge = Label("reserve", cls=LabelT.destructive)
            
            # Pantry indicator for placed pieces
            pantry_info = ""
            if on_board and p.place.pantry is not None:
                pantry_info = Small(f"🍞 {p.place.pantry}", cls=TextPresets.muted_sm)
            
            row = Div(
                DivFullySpaced(
                    DivLAligned(
                        avatar,
                        Div(
                            DivLAligned(
                                Span(pt.icon),
                                Strong(p.name),
                                pantry_info,
                            ),
                            P(f"{p.year} · {p.food.diet}🍽️ {p.food.drain}⚔️", 
                              cls=TextPresets.muted_sm),
                        ),
                    ),
                    badge,
                ),
                cls=f"p-2 rounded-md cursor-pointer transition-all "
                    f"{'ring-2 ring-primary bg-primary/10' if is_selected else 'hover:bg-muted/50'}",
                **attrs,
            )
            rows.append(row)
        
        sections.append(Div(
            DivLAligned(
                Span(pt.icon, cls=TextT.lg), 
                H5(pt.value.title()),
                Small(f"({len(pieces)})", cls=TextPresets.muted_sm),
            ),
            *rows,
            cls="space-y-1",
        ))
    
    # Selection status message
    status = ""
    if GS.selected_piece:
        p = GS.selected_piece
        status = P(f"Click a hex to place {p.type.icon} {p.name}", 
                   cls="text-primary text-sm text-center py-1")
    elif GS.pending_placement:
        p = GS.pending_placement.piece
        status = P(f"Pending: {p.type.icon} {p.name}. Click piece to undo.",
                   cls="text-warning text-sm text-center py-1")
    
    return Div(
        Card(
            *sections,
            header=DivFullySpaced(
                DivLAligned(Span("👑", cls=TextT.lg), H4(squad.name)),
                Label(f"🍞 {pantry}", cls=LabelT.primary),
            ),
            cls="space-y-3",
        ),
        status,
        Button("✅ Commit Move", cls=(ButtonT.primary, "w-full mt-3"),
               hx_post="/commit", hx_target="body"),
        id="piece-panel",
        hx_target="#piece-panel",  # inherited by all children
    )


In [ ]:
@rt("/panel/history")
def panel_history():
    if GS.tp is None:
        return Div(P("No game active", cls=TextPresets.muted_sm))
    
    status_df = GS.tp.status()
    
    status_rows = [Tr(
        Td(row['king']),
        Td(row['squad'], cls=TextT.medium),
        Td(str(row['king_pantry'])),
        Td(str(row['placed'])),
        Td(GS.ai_strategies.get(row['squad'], 
           PlacementStrategy("you")).name if row['squad'] != GS.player_squad.name 
           else "🎮 you",
           cls=TextPresets.muted_sm),
    ) for _, row in status_df.iterrows()]
    
    status_table = Card(
        Table(
            Thead(Tr(Th(""), Th("Squad"), Th("🍞"), Th("#"), Th("AI"))),
            Tbody(*status_rows),
        ),
        header=H4(f"Turn {GS.turn}"),
    )
    
    # ... log table same as before
    # Move log
    log_rows = []
    if len(GS.tp.log.df) > 0:
        recent = GS.tp.log.df.tail(20)
        for _, row in recent.iterrows():
            log_rows.append(Tr(
                Td(str(int(row['turn'])), cls=TextPresets.muted_sm),
                Td(row['squad'], cls=TextT.medium),
                Td(row['piece']),
                Td(row['action']),
                Td(str(row['detail']), cls=TextPresets.muted_sm),
            ))
    
    log_table = Card(
        Table(
            Thead(Tr(Th("T"), Th("Squad"), Th("Piece"), Th("Act"), Th("Detail"))),
            Tbody(*log_rows),
        ) if log_rows else P("No moves yet", cls=TextPresets.muted_sm),
        header=H4("Move Log"),
    )
    
    return Div(status_table, log_table, cls="space-y-3")


In [ ]:






@rt("/hex_click", methods=["POST"])
def hex_click(hex_id: int):
    log.info(f"HEX_CLICK hex_id={hex_id} selected={GS.selected_piece.name if GS.selected_piece else None} "
             f"pending={GS.pending_placement.piece.name if GS.pending_placement else None}")
    if GS.selected_piece is not None:
        GS.place_piece(hex_id)
        log.info(f"  → placed {GS.pending_placement.piece.name if GS.pending_placement else '?'} at {hex_id}")
    elif GS.pending_placement:
        idx = GS.parts.hp2i(GS.pending_placement.piece.place.location)
        if idx == hex_id:
            log.info(f"  → unplaced {GS.pending_placement.piece.name}")
            GS.unplace_piece(GS.pending_placement.piece)
    return panel_center()



@rt("/commit", methods=["POST"])
def commit():
    log.info(f"COMMIT turn={GS.turn} "
             f"rotations={len(GS.pending_rotations)} "
             f"placement={GS.pending_placement.piece.name if GS.pending_placement else None}")
    result = GS.commit_move()
    if result and 'error' in result:
        log.warning(f"  → ERROR: {result['error']}")
        return Div(
            P(f"⚠️ {result['error']}", cls="text-destructive font-medium text-center py-4"),
            Button("Back", hx_get="/game", hx_target="body"),
        )
    log.info(f"  → turn={GS.turn} king_pantry={result['player']['king_pantry_after']} "
             f"starved={result['player']['starved']}")
    return RedirectResponse("/game", status_code=303)


In [ ]:
@rt("/piece/rotate", methods=["POST"])
def piece_rotate(piece_id: str):
    piece = next((p for p in GS.player_squad.pieces if p.id == piece_id), None)
    old_facing = piece.place.facing.label if piece and piece.place else "?"
    if piece:
        GS.rotate_piece(piece)
    new_facing = piece.place.facing.label if piece and piece.place else "?"
    log.info(f"ROTATE piece_id={piece_id} name={piece.name if piece else '?'} "
             f"{old_facing} → {new_facing}")
    
    # Return pieces panel as main swap + map as OOB swap
    map_update = Div(panel_center(), hx_swap_oob="innerHTML:#center-panel")
    return panel_pieces(), map_update


In [ ]:
@rt("/piece/select", methods=["POST"])
def piece_select(piece_id: str):
    piece = next((p for p in GS.player_squad.pieces if p.id == piece_id), None)
    log.info(f"SELECT piece_id={piece_id} found={piece is not None} "
             f"name={piece.name if piece else '?'} "
             f"on_board={piece.place is not None if piece else '?'}")
    if piece:
        GS.select_piece(piece)
    map_update = Div(panel_center(), hx_swap_oob="innerHTML:#center-panel")
    return panel_pieces(), map_update

@rt("/piece/deselect", methods=["POST"])
def piece_deselect(piece_id: str):
    log.info(f"DESELECT piece_id={piece_id}")
    GS.selected_piece = None
    map_update = Div(panel_center(), hx_swap_oob="innerHTML:#center-panel")
    return panel_pieces(), map_update


In [ ]:
@rt("/panel/center")
def panel_center(view: str = "map"):
    tabs = DivLAligned(
        Button("🗺️ Map",
               cls=(ButtonT.primary if view == "map" else ButtonT.ghost, "text-sm"),
               hx_get="/panel/center?view=map", hx_target="#center-panel",
               hx_swap="outerHTML"),
        Button("👥 Squad",
               cls=(ButtonT.primary if view == "squad" else ButtonT.ghost, "text-sm"),
               hx_get="/panel/center?view=squad", hx_target="#center-panel",
               hx_swap="outerHTML"),
        Button("📊 Charts",
               cls=(ButtonT.primary if view == "charts" else ButtonT.ghost, "text-sm"),
               hx_get="/panel/center?view=charts", hx_target="#center-panel",
               hx_swap="outerHTML"),
        cls="gap-1 pb-2",
    )
    
    if view == "squad":
        content = panel_squad_detail()
    elif view == "charts":
        content = panel_charts()
    else:
        content = panel_map_content()
    
    return Div(tabs, content, id="center-panel")


In [ ]:
def panel_squad_detail():
    """Squad detail view — reuses squad table + network info."""
    squad = GS.player_squad
    
    # Squad table (reuse existing .table() method)
    table_card = squad.table()
    
    # Network health summary
    G = squad.to_nx()
    U = G.to_undirected(as_view=True) if len(G) > 1 else None
    bottlenecks = list(nx.articulation_points(U)) if U else []
    isolated = squad.isolated_pieces()
    sources = squad.food_sources()
    sinks = squad.food_sinks()
    
    king = next((p for p in squad.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    
    network_card = Card(
        Grid(
            Div(
                DivLAligned(Span("🔗", cls=TextT.lg), Strong("Connected")),
                P(f"{len(G.nodes)} pieces", cls=TextPresets.muted_sm),
            ),
            Div(
                DivLAligned(Span("⚠️", cls=TextT.lg), Strong("Bottlenecks")),
                P(f"{len(bottlenecks)}", cls=TextPresets.muted_sm),
            ),
            Div(
                DivLAligned(Span("🌾", cls=TextT.lg), Strong("Sources")),
                P(f"{len(sources)} pieces", cls=TextPresets.muted_sm),
            ),
            Div(
                DivLAligned(Span("🍽️", cls=TextT.lg), Strong("Sinks")),
                P(f"{len(sinks)} pieces", cls=TextPresets.muted_sm),
            ),
            cols_sm=2, cols_md=4,
        ),
        header=DivFullySpaced(
            H4("Network Health"),
            Label(f"🍞 {king.place.pantry if king else 0}", cls=LabelT.primary),
        ),
    )
    
    # Isolated pieces warning
    warnings = []
    if isolated:
        names = ", ".join(f"{p.type.icon} {p.name}" for p in isolated)
        warnings.append(P(f"⚠️ Isolated: {names}", cls="text-destructive text-sm"))
    if bottlenecks:
        names = ", ".join(f"{G.nodes[n]['piece'].type.icon} {G.nodes[n]['piece'].name}" 
                          for n in bottlenecks)
        warnings.append(P(f"🔗 Bottlenecks: {names}", cls="text-warning text-sm"))
    
    return Div(network_card, *warnings, table_card, cls="space-y-3")


#### Squads and left panel

I think I have some duplicated functions


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Looking through your code, here are the duplications I see:

1. **`resolve_move`** — defined in the `TurnProcessor` class body *and* again as a `@patch`. The patch overwrites it, so the class version is dead code.

2. **`score_lone_king_dies`** — defined in two separate code cells (`_5d953ca1` and `_5df23904`).

3. **`PlacementStrategy`** — defined first as a simple class with `BOTTOM_UP`/`TOP_DOWN`, then completely redefined with playbooks. The first definition is dead code (and `BOTTOM_UP`/`TOP_DOWN` no longer exist on the new class, which could break `score_bottomup_beats_topdown` if you ever run it).

4. **`FOOD_TABLE` and `LIFESPAN`** — each set twice in the same cell (initial values then immediately overwritten). Minor but noisy.

Which of these do you want to clean up first?

In [ ]:
def optimal_hex_info(pt, picks):
    """Render a compact list of top hex picks for a piece type."""
    if not picks:
        return ""
    
    parts = GS.parts
    rows = []
    for rank, (idx, score) in enumerate(picks):
        loc = parts.location_by_index(idx)
        tier = int(parts.fy.tiers[idx])
        level = loc.level
        
        # Quick terrain label
        terrain = parts.terr.elevationLevel(idx) if hasattr(parts.terr, 'elevationLevel') else f"L{level}"
        
        # ROI estimate
        fs = FOOD_TABLE[pt]
        if fs.harvest > 0:
            net = int(fs.harvest * tier) - fs.diet
            roi_str = f"{net:+d}/t"
        else:
            roi_str = f"⚔️{fs.drain}"
        
        rows.append(
            Div(
                DivFullySpaced(
                    DivLAligned(
                        Small(f"#{rank+1}", cls="font-bold w-5"),
                        Small(f"Hex {idx}", cls=TextT.medium),
                    ),
                    DivLAligned(
                        Small(f"🌾{tier}", cls="text-green-600"),
                        Small(f"⛰️{level}"),
                        Small(roi_str, cls="text-amber-600"),
                        cls="gap-1",
                    ),
                ),
                cls="px-2 py-0.5 text-xs bg-muted/30 rounded cursor-pointer hover:bg-muted/60",
                hx_post="/hex_focus",
                hx_vals=json.dumps({"hex_id": idx}),
                hx_target="#center-panel",
            )
        )
    
    return Div(*rows, cls="space-y-0.5 pl-6 pb-1")


In [ ]:
def piece_icon(piece, size=24):
    """Render a tiny inline SVG of a chess piece using flag colors."""
    b = SVGBuilder()
    b.width, b.height = size, size
    scale = size / 45.0
    piece.flag.draw_piece(
        piece.type, MapCord(size/2, size/2), b,
        scale=scale, size='board', layer="icon",
        piece_id=f"ico_{piece.id[:8]}")
    return Div(NotStr(b.xml()), 
               style=f"width:{size}px; height:{size}px; flex-shrink:0; line-height:0;")

@rt("/panel/pieces")
def panel_pieces():
    squad = GS.player_squad
    king = next((p for p in squad.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    pantry = king.place.pantry if king else 0
    
    show_optimal = 'optimal' in GS.active_overlays
    optimal = GS.compute_optimal() if show_optimal else {}
    
    by_type = {}
    for p in squad.pieces:
        by_type.setdefault(p.type, []).append(p)
    
    sections = []
    hexPlaces = []

    for pt in [PieceType.KING, PieceType.QUEEN, PieceType.BISHOP, 
               PieceType.KNIGHT, PieceType.PAWN]:
        pieces = by_type.get(pt, [])
        if not pieces:
            continue
        
        rows = []
        for p in pieces:
            on_board = p.place is not None
            is_selected = GS.selected_piece is p
            is_pending = GS.pending_placement and GS.pending_placement.piece is p
            
            # Turns remaining
            if on_board and p.type != PieceType.KING:
                remaining = LIFESPAN[p.type] - p.place.tenure
                life_str = f"{remaining}t"
                life_cls = "text-destructive" if remaining <= 1 else "text-muted-foreground"
            elif on_board:
                life_str = "∞"
                life_cls = "text-muted-foreground"
            else:
                life_str = ""
                life_cls = ""
            
            # Facing indicator
            facing_str = p.place.facing.label if on_board else ""
            
            # Food status
            food_str = ""
            if on_board:
                net = int(p.food.harvest * p.place.harvest) - p.food.diet
                food_str = f"{net:+d}"
            
            # Row styling
            if is_selected:
                row_cls = "bg-primary/15 ring-1 ring-primary"
            elif is_pending:
                row_cls = "bg-warning/15 ring-1 ring-warning"
            elif not on_board:
                row_cls = "opacity-60"
            else:
                row_cls = "hover:bg-muted/50"
            
            # Action
            if on_board:
                attrs = dict(hx_post="/piece/rotate",
                             hx_vals=json.dumps({"piece_id": p.id}))
            elif is_selected:
                attrs = dict(hx_post="/piece/deselect",
                             hx_vals=json.dumps({"piece_id": p.id}))
            else:
                attrs = dict(hx_post="/piece/select",
                             hx_vals=json.dumps({"piece_id": p.id}))
            
            row = Div(
                DivFullySpaced(
                    DivLAligned(
                        piece_icon(p, size=22),
                        Span(p.name, cls="text-xs font-medium"),
                    ),
                    DivLAligned(
                        Small(food_str, 
                              cls=f"w-6 text-right text-xs "
                                  f"{'text-green-600' if food_str.startswith('+') else 'text-destructive' if food_str.startswith('-') else ''}"),
                        Small(facing_str, cls="w-6 text-center text-xs text-muted-foreground"),
                        Small(life_str, cls=f"w-5 text-right text-xs {life_cls}"),
                        Small(f"{p.place.pantry}" if on_board else "", 
                              cls="w-7 text-right text-xs text-amber-600"),
                        cls="gap-1",
                    ),
                ),
                cls=f"px-2 py-1 rounded cursor-pointer transition-all {row_cls}",
                **attrs,
            )
            rows.append(row)
       
        # Optimal picks for this piece type (only if overlay active + unplaced pieces exist)
        has_unplaced = any(p.place is None for p in pieces)
        picks_section = optimal_hex_info(pt, optimal.get(pt, [])) \
            if show_optimal and has_unplaced else ""


        hexPlaces.append(Div(
            DivLAligned(
                piece_icon(pieces[0], size=18),
                Small(pt.value.title(), cls="font-medium text-xs"),
                Small(f"({sum(1 for p in pieces if p.place is None)} avail)", 
                      cls=TextPresets.muted_sm),
            ),
            picks_section,
            cls="space-y-0.5",
        ))
        
        sections.append(Div(
            DivLAligned(
                piece_icon(pieces[0], size=18),
                Small(pt.value.title(), cls="font-medium text-xs"),
                Small(f"({sum(1 for p in pieces if p.place is None)} avail)", 
                      cls=TextPresets.muted_sm),
            ),
            *rows,
            cls="space-y-0.5",
        ))

    
    sections.extend(hexPlaces)
    # Status line
    status = ""
    if GS.selected_piece:
        p = GS.selected_piece
        status = P(f"📍 Place {p.type.icon} {p.name}", 
                   cls="text-primary text-xs text-center py-1")
    elif GS.pending_placement:
        p = GS.pending_placement.piece
        status = P(f"Pending: {p.type.icon} {p.name}",
                   cls="text-warning text-xs text-center py-1")
    
    # Column headers
    header_row = DivFullySpaced(
        DivLAligned(Small("", cls="w-6"), Small("Name", cls=TextPresets.muted_sm)),
        DivLAligned(
            Small("±", cls="w-6 text-right text-muted-foreground text-xs"),
            Small("Dir", cls="w-6 text-center text-muted-foreground text-xs"),
            Small("T", cls="w-5 text-right text-muted-foreground text-xs"),
            Small("🍞", cls="w-7 text-right text-xs"),
            cls="gap-1",
        ),
        cls="px-2 pb-1 border-b",
    )
    
    return Div(
        DivFullySpaced(
            DivLAligned(
                Strong(squad.name, cls="text-sm"),
                Label(f"🍞{pantry}", cls=LabelT.primary),
            ),
            Button("◀", cls=(ButtonT.ghost, "text-xs px-1"),
                   hx_post="/toggle_panel", hx_target="#left-col"),
        ),
        header_row,
        Div(*sections, cls="space-y-2"),
        status,
        Button("✅ Commit", cls=(ButtonT.primary, "w-full mt-2 text-sm"),
               hx_post="/commit", hx_target="body"),
        id="piece-panel",
        hx_target="#piece-panel",
        cls="space-y-1",
    )


In [ ]:
@rt("/toggle_panel", methods=["POST"])
def toggle_panel():
    GS.panel_collapsed = not getattr(GS, 'panel_collapsed', False)
    if GS.panel_collapsed:
        return Div(
            Button("▶", cls=(ButtonT.ghost, "text-xs"),
                   hx_post="/toggle_panel", hx_target="#left-col"),
            id="left-col", cls="w-8 flex-shrink-0",
        )
    return Div(
        Div(hx_get="/panel/pieces", hx_trigger="load",
            hx_target="#piece-panel", id="piece-panel"),
        id="left-col", cls="w-[250px] flex-shrink-0 overflow-y-auto max-h-[80vh]",
    )


In [ ]:
def panel_squad_detail():
    """Rich squad view for the center panel — card grid + network health."""
    squad = GS.player_squad
    king = next((p for p in squad.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    
    # ── Network health banner ──
    G = squad.to_nx()
    U = G.to_undirected(as_view=True) if len(G) > 1 else None
    bottlenecks = list(nx.articulation_points(U)) if U else []
    isolated = squad.isolated_pieces()
    sources = squad.food_sources()
    sinks = squad.food_sinks()
    placed = [p for p in squad.pieces if p.place is not None]
    
    health_items = [
        ("🔗", "Connected", f"{len(placed)}/{len(squad.pieces)}"),
        ("⚠️", "Bottlenecks", str(len(bottlenecks))),
        ("🌾", "Sources", str(len(sources))),
        ("🍽️", "Sinks", str(len(sinks))),
        ("👑", "Treasury", f"🍞{king.place.pantry}" if king else "—"),
    ]
    health_bar = DivFullySpaced(
        *[DivCentered(
            Span(icon, cls=TextT.lg),
            Small(label, cls=TextPresets.muted_sm),
            Strong(val, cls="text-sm"),
            cls="space-y-0.5",
        ) for icon, label, val in health_items],
        cls="py-3 px-4 bg-muted/30 rounded-lg",
    )
    
    # ── Warnings ──
    warnings = []
    if isolated:
        names = ", ".join(f"{p.type.icon} {p.name}" for p in isolated)
        warnings.append(
            Div(DivLAligned(Span("🚨"), P(f"Isolated: {names}")),
                cls="text-destructive text-sm bg-destructive/10 rounded px-3 py-2"))
    if bottlenecks:
        names = ", ".join(f"{G.nodes[n]['piece'].type.icon} {G.nodes[n]['piece'].name}" 
                          for n in bottlenecks)
        warnings.append(
            Div(DivLAligned(Span("⚠️"), P(f"Bottlenecks: {names}")),
                cls="text-warning text-sm bg-warning/10 rounded px-3 py-2"))
    
    # ── Piece cards grid ──
    cards = []
    for p in squad.pieces:
        on_board = p.place is not None
        fs = p.food
        
        # Avatar
        b = SVGBuilder()
        b.width, b.height = 60, 60
        p.draw_avatar(MapCord(30, 30), b, size=56, layer="avatar")
        avatar = Div(NotStr(b.xml()), style="flex-shrink:0; line-height:0;")
        
        # Stats
        if on_board:
            tier = p.place.harvest
            income = int(fs.harvest * tier)
            net = income - fs.diet
            remaining = LIFESPAN[p.type] - p.place.tenure if p.type != PieceType.KING else "∞"
            facing = p.place.facing.label
            
            stats = Div(
                DivFullySpaced(
                    Small("Income"), Small(f"+{income}", cls="text-green-600"),
                ),
                DivFullySpaced(
                    Small("Diet"), Small(f"-{fs.diet}", cls="text-destructive"),
                ),
                DivFullySpaced(
                    Small("Net"), 
                    Small(f"{net:+d}", cls="text-green-600 font-bold" if net >= 0 else "text-destructive font-bold"),
                ),
                DivFullySpaced(
                    Small("Drain"), Small(f"⚔️{fs.drain}"),
                ),
                DivFullySpaced(
                    Small("Pantry"), Small(f"🍞{p.place.pantry}", cls="text-amber-600"),
                ),
                DivFullySpaced(
                    Small("Facing"), Small(facing),
                ),
                DivFullySpaced(
                    Small("Turns left"), Small(str(remaining)),
                ),
                cls="text-xs space-y-0.5",
            )
            
            # Sight coverage count
            sight_count = len(p.sight().hexes)
            location_badge = Label(f"Hex {GS.parts.hp2i(p.place.location)}", cls=LabelT.secondary)
        else:
            stats = Div(
                DivFullySpaced(Small("Cost"), Small(f"🍞{fs.cost}")),
                DivFullySpaced(Small("Diet"), Small(f"{fs.diet}/turn")),
                DivFullySpaced(Small("Harvest"), Small(f"×{fs.harvest}")),
                DivFullySpaced(Small("Drain"), Small(f"⚔️{fs.drain}")),
                DivFullySpaced(Small("Lifespan"), Small(f"{LIFESPAN[p.type]} turns")),
                DivFullySpaced(Small("Sight"), Small(f"+{fs.bonus_sight}" if fs.bonus_sight else "base")),
                cls="text-xs space-y-0.5",
            )
            sight_count = 0
            location_badge = Label("Reserve", cls=LabelT.destructive)
        
        card = Card(
            DivLAligned(avatar, Div(
                DivLAligned(Span(p.type.icon), Strong(p.name)),
                P(f"b. {p.year}", cls=TextPresets.muted_sm),
            )),
            Divider(cls="my-2"),
            stats,
            header=DivFullySpaced(
                Small(p.type.value.title(), cls=TextPresets.muted_sm),
                location_badge,
            ),
            footer=DivFullySpaced(
                Small(f"👁️ {sight_count} hexes" if sight_count else "", cls=TextPresets.muted_sm),
                Button("🔍", cls=(ButtonT.ghost, "text-xs"),
                       hx_post="/piece/focus",
                       hx_vals=json.dumps({"piece_id": p.id}),
                       hx_target="#center-panel") if on_board else "",
            ),
            cls=f"{'ring-1 ring-primary' if on_board else 'opacity-70'}",
        )
        cards.append(card)
    
    return Div(
        health_bar,
        *warnings,
        Grid(*cards, cols_sm=2, cols_md=3, cols_lg=4, cls="gap-3"),
        cls="space-y-3",
    )


In [ ]:
@rt("/piece/focus", methods=["POST"])
def piece_focus(piece_id: str):
    """Switch to map view centered on a specific piece."""
    piece = next((p for p in GS.player_squad.pieces if p.id == piece_id), None)
    if piece and piece.place is not None:
        # Could set a focused_piece on GS to highlight on map
        GS.focused_piece = piece
    return panel_center(view="map")


!cat ../../HexMagic/game/flag.py

We need consistenct between the side panel pieces list and the optimal move overlay. the Country flag has ways of drawing pieces that could work for both. The pieces emoji is much too small for the list and the colors are tricky for the overlay. we could create diagram glyphs if that is easier

Maybe if the OptimalOverlay is being shown we could have a section in the left placement that gives hex information about the optimal locations in additions to the changes you are suggesting

So what should panel_pieces look like?

In [ ]:
def panel_squad_detail():
    """Rich squad view for the center panel — card grid + network health."""
    squad = GS.player_squad
    king = next((p for p in squad.pieces 
                 if p.type == PieceType.KING and p.place is not None), None)
    
    # ── Network health banner ──
    G = squad.to_nx()
    U = G.to_undirected(as_view=True) if len(G) > 1 else None
    bottlenecks = list(nx.articulation_points(U)) if U else []
    isolated = squad.isolated_pieces()
    sources = squad.food_sources()
    sinks = squad.food_sinks()
    placed = [p for p in squad.pieces if p.place is not None]
    
    health_items = [
        ("🔗", "Connected", f"{len(placed)}/{len(squad.pieces)}"),
        ("⚠️", "Bottlenecks", str(len(bottlenecks))),
        ("🌾", "Sources", str(len(sources))),
        ("🍽️", "Sinks", str(len(sinks))),
        ("👑", "Treasury", f"🍞{king.place.pantry}" if king else "—"),
    ]
    health_bar = DivFullySpaced(
        *[DivCentered(
            Span(icon, cls=TextT.lg),
            Small(label, cls=TextPresets.muted_sm),
            Strong(val, cls="text-sm"),
            cls="space-y-0.5",
        ) for icon, label, val in health_items],
        cls="py-3 px-4 bg-muted/30 rounded-lg",
    )
    
    # ── Warnings ──
    warnings = []
    if isolated:
        names = ", ".join(f"{p.type.icon} {p.name}" for p in isolated)
        warnings.append(
            Div(DivLAligned(Span("🚨"), P(f"Isolated: {names}")),
                cls="text-destructive text-sm bg-destructive/10 rounded px-3 py-2"))
    if bottlenecks:
        names = ", ".join(f"{G.nodes[n]['piece'].type.icon} {G.nodes[n]['piece'].name}" 
                          for n in bottlenecks)
        warnings.append(
            Div(DivLAligned(Span("⚠️"), P(f"Bottlenecks: {names}")),
                cls="text-warning text-sm bg-warning/10 rounded px-3 py-2"))
    
    # ── Piece cards grid ──
    cards = []
    for p in squad.pieces:
        on_board = p.place is not None
        fs = p.food
        
        # Avatar
        b = SVGBuilder()
        b.width, b.height = 60, 60
        p.draw_avatar(MapCord(30, 30), b, size=56, layer="avatar")
        avatar = Div(NotStr(b.xml()), style="flex-shrink:0; line-height:0;")
        
        # Stats
        if on_board:
            tier = p.place.harvest
            income = int(fs.harvest * tier)
            net = income - fs.diet
            remaining = LIFESPAN[p.type] - p.place.tenure if p.type != PieceType.KING else "∞"
            facing = p.place.facing.label
            
            stats = Div(
                DivFullySpaced(
                    Small("Income"), Small(f"+{income}", cls="text-green-600"),
                ),
                DivFullySpaced(
                    Small("Diet"), Small(f"-{fs.diet}", cls="text-destructive"),
                ),
                DivFullySpaced(
                    Small("Net"), 
                    Small(f"{net:+d}", cls="text-green-600 font-bold" if net >= 0 else "text-destructive font-bold"),
                ),
                DivFullySpaced(
                    Small("Drain"), Small(f"⚔️{fs.drain}"),
                ),
                DivFullySpaced(
                    Small("Pantry"), Small(f"🍞{p.place.pantry}", cls="text-amber-600"),
                ),
                DivFullySpaced(
                    Small("Facing"), Small(facing),
                ),
                DivFullySpaced(
                    Small("Turns left"), Small(str(remaining)),
                ),
                cls="text-xs space-y-0.5",
            )
            
            # Sight coverage count
            sight_count = len(p.sight().hexes)
            location_badge = Label(f"Hex {GS.parts.hp2i(p.place.location)}", cls=LabelT.secondary)
        else:
            stats = Div(
                DivFullySpaced(Small("Cost"), Small(f"🍞{fs.cost}")),
                DivFullySpaced(Small("Diet"), Small(f"{fs.diet}/turn")),
                DivFullySpaced(Small("Harvest"), Small(f"×{fs.harvest}")),
                DivFullySpaced(Small("Drain"), Small(f"⚔️{fs.drain}")),
                DivFullySpaced(Small("Lifespan"), Small(f"{LIFESPAN[p.type]} turns")),
                DivFullySpaced(Small("Sight"), Small(f"+{fs.bonus_sight}" if fs.bonus_sight else "base")),
                cls="text-xs space-y-0.5",
            )
            sight_count = 0
            location_badge = Label("Reserve", cls=LabelT.destructive)
        
        card = Card(
            DivLAligned(avatar, Div(
                DivLAligned(Span(p.type.icon), Strong(p.name)),
                P(f"b. {p.year}", cls=TextPresets.muted_sm),
            )),
            Divider(cls="my-2"),
            stats,
            header=DivFullySpaced(
                Small(p.type.value.title(), cls=TextPresets.muted_sm),
                location_badge,
            ),
            footer=DivFullySpaced(
                Small(f"👁️ {sight_count} hexes" if sight_count else "", cls=TextPresets.muted_sm),
                Button("🔍", cls=(ButtonT.ghost, "text-xs"),
                       hx_post="/piece/focus",
                       hx_vals=json.dumps({"piece_id": p.id}),
                       hx_target="#center-panel") if on_board else "",
            ),
            cls=f"{'ring-1 ring-primary' if on_board else 'opacity-70'}",
        )
        cards.append(card)
    
    return Div(
        health_bar,
        *warnings,
        Grid(*cards, cols_sm=2, cols_md=3, cols_lg=4, cls="gap-3"),
        cls="space-y-3",
    )


#### charts

In [ ]:

import io, base64, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def panel_charts():
    """Matplotlib charts showing squad economics."""
    squad = GS.player_squad
    parts = GS.parts
    placed = [p for p in squad.pieces if p.place is not None]
    
    if not placed:
        return P("No pieces placed yet", cls=TextPresets.muted_sm)
    
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), facecolor='none')
    fig.patch.set_alpha(0)
    
    # ── Chart 1: Per-piece food balance ──
    ax = axes[0]
    names = [f"{p.type.icon}{p.name[:6]}" for p in placed]
    income = [int(p.food.harvest * p.place.harvest) for p in placed]
    diet = [-p.food.diet for p in placed]
    x = range(len(placed))
    ax.bar(x, income, color='#27ae60', alpha=0.8, label='Income')
    ax.bar(x, diet, color='#e74c3c', alpha=0.8, label='Diet')
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=45, ha='right', fontsize=8)
    ax.set_title('Food Balance', fontsize=10)
    ax.legend(fontsize=7)
    ax.axhline(0, color='gray', lw=0.5)
    
    # ── Chart 2: Pantry levels ──
    ax = axes[1]
    pantries = [p.place.pantry for p in placed]
    colors = ['#27ae60' if v > 5 else '#f39c12' if v > 0 else '#e74c3c' for v in pantries]
    ax.barh(names, pantries, color=colors, alpha=0.8)
    ax.set_title('Pantry Levels', fontsize=10)
    ax.axvline(0, color='gray', lw=0.5)
    
    # ── Chart 3: Drain threat (who's being drained) ──
    ax = axes[2]
    drain_taken = []
    for p in placed:
        idx = parts.hp2i(p.place.location)
        d = sum(ep.food.drain for sq in parts.squads if sq is not squad
                for ep in sq.pieces if ep.place is not None
                and idx in ep.sight().hexes)
        drain_taken.append(d)
    drain_dealt = [p.food.drain * sum(1 for sq in parts.squads if sq is not squad
                   for ep in sq.pieces if ep.place is not None
                   and parts.hp2i(ep.place.location) in p.sight().hexes)
                   for p in placed]
    ax.barh(names, drain_dealt, color='#3498db', alpha=0.7, label='Dealing')
    ax.barh(names, [-d for d in drain_taken], color='#e74c3c', alpha=0.7, label='Taking')
    ax.set_title('Drain', fontsize=10)
    ax.legend(fontsize=7)
    ax.axvline(0, color='gray', lw=0.5)
    
    plt.tight_layout()
    
    # Render to inline SVG
    buf = io.BytesIO()
    fig.savefig(buf, format='svg', bbox_inches='tight', transparent=True)
    plt.close(fig)
    buf.seek(0)
    svg_str = buf.read().decode('utf-8')
    
    # Network stats summary
    G = squad.to_nx()
    king = next((p for p in placed if p.type == PieceType.KING), None)
    total_income = sum(int(p.food.harvest * p.place.harvest) for p in placed)
    total_diet = sum(p.food.diet for p in placed)
    net = total_income - total_diet
    
    summary = DivFullySpaced(
        DivCentered(Strong(f"+{total_income}"), Small("Income", cls=TextPresets.muted_sm)),
        DivCentered(Strong(f"-{total_diet}", cls="text-destructive"), Small("Diet", cls=TextPresets.muted_sm)),
        DivCentered(Strong(f"{net:+d}", cls="text-green-600 text-lg" if net >= 0 else "text-destructive text-lg"),
                    Small("Net/turn", cls=TextPresets.muted_sm)),
        DivCentered(Strong(f"🍞{king.place.pantry}" if king else "—"), Small("Treasury", cls=TextPresets.muted_sm)),
        cls="py-3 px-4 bg-muted/30 rounded-lg",
    )
    
    return Div(summary, NotStr(svg_str), cls="space-y-3")


#### Toggle Overlays

In [ ]:
@rt("/toggle_debug", methods=["POST"])
def toggle_debug():
    GS.debug_overlays = not GS.debug_overlays
    log.info(f"DEBUG overlays={'ON' if GS.debug_overlays else 'OFF'}")
    return panel_center()


In [ ]:
@rt("/zoom", methods=["POST"])
def zoom(radius: int):
    GS.radius = max(10, min(45, radius))  # ← is this being called?
    log.info(f"ZOOM radius={GS.radius}")
    return panel_center()

In [ ]:
@rt("/toggle_overlay", methods=["POST"])
def toggle_overlay(overlay: str):
    log.info(f"TOGGLE overlay={overlay} active={GS.active_overlays}")
    if overlay in GS.active_overlays:
        GS.active_overlays.discard(overlay)
    else:
        GS.active_overlays.add(overlay)
    return panel_center()

In [ ]:
@rt("/piece/focus", methods=["POST"])
def piece_focus(piece_id: str):
    piece = next((p for p in GS.player_squad.pieces if p.id == piece_id), None)
    if piece and piece.place is not None:
        GS.focused_piece = piece
    return panel_center(view="map")


In [ ]:
@rt("/piece/unfocus", methods=["POST"])
def piece_unfocus():
    GS.focused_piece = None
    return panel_center(view="map")


In [ ]:
def _build_overlays():
    """Assemble overlay list from GS.active_overlays."""
    if 'terrain' in GS.active_overlays:
        overlays = [TerrainOverlay(), RiverOverlay(max_width=4)]
    else:
        overlays = [CreamOverlay(stylized=True), RiverOverlay(max_width=4)]
    
    OVERLAY_MAP = {
        'food':          FoodOverlay,
        'lanterns':      LanternOverlay,
        'network':       FoodNetworkOverlay,
        'optimal':       lambda: OptimalPlacementOverlay(top_n=3),
        'ocean':         OceanWaveOverlay,
        'soil':          SoilOverlay,
        'climate':       ClimateOverlay,
        'temperature':   TemperatureOverlay,
        'precipitation': PrecipitationOverlay,
    }
    for key, factory in OVERLAY_MAP.items():
        if key in GS.active_overlays:
            overlays.append(factory())
    
    overlays.append(FogOverlay())
    overlays.append(PieceOverlay())
    if 'names' in GS.active_overlays:
        overlays.append(SquadNamesOverlay())
    
    return overlays


def _focus_bar():
    """Focus banner + region if a piece is focused, else (None, "")."""
    focused = getattr(GS, 'focused_piece', None)
    if focused and focused.place is not None:
        region = focused.sight()
        bar = DivFullySpaced(
            DivLAligned(
                Span(focused.type.icon),
                Strong(focused.name, cls="text-sm"),
                Small(f"Hex {GS.parts.hp2i(focused.place.location)}",
                      cls=TextPresets.muted_sm),
                Small(f"👁️ {len(region.hexes)} hexes", cls=TextPresets.muted_sm),
            ),
            Button("✕ Full Map", cls=(ButtonT.ghost, "text-xs"),
                   hx_post="/piece/unfocus", hx_target="#center-panel"),
            cls="px-3 py-1 bg-primary/10 rounded-md mb-2",
        )
        return region, bar
    return None, ""


def _map_controls():
    """Zoom slider, overlay toggles, debug button."""
    gameplay_overlays = [
        ("terrain", "🏔️ Elevation"), ("food", "🌾 Food"),
        ("lanterns", "🔦 Sight"),     ("network", "🔗 Network"),
        ("optimal", "🎯 Optimal"),    ("names", "🏷️ Names"),
    ]
    detail_overlays = [
        ("ocean", "🌊 Ocean"),        ("soil", "🪨 Soil"),
        ("climate", "🌡️ Climate"),    ("temperature", "🌤️ Temp"),
        ("precipitation", "🌧️ Rain"),
    ]
    
    def overlay_row(items):
        return DivLAligned(
            *[Button(
                label,
                cls=(ButtonT.primary if key in GS.active_overlays else ButtonT.ghost, "text-xs"),
                hx_post="/toggle_overlay",
                hx_vals=json.dumps({"overlay": key}),
                hx_target="#center-panel",
            ) for key, label in items],
            cls="gap-1",
        )
    
    zoom_slider = Input(type="range", id="radius", name="radius",
        value=str(GS.radius), min="10", max="45", step="1",
        hx_post="/zoom", hx_trigger="change",
        hx_target="#center-panel", cls="uk-range flex-1")
    
    debug_button = Button(
        f"🐛 {'ON' if GS.debug_overlays else 'OFF'}",
        cls=(ButtonT.primary if GS.debug_overlays else ButtonT.ghost, "text-xs"),
        hx_post="/toggle_debug", hx_target="#center-panel",
    )
    
    return Div(
        DivLAligned(
            Small(f"🔍 {GS.radius}", cls=TextPresets.muted_sm),
            zoom_slider, debug_button, cls="flex-1 gap-2",
        ),
        overlay_row(gameplay_overlays),
        overlay_row(detail_overlays),
        cls="space-y-1 py-2",
    )


def _placement_status():
    """Status line showing pending selection/placement."""
    if GS.selected_piece:
        p = GS.selected_piece
        return P(f"📍 Click map to place {p.type.icon} {p.name}",
                 cls="text-primary font-medium text-center")
    if GS.pending_placement:
        p = GS.pending_placement.piece
        return P(f"Pending: {p.type.icon} {p.name} → hex. Commit or click to undo.",
                 cls="text-amber-600 text-center text-sm")
    return ""

In [ ]:
def panel_map_content():
    """Map view — zoomed to focused piece or full map."""
    overlays = _build_overlays()
    region, focus_bar = _focus_bar()
    
    ctx = GS.parts.overlayContext(
        region=region, padding=2 if region else 0, radius=GS.radius)
    ctx.grid.builder.layers = []
    ctx.grid.adjustRadius(GS.radius)
    TerrainDisplay(*overlays, radius=GS.radius, ctx=ctx, debug=GS.debug_overlays)
    
    map_div = HexTouchMap(
        ctx.grid,
        on_click=HexWrapper.route("/hex_click", "#center-panel"),
        cls="w-full h-full",
    )
    
    return Div(focus_bar, map_div, _placement_status(), _map_controls())

Have I put in everything correctly?

I made the change. is there a way to cat the log so I can see what happened?

In [ ]:
print('\n'.join(game_log))


The zoom slider doesn't seem to call the zoom route. is it a paramter name thing?

Did I get this right? The toggle doesn't seem to work

In [ ]:
@rt
def game():
    if not GS.is_active:
        return RedirectResponse("/", status_code=303)
    
    collapsed = getattr(GS, 'panel_collapsed', False)
    
    return Title("HexMagic — Game"), Container(
        DivFullySpaced(
            H3("⬡ HexMagic", cls="uk-h3"),
            Div(
                Span(f"Turn {GS.turn}", cls=TextT.lg),
                Span(f" · {GS.player_squad.name}", cls=TextPresets.muted_sm),
            ),
            Button("🏠 Menu", cls=ButtonT.ghost, hx_get="/", hx_target="body"),
            cls="py-2",
        ),
        Divider(),
        # Use flex instead of grid so left column can collapse
        Div(
            Div(
                Div(hx_get="/panel/pieces", hx_trigger="load",
                    hx_target="#piece-panel", id="piece-panel") if not collapsed
                else Button("▶", cls=(ButtonT.ghost, "text-xs"),
                           hx_post="/toggle_panel", hx_target="#left-col"),
                id="left-col",
                cls="w-8 flex-shrink-0" if collapsed else "w-[250px] flex-shrink-0 overflow-y-auto max-h-[80vh]",
            ),
            Div(id="center-panel", hx_get="/panel/center", hx_trigger="load",
                hx_target="#center-panel", hx_swap="outerHTML",
                cls="flex-1 min-w-0"),
            Div(
                Div(id="history-panel", hx_get="/panel/history", hx_trigger="load",
                    hx_target="#history-panel"),
                debug_log(),
                cls="w-[280px] flex-shrink-0 overflow-y-auto max-h-[80vh] space-y-3",
            ),
            cls="flex gap-3",
        ),
    )


is def game() okay can we toggle the left panel?

I want to be able to toggle the middle section for the map and something like the matplotlib to show how pieces are doing


In [ ]:
@rt("/debug/ping", methods=["POST"])
def ping():
    log.info("PING received")
    return P("pong", cls="text-green-500")


In [ ]:
webMe(Div(hx_get="/", hx_trigger="load", hx_target="this"))

In [ ]:
To debug the overlay I need to able to toggle the debug paramter